In [2]:
# ==== RESTORE CHECKPOINT (supports .tar.gz OR a /checkpoint folder) ====
import os, glob, tarfile, shutil, numpy as np, pandas as pd

IN_ROOT = "/kaggle/input"
RESTORE_ROOT = "/kaggle/working/restore"
CKP_DIR = f"{RESTORE_ROOT}/checkpoint"

# 1) Find sources
cand_tar = glob.glob(f"{IN_ROOT}/*/checkpoint_bundle.tar.gz")
cand_dir = glob.glob(f"{IN_ROOT}/*/checkpoint")

print("Found tar:", cand_tar)
print("Found checkpoint dir:", cand_dir)

# 2) Materialize checkpoint -> /kaggle/working/restore/checkpoint
os.makedirs(RESTORE_ROOT, exist_ok=True)
if cand_tar:
    src = cand_tar[0]
    print("Extracting:", src)
    with tarfile.open(src, "r:gz") as tar:
        tar.extractall(RESTORE_ROOT)
    assert os.path.exists(CKP_DIR), "Extracted tar but /checkpoint not found inside."
elif cand_dir:
    src = cand_dir[0]
    print("Copying directory:", src)
    # Kaggle Python has dirs_exist_ok in 3.11
    shutil.copytree(src, CKP_DIR, dirs_exist_ok=True)
else:
    raise AssertionError("Attach either a dataset with checkpoint_bundle.tar.gz OR a dataset exposing /checkpoint")

# 3) Quick sanity
expected = [
    "img_emb_all.f16", "img_has.npy",
    "img_emb_L14_all.f16", "img_has_L14.npy",
    "text_emb_shards", "out", "runs"
]
for name in expected:
    path = os.path.join(CKP_DIR, name)
    print(f"{name:>22}: {'OK' if os.path.exists(path) else 'MISSING'}")

# 4) Re-wire small arrays into memory if present
def _maybe(path, key):
    if os.path.exists(path):
        arr = np.load(path, allow_pickle=False)
        globals()[key] = arr
        print(f"Loaded {key}: {arr.shape}")
    else:
        print(f"Skip {key} (missing)")

_maybe(os.path.join(CKP_DIR, "fe_tr.npy"),        "FE_TR")
_maybe(os.path.join(CKP_DIR, "fe_te.npy"),        "FE_TE")
_maybe(os.path.join(CKP_DIR, "oof_stack_l.npy"),  "oof_stack_l")
_maybe(os.path.join(CKP_DIR, "te_stack_l.npy"),   "te_stack_l")
_maybe(os.path.join(CKP_DIR, "oof_img2_l.npy"),   "oof_img2_l")
_maybe(os.path.join(CKP_DIR, "oof_e5_l.npy"),     "oof_e5_l")

print("\nCheckpoint restored to:", CKP_DIR)


Found tar: []
Found checkpoint dir: ['/kaggle/input/saved-ml-model/checkpoint']
Copying directory: /kaggle/input/saved-ml-model/checkpoint
       img_emb_all.f16: OK
           img_has.npy: OK
   img_emb_L14_all.f16: OK
       img_has_L14.npy: OK
       text_emb_shards: OK
                   out: OK
                  runs: OK
Loaded FE_TR: (75000, 222)
Loaded FE_TE: (75000, 222)
Loaded oof_stack_l: (75000,)
Loaded te_stack_l: (75000,)
Loaded oof_img2_l: (75000,)
Loaded oof_e5_l: (75000,)

Checkpoint restored to: /kaggle/working/restore/checkpoint


In [3]:
# ==== Rebuild dataframes and hook memmaps ====
import os, glob, numpy as np, pandas as pd

# Locate train/test in the attached competition data
train_candidates = glob.glob("/kaggle/input/*/*/*/train.csv") + glob.glob("/kaggle/input/*/*/train.csv")
test_candidates  = glob.glob("/kaggle/input/*/*/*/test.csv")  + glob.glob("/kaggle/input/*/*/test.csv")
assert train_candidates and test_candidates, "Attach the dataset that has dataset/train.csv and dataset/test.csv."

TRAIN_CSV = sorted(train_candidates)[0]
TEST_CSV  = sorted(test_candidates)[0]
print("Using:", TRAIN_CSV)
print("Using:", TEST_CSV)

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

# Standard concat view
df_all = pd.concat([
    train[['sample_id','catalog_content','image_link']].assign(is_train=1),
    test[['sample_id','catalog_content','image_link']].assign(is_train=0)
], ignore_index=True)

# Hook OpenCLIP B/32 memmap + flags
CKP_DIR = "/kaggle/working/restore/checkpoint"
b32_path = os.path.join(CKP_DIR, "img_emb_all.f16")
b32_flag = os.path.join(CKP_DIR, "img_has.npy")
assert os.path.exists(b32_path) and os.path.exists(b32_flag), "B/32 memmap or flags missing in checkpoint."
N = len(df_all)
dim_b32 = (os.path.getsize(b32_path)//2)//N
IMG_B32_ALL = np.memmap(b32_path, dtype='float16', mode='r', shape=(N, dim_b32))
HAS_B32_ALL = np.load(b32_flag).astype(np.uint8)

# Hook OpenCLIP L/14 memmap + flags (may be partial; that's fine)
l14_path = os.path.join(CKP_DIR, "img_emb_L14_all.f16")
l14_flag = os.path.join(CKP_DIR, "img_has_L14.npy")
HAS_L14_ALL = None
IMG_L14_ALL = None
if os.path.exists(l14_path) and os.path.exists(l14_flag):
    dim_l14 = (os.path.getsize(l14_path)//2)//N
    IMG_L14_ALL = np.memmap(l14_path, dtype='float16', mode='r', shape=(N, dim_l14))
    HAS_L14_ALL = np.load(l14_flag).astype(np.uint8)
    print(f"L/14 dim={dim_l14} | coverage={(HAS_L14_ALL.mean()*100):.1f}%")
else:
    print("L/14 memmap/flags not found; you can resume the encoder later.")

print("B/32 dim=", dim_b32, "| coverage=", HAS_B32_ALL.mean()*100, "%")
print("Restore wiring complete.")


Using: /kaggle/input/amazon-ml-challenge/student_resource/dataset/train.csv
Using: /kaggle/input/amazon-ml-challenge/student_resource/dataset/test.csv
L/14 dim=768 | coverage=10.9%
B/32 dim= 512 | coverage= 41.664 %
Restore wiring complete.


In [4]:
# ===========================
# FAST RESTORE: train/test/df_all + FE_TR/FE_TE + EMB_TR/EMB_TE (+ optional E5)
# ===========================
import os, glob, numpy as np, pandas as pd

def pick_first(paths):
    for p in paths:
        if isinstance(p, str) and os.path.exists(p): 
            return p
    return None

# ---- 1) Locate checkpoint root
CKP = pick_first([
    "/kaggle/working/restore/checkpoint",
    "/kaggle/input/saved-ml-model/checkpoint",
    "/kaggle/working/checkpoint",
])
assert CKP, "Checkpoint folder not found. Mount your saved bundle as an input and ensure /checkpoint exists."

print("Checkpoint:", CKP)

# ---- 2) Load train/test (only if not already loaded)
if 'train' not in globals() or 'test' not in globals():
    # Try common dataset mount points
    train_csv = pick_first([
        "/kaggle/input/amazonml/train.csv",
        "/kaggle/input/amazon-ml/train.csv",
        "/kaggle/input/*/train.csv"
    ])
    test_csv = pick_first([
        "/kaggle/input/amazonml/test.csv",
        "/kaggle/input/amazon-ml/test.csv",
        "/kaggle/input/*/test.csv"
    ])
    # Fallback to any train/test under /kaggle/input
    if not train_csv or "*" in str(train_csv):
        cands = glob.glob("/kaggle/input/*/train.csv")
        train_csv = cands[0] if cands else None
    if not test_csv or "*" in str(test_csv):
        cands = glob.glob("/kaggle/input/*/test.csv")
        test_csv = cands[0] if cands else None

    assert train_csv and test_csv, "Could not find train.csv/test.csv under /kaggle/input/*"
    train = pd.read_csv(train_csv)
    test  = pd.read_csv(test_csv)
    print("Loaded train/test:", train.shape, test.shape)
else:
    print("Using train/test already in memory:", train.shape, test.shape)

# ---- 3) Build df_all (needed by later steps)
if 'df_all' not in globals():
    tr = train.copy(); tr['is_train'] = 1
    te = test.copy();  te['is_train'] = 0
    df_all = pd.concat([tr, te], ignore_index=True)
    print("Built df_all:", df_all.shape)

# ---- 4) Restore FE matrices (required)
fe_tr_path = pick_first([os.path.join(CKP, "fe_tr.npy"), "/kaggle/working/fe_tr.npy"])
fe_te_path = pick_first([os.path.join(CKP, "fe_te.npy"), "/kaggle/working/fe_te.npy"])
assert fe_tr_path and fe_te_path, "Missing fe_tr.npy/fe_te.npy in checkpoint."

FE_TR = np.load(fe_tr_path, mmap_mode='r').astype('float32', copy=False)
FE_TE = np.load(fe_te_path, mmap_mode='r').astype('float32', copy=False)
assert FE_TR.shape[0] == len(train) and FE_TE.shape[0] == len(test), "FE rows do not match train/test."
print("FE_TR/FE_TE:", FE_TR.shape, FE_TE.shape)

# ---- 5) Restore MiniLM shards -> EMB_TR/EMB_TE (required)
shard_dir = pick_first([os.path.join(CKP, "text_emb_shards"), "/kaggle/working/text_emb_shards"])
assert shard_dir and os.path.isdir(shard_dir), "MiniLM shards folder not found in checkpoint."
shards = sorted(glob.glob(os.path.join(shard_dir, "emb_shard_*.npy")))
assert shards, "No MiniLM shards found."
emb_all = np.vstack([np.load(p, mmap_mode='r') for p in shards]).astype('float32', copy=False)
is_tr = (df_all['is_train'].values == 1)
EMB_TR = emb_all[is_tr]
EMB_TE = emb_all[~is_tr]
print("MiniLM EMB_TR/EMB_TE:", EMB_TR.shape, EMB_TE.shape)

# ---- 6) (Optional) Restore E5 memmap -> E5_TR/E5_TE
e5_mem = pick_first([os.path.join(CKP, "text_e5_emb_all.f16"), "/kaggle/working/text_e5_emb_all.f16"])
if e5_mem and os.path.exists(e5_mem):
    N = len(df_all)
    e5_dim = (os.path.getsize(e5_mem)//2)//N  # float16 -> 2 bytes
    e5_mm  = np.memmap(e5_mem, dtype='float16', mode='r', shape=(N, e5_dim))
    E5_TR = np.array(e5_mm[is_tr], dtype='float32')
    E5_TE = np.array(e5_mm[~is_tr], dtype='float32')
    print("E5_TR/E5_TE:", E5_TR.shape, E5_TE.shape)
else:
    print("E5 memmap not found (that's OK).")

print("RESTORE OK — you can re-run the router/seg-blend cell now.")


Checkpoint: /kaggle/working/restore/checkpoint
Using train/test already in memory: (75000, 4) (75000, 3)
FE_TR/FE_TE: (75000, 222) (75000, 222)
MiniLM EMB_TR/EMB_TE: (75000, 384) (75000, 384)
E5_TR/E5_TE: (75000, 1024) (75000, 1024)
RESTORE OK — you can re-run the router/seg-blend cell now.


In [5]:
# === Self-healing router + segment blend (robust masks; no index mismatch) ===
import os, glob, time, json
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

# --------- prereqs ----------
assert all(v in globals() for v in ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE']), "Load FE/EMB/train/test first."
use_e5 = ('E5_TR' in globals()) and ('E5_TE' in globals())

def to2(a, dtype='float32'):
    a = np.asarray(a);  return (a if a.ndim==2 else a.reshape(-1,1)).astype(dtype, copy=False)

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

CKP_BASE = "/kaggle/input/saved-ml-model/checkpoint"
CAND_DIRS = [
    "out",                                  # current run
    os.path.join(CKP_BASE, "out"),          # saved checkpoint's /out
    CKP_BASE                                 # saved checkpoint root
]

def find_one(patterns):
    for base in CAND_DIRS:
        for pat in patterns:
            for p in sorted(glob.glob(os.path.join(base, pat))):
                if os.path.exists(p):
                    return p
    return None

# --------- base model (good OOF/TEST already saved) ----------
base_oof_csv  = find_one(["oof_e5_brand_cluster.csv", "oof_e5_fused.csv", "oof_xgb_best_promoted.csv"])
base_test_csv = find_one(["test_predictions_e5_brand_cluster.csv",
                          "test_predictions_e5_fused.csv",
                          "test_predictions_xgb_best_promoted.csv"])
assert base_oof_csv and base_test_csv, "Couldn't find base OOF/TEST CSVs."

log(f"BASE OOF : {base_oof_csv}")
log(f"BASE TEST: {base_test_csv}")

# Load base OOF log (handle different column names)
df_base = pd.read_csv(base_oof_csv).set_index('sample_id')
base_col = [c for c in ["oof_log","oof_log_router","oof","pred_log"] if c in df_base][0]
oof_base_l = df_base[base_col].reindex(train['sample_id']).values
te_base_p  = pd.read_csv(base_test_csv).set_index('sample_id')['price'].reindex(test['sample_id']).values

# --------- try to load router outputs; if missing, train now ----------
router_oof_csv  = find_one(["oof_router_lowprice.csv", "oof_router.csv"])
router_test_csv = find_one(["test_predictions_router_blended.csv", "test_predictions_router.csv"])

def train_router_now():
    log("Router artifacts not found; training router now...")
    parts_tr = [to2(FE_TR), to2(EMB_TR)]
    parts_te = [to2(FE_TE), to2(EMB_TE)]
    if use_e5:
        parts_tr.append(to2(E5_TR)); parts_te.append(to2(E5_TE))
    Xr_tr = np.hstack(parts_tr).astype('float32')
    Xr_te = np.hstack(parts_te).astype('float32')
    log(f"Router matrices -> Xr_tr={Xr_tr.shape}, Xr_te={Xr_te.shape}")

    y = train['price'].astype(float).values
    y_log = np.log(np.clip(y, 0.01, None))
    thr = np.quantile(y, 0.15)
    y_low = (y <= thr).astype(np.int32)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    clf_params = dict(objective='binary:logistic', max_depth=6, n_estimators=1200, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                      device='cuda', eval_metric='logloss', random_state=42, early_stopping_rounds=100)
    oof_p_low = np.zeros(len(y), dtype='float32'); te_p_low = np.zeros(len(test), dtype='float32')
    for f,(tr,va) in enumerate(skf.split(Xr_tr, y_low)):
        m = xgb.XGBClassifier(**clf_params)
        m.fit(Xr_tr[tr], y_low[tr], eval_set=[(Xr_tr[va], y_low[va])], verbose=False)
        oof_p_low[va] = m.predict_proba(Xr_tr[va])[:,1]
        te_p_low += m.predict_proba(Xr_te)[:,1] / skf.n_splits
        log(f"  [clf fold {f}] done")

    reg_params_main = dict(objective='reg:absoluteerror', max_depth=7, n_estimators=3500, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0,
                           tree_method='hist', device='cuda', eval_metric='mae', random_state=42,
                           early_stopping_rounds=150)
    reg_params_low  = {**reg_params_main, 'max_depth': 6, 'n_estimators': 3000}

    oof_router_l = np.zeros_like(y_log, dtype='float32')
    te_router_l  = np.zeros(len(test), dtype='float32')
    for f,(tr,va) in enumerate(skf.split(Xr_tr, y_low)):
        low_tr = tr[y_low[tr]==1];  low_tr = low_tr if len(low_tr) >= 500 else tr
        m_low  = xgb.XGBRegressor(**reg_params_low)
        m_main = xgb.XGBRegressor(**reg_params_main)
        m_low.fit(Xr_tr[low_tr],  y_log[low_tr],  eval_set=[(Xr_tr[va], y_log[va])],  verbose=False)
        m_main.fit(Xr_tr[tr],     y_log[tr],     eval_set=[(Xr_tr[va], y_log[va])],  verbose=False)
        p_low_va  = m_low.predict(Xr_tr[va])
        p_main_va = m_main.predict(Xr_tr[va])
        mix_va = np.log(np.clip(oof_p_low[va]*np.exp(p_low_va) + (1-oof_p_low[va])*np.exp(p_main_va), 1e-4, None))
        oof_router_l[va] = mix_va

        p_low_te  = m_low.predict(Xr_te)
        p_main_te = m_main.predict(Xr_te)
        te_mix = np.log(np.clip(te_p_low*np.exp(p_low_te) + (1-te_p_low)*np.exp(p_main_te), 1e-4, None))
        te_router_l += te_mix / skf.n_splits

        sm = smape(y[va], np.exp(mix_va).clip(0.01))
        log(f"  [router fold {f}] SMAPE={sm:.3f}%")

    cv_router = smape(y, np.exp(oof_router_l).clip(0.01))
    log(f"Router CV SMAPE: {cv_router:.3f}%")

    os.makedirs("out", exist_ok=True)
    pd.DataFrame({'sample_id': train['sample_id'],
                  'oof_log_router': oof_router_l,
                  'oof_price_router': np.exp(oof_router_l).clip(0.01)}).to_csv("out/oof_router_lowprice.csv", index=False)
    pd.DataFrame({'sample_id': test['sample_id'],
                  'price': np.exp(te_router_l).clip(0.01).astype(float)}).to_csv("out/test_predictions_router.csv", index=False)
    return "out/oof_router_lowprice.csv", "out/test_predictions_router.csv"

if router_oof_csv is None or router_test_csv is None:
    router_oof_csv, router_test_csv = train_router_now()
else:
    log(f"ROUTER OOF : {router_oof_csv}")
    log(f"ROUTER TEST: {router_test_csv}")

# --------- load router preds ----------
oof_router_df = pd.read_csv(router_oof_csv).set_index('sample_id')
router_oof_col = [c for c in ["oof_log_router","oof_log","oof"] if c in oof_router_df][0]
oof_router_l = oof_router_df[router_oof_col].reindex(train['sample_id']).values
te_router_p  = pd.read_csv(router_test_csv).set_index('sample_id')['price'].reindex(test['sample_id']).values

# --------- segments (has_img ∧ missing_qty) — BOOLEAN MASKS in train/test space ----------
N = len(df_all)
is_tr = (df_all['is_train'].values == 1)
is_te = ~is_tr

# has_img flags
if os.path.exists(os.path.join(CKP_BASE, "img_has.npy")):
    has_img_all = np.load(os.path.join(CKP_BASE, "img_has.npy")).astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('') != '').values

# missing-qty mask
if 'mask_missing_qty' in globals():
    mmq_all = mask_missing_qty.astype(bool)
else:
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    mmq_all = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values

SEG_A_all = has_img_all & mmq_all
SEG_A_tr_mask = SEG_A_all[is_tr]
SEG_B_tr_mask = ~SEG_A_tr_mask
SEG_A_te_mask = SEG_A_all[is_te]
SEG_B_te_mask = ~SEG_A_te_mask

log(f"Segments | train A={int(SEG_A_tr_mask.sum())}, B={int(SEG_B_tr_mask.sum())} | "
    f"test A={int(SEG_A_te_mask.sum())}, B={int(SEG_B_te_mask.sum())}")

# --------- best weights per segment ----------
y = train['price'].astype(float).values
router_price_tr = np.exp(oof_router_l).clip(0.01)
base_price_tr   = np.exp(oof_base_l).clip(0.01)

def best_w(tr_mask):
    idx = np.where(tr_mask)[0]
    if idx.size == 0:
        return 0.0, float('inf')
    best = (1e9, 0.0)
    for w in np.linspace(0.0, 1.0, 51):
        blend = w*router_price_tr[idx] + (1.0-w)*base_price_tr[idx]
        cv = smape(y[idx], blend)
        if cv < best[0]: best = (cv, w)
    return best[1], best[0]

wA, cvA = best_w(SEG_A_tr_mask)
wB, cvB = best_w(SEG_B_tr_mask)

blend_tr_full = np.empty_like(router_price_tr)
blend_tr_full[SEG_A_tr_mask] = wA*router_price_tr[SEG_A_tr_mask] + (1-wA)*base_price_tr[SEG_A_tr_mask]
blend_tr_full[SEG_B_tr_mask] = wB*router_price_tr[SEG_B_tr_mask] + (1-wB)*base_price_tr[SEG_B_tr_mask]
cv_all = smape(y, blend_tr_full)

log(f"Segment A: w={wA:.2f} | seg-CV={cvA:.3f}%")
log(f"Segment B: w={wB:.2f} | seg-CV={cvB:.3f}%")
log(f"Overall blended CV SMAPE: {cv_all:.3f}%")

# --------- save submission ----------
eps = 5e-4
te_blend = np.empty_like(te_base_p)
te_blend[SEG_A_te_mask] = wA*te_router_p[SEG_A_te_mask] + (1-wA)*te_base_p[SEG_A_te_mask]
te_blend[SEG_B_te_mask] = wB*te_router_p[SEG_B_te_mask] + (1-wB)*te_base_p[SEG_B_te_mask]

os.makedirs("out", exist_ok=True)
sub_path = "out/test_predictions_ROUTER_vs_BASE_segblend.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': np.clip(te_blend, eps, None).astype(float)}).to_csv(sub_path, index=False)
log(f"Saved -> {sub_path}")

with open("out/ROUTER_vs_BASE_segblend_meta.json","w") as f:
    json.dump({
        'base_oof': base_oof_csv, 'base_test': base_test_csv,
        'router_oof': router_oof_csv, 'router_test': router_test_csv,
        'wA': float(wA), 'wB': float(wB), 'cvA': float(cvA), 'cvB': float(cvB), 'cv_all': float(cv_all),
        'eps': float(eps)
    }, f, indent=2)
log("Saved -> out/ROUTER_vs_BASE_segblend_meta.json")


[13:53:01] BASE OOF : /kaggle/input/saved-ml-model/checkpoint/out/oof_e5_brand_cluster.csv | +0.0s
[13:53:01] BASE TEST: /kaggle/input/saved-ml-model/checkpoint/out/test_predictions_e5_brand_cluster.csv | +0.0s
[13:53:01] Router artifacts not found; training router now... | +0.1s
[13:53:02] Router matrices -> Xr_tr=(75000, 1630), Xr_te=(75000, 1630) | +0.8s


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [13:53:40] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


[13:53:42]   [clf fold 0] done | +41.7s
[13:54:23]   [clf fold 1] done | +82.2s
[13:55:06]   [clf fold 2] done | +125.7s
[13:55:49]   [clf fold 3] done | +168.2s
[13:56:28]   [clf fold 4] done | +207.7s
[14:01:00]   [router fold 0] SMAPE=49.751% | +479.5s
[14:05:43]   [router fold 1] SMAPE=50.836% | +762.1s
[14:09:54]   [router fold 2] SMAPE=50.211% | +1013.5s
[14:13:51]   [router fold 3] SMAPE=50.684% | +1249.9s
[14:18:51]   [router fold 4] SMAPE=49.581% | +1549.9s
[14:18:51] Router CV SMAPE: 50.212% | +1549.9s


/tmp/ipykernel_37/3021355068.py:153: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mmq_all = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values


[14:18:53] Segments | train A=23995, B=51005 | test A=22635, B=52365 | +1552.4s
[14:18:53] Segment A: w=0.42 | seg-CV=53.941% | +1552.5s
[14:18:53] Segment B: w=0.28 | seg-CV=45.671% | +1552.5s
[14:18:53] Overall blended CV SMAPE: 48.317% | +1552.5s
[14:18:53] Saved -> out/test_predictions_ROUTER_vs_BASE_segblend.csv | +1552.6s
[14:18:53] Saved -> out/ROUTER_vs_BASE_segblend_meta.json | +1552.6s


In [6]:
# ==== SANITY REPORT: did everything load correctly? ====
import os, glob, json, numpy as np, pandas as pd

ok = True
def check(cond, msg):
    global ok
    print(("✅ " if cond else "❌ ") + msg)
    ok &= bool(cond)

print("=== DATAFRAMES ===")
check('train' in globals(), "train loaded")
check('test'  in globals(), "test loaded")
check('df_all' in globals(), "df_all built (train+test concat)")
if 'train' in globals():
    print("   train rows:", len(train))
if 'test' in globals():
    print("   test  rows:", len(test))
if 'df_all' in globals():
    print("   df_all rows:", len(df_all))

print("\n=== CHECKPOINT FILES (on disk) ===")
CKP = "/kaggle/working/restore/checkpoint"
for p in ["out", "runs", "text_emb_shards", "img_emb_all.f16", "img_has.npy",
          "img_emb_L14_all.f16", "img_has_L14.npy", "text_e5_emb_all.f16",
          "fe_tr.npy", "fe_te.npy", "oof_stack_l.npy", "te_stack_l.npy",
          "oof_img2_l.npy", "oof_e5_l.npy"]:
    path = os.path.join(CKP, p)
    print(f" - {p:20}:", "OK" if os.path.exists(path) else "missing")

print("\n=== MINI-LM TEXT EMBEDDINGS (disk) ===")
shards = sorted(glob.glob(os.path.join(CKP, "text_emb_shards", "emb_shard_*.npy")))
check(len(shards) > 0, f"found {len(shards)} MiniLM shard(s)")
if shards:
    # Do not fully load to RAM—peek one shard
    arr = np.load(shards[0], mmap_mode='r')
    print("   shard[0] shape:", arr.shape)

print("\n=== FE (engineered features) in memory ===")
for name in ["FE_TR","FE_TE"]:
    if name in globals():
        print(f" - {name}: {np.asarray(globals()[name]).shape}")
    else:
        print(f" - {name}: MISSING")

print("\n=== OOF/STACK arrays in memory ===")
for name in ["oof_stack_l","te_stack_l","oof_img2_l","oof_e5_l"]:
    if name in globals():
        print(f" - {name}: {np.asarray(globals()[name]).shape}")
    else:
        print(f" - {name}: (optional) missing")

print("\n=== IMAGE MEMMAPS (B/32 & L/14) ===")
def memmap_info(path, N):
    if not os.path.exists(path): return None
    dim = (os.path.getsize(path)//2)//N
    return dim

if 'df_all' in globals():
    N = len(df_all)
    b32_dim = memmap_info(os.path.join(CKP,"img_emb_all.f16"), N)
    has_b32 = os.path.join(CKP,"img_has.npy")
    if b32_dim:
        print(f" - B/32 dim={b32_dim}", end="")
        if os.path.exists(has_b32):
            h = np.load(has_b32)
            print(f" | coverage={h.mean()*100:.1f}% | flags shape={h.shape}")
        else:
            print(" | flags MISSING")
    else:
        print(" - B/32 memmap MISSING")

    l14_dim = memmap_info(os.path.join(CKP,"img_emb_L14_all.f16"), N)
    has_l14 = os.path.join(CKP,"img_has_L14.npy")
    if l14_dim:
        print(f" - L/14 dim={l14_dim}", end="")
        if os.path.exists(has_l14):
            h2 = np.load(has_l14)
            print(f" | coverage={h2.mean()*100:.1f}% | flags shape={h2.shape}")
        else:
            print(" | flags MISSING")
    else:
        print(" - L/14 memmap MISSING (OK if you haven’t finished encoding)")

print("\n=== E5 TEXT MEMMAP (optional) ===")
e5_path = os.path.join(CKP, "text_e5_emb_all.f16")
if os.path.exists(e5_path) and 'df_all' in globals():
    e5_dim = (os.path.getsize(e5_path)//2)//len(df_all)
    print(f" - E5 memmap dim={e5_dim}")
else:
    print(" - E5 memmap not found here (OK if you’re not using it this run)")

print("\n=== QUICK STACK SHAPE TEST (no training) ===")
try:
    # MiniLM (load all shards quickly)
    emb_all = np.vstack([np.load(p, mmap_mode='r') for p in shards]).astype('float32')
    is_tr = (df_all['is_train'].values==1)
    EMB_TR = emb_all[is_tr]; EMB_TE = emb_all[~is_tr]

    # Build minimal matrices with what we *have* in memory
    def to2(a): a = np.asarray(a); return a if a.ndim==2 else a.reshape(-1,1)

    parts_tr = []
    parts_te = []
    if 'FE_TR' in globals(): parts_tr.append(to2(FE_TR)); parts_te.append(to2(FE_TE))
    parts_tr.append(to2(EMB_TR)); parts_te.append(to2(EMB_TE))
    if 'oof_stack_l' in globals(): parts_tr.append(to2(oof_stack_l)); parts_te.append(to2(te_stack_l))

    # B/32
    if b32_dim:
        IMG_B32_ALL = np.memmap(os.path.join(CKP,"img_emb_all.f16"), dtype='float16', mode='r', shape=(len(df_all), b32_dim))
        HAS_B32 = np.load(os.path.join(CKP,"img_has.npy")).astype('float32')
        parts_tr += [np.array(IMG_B32_ALL[is_tr], 'float32'), HAS_B32[is_tr].reshape(-1,1)]
        parts_te += [np.array(IMG_B32_ALL[~is_tr], 'float32'), HAS_B32[~is_tr].reshape(-1,1)]
    # L/14 (optional)
    if l14_dim:
        IMG_L14_ALL = np.memmap(os.path.join(CKP,"img_emb_L14_all.f16"), dtype='float16', mode='r', shape=(len(df_all), l14_dim))
        HAS_L14 = np.load(os.path.join(CKP,"img_has_L14.npy")).astype('float32')
        parts_tr += [np.array(IMG_L14_ALL[is_tr], 'float32'), HAS_L14[is_tr].reshape(-1,1)]
        parts_te += [np.array(IMG_L14_ALL[~is_tr], 'float32'), HAS_L14[~is_tr].reshape(-1,1)]

    X_tr = np.hstack(parts_tr).astype('float32')
    X_te = np.hstack(parts_te).astype('float32')
    print(" - X_tr:", X_tr.shape, "| X_te:", X_te.shape)
    check(X_tr.shape[0]==len(train) and X_te.shape[0]==len(test), "row counts match train/test")
except Exception as e:
    print("Could not build quick stack:", e)
    ok = False

print("\n=== SAVED OUTPUTS PRESENT ===")
outs = sorted(glob.glob(os.path.join(CKP, "out", "*.csv")))
print(" - submissions/oof files:", len(outs))
for p in outs[:5]:
    print("   •", os.path.basename(p))
if len(outs) > 5:
    print("   …", len(outs)-5, "more")

print("\nREADY:", ok)


=== DATAFRAMES ===
✅ train loaded
✅ test loaded
✅ df_all built (train+test concat)
   train rows: 75000
   test  rows: 75000
   df_all rows: 150000

=== CHECKPOINT FILES (on disk) ===
 - out                 : OK
 - runs                : OK
 - text_emb_shards     : OK
 - img_emb_all.f16     : OK
 - img_has.npy         : OK
 - img_emb_L14_all.f16 : OK
 - img_has_L14.npy     : OK
 - text_e5_emb_all.f16 : OK
 - fe_tr.npy           : OK
 - fe_te.npy           : OK
 - oof_stack_l.npy     : OK
 - te_stack_l.npy      : OK
 - oof_img2_l.npy      : OK
 - oof_e5_l.npy        : OK

=== MINI-LM TEXT EMBEDDINGS (disk) ===
✅ found 15 MiniLM shard(s)
   shard[0] shape: (10000, 384)

=== FE (engineered features) in memory ===
 - FE_TR: (75000, 222)
 - FE_TE: (75000, 222)

=== OOF/STACK arrays in memory ===
 - oof_stack_l: (75000,)
 - te_stack_l: (75000,)
 - oof_img2_l: (75000,)
 - oof_e5_l: (75000,)

=== IMAGE MEMMAPS (B/32 & L/14) ===
 - B/32 dim=512 | coverage=41.7% | flags shape=(150000,)
 - L/14 di

In [7]:
# ===========================
# Meta-blend (Ridge, per-segment interactions) over all saved models
# ===========================
import os, glob, json, time
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

# --- prereqs in memory
assert all(v in globals() for v in ['train','test','df_all']), "Load train/test/df_all first."

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

is_tr = (df_all['is_train'].values==1)
y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 1e-2, None)).astype('float32')

# --- find candidates (OOF + TEST) in current run and checkpoint bundle
CAND_DIRS = [
    "out",
    "/kaggle/input/saved-ml-model/checkpoint/out",
    "/kaggle/input/saved-ml-model/checkpoint"
]

def list_preds():
    oofs, tests = {}, {}
    for base in CAND_DIRS:
        if not os.path.exists(base): continue
        for p in glob.glob(os.path.join(base, "oof_*.csv")):
            name = os.path.splitext(os.path.basename(p))[0]
            oofs[name] = p
        for p in glob.glob(os.path.join(base, "test_predictions*.csv")):
            name = os.path.splitext(os.path.basename(p))[0]
            tests[name] = p
    return oofs, tests

oof_files, te_files = list_preds()
log(f"Found OOF files: {len(oof_files)} | TEST files: {len(te_files)}")

def load_oof_vec(path):
    df = pd.read_csv(path)
    assert 'sample_id' in df.columns
    df = df.set_index('sample_id')
    # common column names across our pipeline
    for c in ['oof_log','oof','oof_log_router','pred_log']:
        if c in df.columns:
            return df[c].reindex(train['sample_id']).values
    # fallback: if only price present, convert to log
    if 'oof_price' in df.columns:
        return np.log(np.clip(df['oof_price'].reindex(train['sample_id']).values, 1e-2, None))
    raise ValueError(f"Unsupported OOF schema in {path}")

def load_test_vec(path):
    df = pd.read_csv(path).set_index('sample_id')
    return df['price'].reindex(test['sample_id']).values

# assemble common set: require both OOF and TEST available (matching model family)
cand = []
for oof_name, oof_path in sorted(oof_files.items()):
    # try to map oof_name -> a test file with same suffix after 'oof_'
    suffix = oof_name.replace("oof_", "")
    # look for test file that endswith suffix
    match_key = None
    for te_name, te_path in te_files.items():
        if te_name.endswith(suffix):
            match_key = te_name; break
    if match_key is None:
        continue
    try:
        oof_l = load_oof_vec(oof_path)
        te_p  = load_test_vec(te_files[match_key])
        cand.append((suffix, oof_l, te_p, oof_path, te_files[match_key]))
        log(f"Using candidate: {suffix} | OOF={os.path.basename(oof_path)} | TEST={os.path.basename(te_files[match_key])}")
    except Exception as e:
        print("Skip", suffix, "->", e)

assert len(cand) >= 2, "Need at least 2 models (OOF+TEST) discovered to meta-blend."

# build feature matrix: log(preds) for each model + segment flag + interactions
eps = 1e-4
F_tr_list, F_te_list, names = [], [], []
for name, oof_l, te_p, _, _ in cand:
    F_tr_list.append(oof_l.reshape(-1,1))             # already log-space
    F_te_list.append(np.log(np.clip(te_p, eps, None)).reshape(-1,1))
    names.append(name)

# segment flag (has_img ∧ missing_qty)
# has_img flags (prefer checkpoint flags if present)
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all = np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

if 'mask_missing_qty' in globals():
    mmq_all = mask_missing_qty.astype(bool)
else:
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    mmq_all = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values

segA_all = (has_img_all & mmq_all)
segA_tr  = segA_all[is_tr].astype(np.float32).reshape(-1,1)
segA_te  = segA_all[~is_tr].astype(np.float32).reshape(-1,1)

# stack base features
Xtr = np.hstack(F_tr_list)           # shape (n, K)
Xte = np.hstack(F_te_list)
# add seg flag and interactions
Xtr_full = np.hstack([Xtr, segA_tr, Xtr*segA_tr])
Xte_full = np.hstack([Xte, segA_te, Xte*segA_te])
log(f"Meta features: base K={len(names)} -> Xtr_full={Xtr_full.shape}, Xte_full={Xte_full.shape}")

# CV ridge
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)

alphas = [0.05, 0.1, 0.2, 0.3, 0.5]
best = {'cv': 1e9}
for a in alphas:
    oof_pred = np.zeros_like(y_log, dtype='float32')
    te_pred  = np.zeros(Xte_full.shape[0], dtype='float32')
    for f,(tr_idx, va_idx) in enumerate(skf.split(Xtr_full, bins)):
        m = Ridge(alpha=a, solver='auto', random_state=42)
        m.fit(Xtr_full[tr_idx], y_log[tr_idx])
        oof_pred[va_idx] = m.predict(Xtr_full[va_idx])
        te_pred += m.predict(Xte_full) / skf.n_splits
    cv = smape(y, np.exp(oof_pred).clip(0.01))
    log(f"[alpha={a}] CV SMAPE={cv:.3f}%")
    if cv < best['cv']:
        best = {'alpha': a, 'cv': cv, 'oof': oof_pred.copy(), 'te': te_pred.copy()}

cv_meta = best['cv']
log(f"Meta-blend best alpha={best['alpha']} | CV={cv_meta:.3f}%")

# save submission + meta
os.makedirs("out", exist_ok=True)
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(best['te']).clip(0.01).astype(float)})
sub_path = "out/test_predictions_meta_blend_ridge.csv"
sub.to_csv(sub_path, index=False)
with open("out/meta_blend_ridge_meta.json","w") as f:
    json.dump({'alpha': best['alpha'], 'cv_smape': float(cv_meta), 'models': names}, f, indent=2)
print("Saved ->", sub_path)


[14:18:56] Found OOF files: 6 | TEST files: 30 | +0.0s
[14:18:56] Using candidate: catboost_lprice | OOF=oof_catboost_lprice.csv | TEST=test_predictions_catboost_lprice.csv | +0.1s
[14:18:56] Using candidate: e5_brand_cluster | OOF=oof_e5_brand_cluster.csv | TEST=test_predictions_e5_brand_cluster.csv | +0.1s
[14:18:56] Using candidate: e5_fused | OOF=oof_e5_fused.csv | TEST=test_predictions_e5_fused.csv | +0.2s
[14:18:56] Using candidate: e5_mono | OOF=oof_e5_mono.csv | TEST=test_predictions_e5_mono.csv | +0.2s
[14:18:56] Using candidate: xgb_best_promoted | OOF=oof_xgb_best_promoted.csv | TEST=test_predictions_xgb_best_promoted.csv | +0.3s


/tmp/ipykernel_37/3720245338.py:107: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mmq_all = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values


[14:18:58] Meta features: base K=5 -> Xtr_full=(75000, 11), Xte_full=(75000, 11) | +2.4s
[14:18:58] [alpha=0.05] CV SMAPE=48.645% | +2.7s
[14:18:58] [alpha=0.1] CV SMAPE=48.645% | +2.8s
[14:18:58] [alpha=0.2] CV SMAPE=48.645% | +2.9s
[14:18:58] [alpha=0.3] CV SMAPE=48.645% | +3.0s
[14:18:59] [alpha=0.5] CV SMAPE=48.645% | +3.0s
[14:18:59] Meta-blend best alpha=0.05 | CV=48.645% | +3.0s
Saved -> out/test_predictions_meta_blend_ridge.csv


In [8]:
# ===========================
# Segment-aware META ridge blend (fixed test index mapping)
# ===========================
import os, glob, time, json
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold

t0=time.time()
log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

assert 'train' in globals() and 'test' in globals() and 'df_all' in globals()
y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 1e-2, None)).astype(np.float32)

# ---------- helpers ----------
def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_test_pairs():
    """Return dict name -> (oof_price, test_price) aligned to train/test sample_id order."""
    cand_dirs = [
        "out",
        "/kaggle/working/out",
        "/kaggle/input/saved-ml-model/checkpoint/out",
        "/kaggle/input/saved-ml-model/checkpoint",
    ]
    pairs = [
        ("oof_e5_brand_cluster.csv", "test_predictions_e5_brand_cluster.csv"),
        ("oof_e5_fused.csv",         "test_predictions_e5_fused.csv"),
        ("oof_xgb_best_promoted.csv","test_predictions_xgb_best_promoted.csv"),
        ("oof_fused_L14.csv",        "test_predictions_fused_L14.csv"),
        ("oof_catboost_lprice.csv",  "test_predictions_catboost_lprice.csv"),
        ("oof_router_lowprice.csv",  "test_predictions_router_blended.csv"),
        ("oof_router.csv",           "test_predictions_router.csv"),
        ("oof_tfidf_ridge.csv",      "test_predictions_tfidf_ridge.csv"),
        ("oof_stack_img_memmap.csv", "test_predictions_stack_img_expanded.csv"),
        ("oof_stack_only.csv",       "test_predictions_stack.csv"),
    ]
    out={}
    train_ids = train['sample_id'].values
    test_ids  = test['sample_id'].values
    for oof_name,test_name in pairs:
        oof_path=test_path=None
        for d in cand_dirs:
            p=os.path.join(d,oof_name)
            if os.path.exists(p): oof_path=p; break
        for d in cand_dirs:
            p=os.path.join(d,test_name)
            if os.path.exists(p): test_path=p; break
        if not oof_path or not test_path:
            continue
        df_oof=pd.read_csv(oof_path).set_index('sample_id')
        col=None
        for c in ["oof_price","oof_log_router","oof_log","oof"]:
            if c in df_oof.columns: col=c; break
        if col is None: 
            continue
        # convert to price if log-based
        if "log" in col:
            oof_price = np.exp(df_oof[col].reindex(train_ids).values).clip(0.01)
        else:
            oof_price = df_oof[col].reindex(train_ids).values
        te_price = pd.read_csv(test_path).set_index('sample_id')['price'].reindex(test_ids).values
        key=os.path.splitext(os.path.basename(oof_path))[0].replace("oof_","")
        out[key]=(oof_price.astype(np.float32), te_price.astype(np.float32))
    return out

log("Loading meta candidates...")
pairs=load_oof_test_pairs()
assert pairs, "No OOF/Test files found. Make sure you saved previous model outputs to /out or checkpoint/out."
names=sorted(pairs.keys())
Xtr = np.column_stack([pairs[k][0] for k in names]).astype(np.float32)
Xte = np.column_stack([pairs[k][1] for k in names]).astype(np.float32)
log(f"Meta features -> {len(names)} models: {names}")
log(f"Shapes: Xtr={Xtr.shape}, Xte={Xte.shape}")

# ---------- segments (A = has_img ∧ missing_qty) ----------
CKP_BASE = "/kaggle/input/saved-ml-model/checkpoint"
if os.path.exists(os.path.join(CKP_BASE,"img_has.npy")):
    has_img_all = np.load(os.path.join(CKP_BASE,"img_has.npy")).astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

if 'mask_missing_qty' in globals():
    mmq = mask_missing_qty
else:
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    mmq = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values

is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
SEG = (has_img_all & mmq)

# --- map GLOBAL indices -> LOCAL positions for train/test ---
N = len(df_all)
tr_glob = np.where(is_tr)[0]
te_glob = np.where(is_te)[0]
tr_pos_map = np.full(N, -1, dtype=np.int32); tr_pos_map[tr_glob] = np.arange(tr_glob.size, dtype=np.int32)
te_pos_map = np.full(N, -1, dtype=np.int32); te_pos_map[te_glob] = np.arange(te_glob.size, dtype=np.int32)

A_tr_glob = np.where(is_tr & SEG)[0];  B_tr_glob = np.where(is_tr & ~SEG)[0]
A_te_glob = np.where(is_te & SEG)[0];  B_te_glob = np.where(is_te & ~SEG)[0]

A_tr = tr_pos_map[A_tr_glob]; A_tr = A_tr[A_tr>=0]
B_tr = tr_pos_map[B_tr_glob]; B_tr = B_tr[B_tr>=0]
A_te = te_pos_map[A_te_glob]; A_te = A_te[A_te>=0]
B_te = te_pos_map[B_te_glob]; B_te = B_te[B_te>=0]

log(f"Segments -> train A={len(A_tr)} B={len(B_tr)} | test A={len(A_te)} B={len(B_te)}")

# ---------- CV ridge per-segment ----------
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
alphas=[0.05,0.1,0.2,0.3,0.5]

def cv_segment(idx):
    # handle tiny/empty segments by falling back to all rows
    if idx.size < 100:
        idx = np.arange(len(y))
    best=(1e9,None,None)
    for a in alphas:
        oof=np.zeros(len(y), dtype=np.float32)
        for f,(tr,va) in enumerate(skf.split(Xtr[idx], bins[idx])):
            m=Ridge(alpha=a, random_state=42)
            m.fit(Xtr[idx][tr], y_log[idx][tr])
            oof[idx[va]] = m.predict(Xtr[idx][va])
        cv = smape(y, np.exp(oof).clip(0.01))
        if cv < best[0]: best=(cv,a,oof)
        log(f"  [alpha={a:.2f}] seg-CV={cv:.3f}%")
    return best

log("Tuning segment A...")
cvA, aA, oofA = cv_segment(A_tr)
log(f"Segment A best: alpha={aA} | seg-CV={cvA:.3f}%")

log("Tuning segment B...")
cvB, aB, oofB = cv_segment(B_tr)
log(f"Segment B best: alpha={aB} | seg-CV={cvB:.3f}%")

# combine OOF
oof_meta_l = np.zeros_like(y_log)
oof_meta_l[A_tr] = oofA[A_tr]
oof_meta_l[B_tr] = oofB[B_tr]
cv_all = smape(y, np.exp(oof_meta_l).clip(0.01))
log(f"Overall META (seg-aware) CV SMAPE: {cv_all:.3f}%")

# ---------- fit on full data per-seg and predict test (LOCAL indices) ----------
mA = Ridge(alpha=aA, random_state=42).fit(Xtr[A_tr], y_log[A_tr])
mB = Ridge(alpha=aB, random_state=42).fit(Xtr[B_tr], y_log[B_tr])

te_log = np.zeros(Xte.shape[0], dtype=np.float32)
te_log[A_te] = mA.predict(Xte[A_te]) if len(A_te)>0 else 0.0
te_log[B_te] = mB.predict(Xte[B_te]) if len(B_te)>0 else 0.0
te_price = np.exp(te_log).clip(0.01)

# ---------- save ----------
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'],
              'oof_log': oof_meta_l,
              'oof_price': np.exp(oof_meta_l).clip(0.01)}).to_csv("out/oof_meta_seg_ridge.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': te_price.astype(float)}).to_csv(
    "out/test_predictions_meta_seg_ridge.csv", index=False)
with open("out/meta_seg_ridge_meta.json","w") as f:
    json.dump({'models_used': names, 'alpha_A': float(aA), 'alpha_B': float(aB),
               'cv_segA': float(cvA), 'cv_segB': float(cvB), 'cv_overall': float(cv_all)}, f, indent=2)
log("Saved -> out/oof_meta_seg_ridge.csv, out/test_predictions_meta_seg_ridge.csv")


[14:18:59] Loading meta candidates... | +0.0s
[14:18:59] Meta features -> 4 models: ['catboost_lprice', 'e5_brand_cluster', 'e5_fused', 'xgb_best_promoted'] | +0.3s
[14:18:59] Shapes: Xtr=(75000, 4), Xte=(75000, 4) | +0.3s


/tmp/ipykernel_37/893938329.py:93: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mmq = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values


[14:19:01] Segments -> train A=23995 B=51005 | test A=22635 B=52365 | +2.4s
[14:19:01] Tuning segment A... | +2.4s
[14:19:01]   [alpha=0.05] seg-CV=129.627% | +2.5s
[14:19:01]   [alpha=0.10] seg-CV=129.627% | +2.5s
[14:19:01]   [alpha=0.20] seg-CV=129.627% | +2.5s
[14:19:01]   [alpha=0.30] seg-CV=129.627% | +2.6s
[14:19:01]   [alpha=0.50] seg-CV=129.627% | +2.6s
[14:19:01] Segment A best: alpha=0.5 | seg-CV=129.627% | +2.6s
[14:19:01] Tuning segment B... | +2.6s
[14:19:01]   [alpha=0.05] seg-CV=88.637% | +2.7s
[14:19:01]   [alpha=0.10] seg-CV=88.637% | +2.7s
[14:19:02]   [alpha=0.20] seg-CV=88.637% | +2.8s
[14:19:02]   [alpha=0.30] seg-CV=88.637% | +2.8s
[14:19:02]   [alpha=0.50] seg-CV=88.637% | +2.9s
[14:19:02] Segment B best: alpha=0.05 | seg-CV=88.637% | +2.9s
[14:19:02] Overall META (seg-aware) CV SMAPE: 56.931% | +2.9s
[14:19:02] Saved -> out/oof_meta_seg_ridge.csv, out/test_predictions_meta_seg_ridge.csv | +3.2s


In [9]:
# ============================================
# OOF-safe kNN price priors (FAISS) + monotone XGB refit  [REVISED]
# ============================================
import os, time, gc, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

# ---- prereq checks ----
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE']
assert all(v in globals() for v in need), "Load FE/EMB/train/test first."

y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 1e-2, None)).astype('float32')
is_tr = (df_all['is_train'].values==1)
is_te = ~is_tr

# ---- try FAISS (CPU) ----
try:
    import faiss
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=False)
    import faiss

t0=time.time()
log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def l2n(X):
    X = np.asarray(X, dtype='float32', order='C')
    n = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
    return X / n

# ---- build unified kNN vector: [E5? | MiniLM | IMG(B/32)?] with block weights ----
blocks_tr, blocks_te = [], []

def add_block(tr, te, w=1.0):
    if tr is None or te is None: return
    blocks_tr.append(l2n(tr) * w)
    blocks_te.append(l2n(te) * w)

# E5 (optional)
if 'E5_TR' in globals() and 'E5_TE' in globals():
    add_block(E5_TR, E5_TE, w=1.0)

# MiniLM (required)
add_block(EMB_TR, EMB_TE, w=0.7)

# Image CLIP B/32 (memmap or in-memory)
def try_load_img_b32():
    # in-memory first
    if 'IMG_TR' in globals() and 'IMG_TE' in globals():
        return np.asarray(IMG_TR, 'float32'), np.asarray(IMG_TE, 'float32')
    # checkpoint memmap
    ckp = "/kaggle/input/saved-ml-model/checkpoint"
    p = os.path.join(ckp, "img_emb_all.f16")
    if os.path.exists(p):
        N = len(df_all); dim = (os.path.getsize(p)//2)//N
        mm = np.memmap(p, dtype='float16', mode='r', shape=(N, dim))
        return np.array(mm[is_tr], 'float32'), np.array(mm[is_te], 'float32')
    # working dir (if not mounted via dataset)
    p2 = "/kaggle/working/img_emb_all.f16"
    if os.path.exists(p2):
        N = len(df_all); dim = (os.path.getsize(p2)//2)//N
        mm = np.memmap(p2, dtype='float16', mode='r', shape=(N, dim))
        return np.array(mm[is_tr], 'float32'), np.array(mm[is_te], 'float32')
    return None, None

img_tr, img_te = try_load_img_b32()
add_block(img_tr, img_te, w=0.6)

V_tr = l2n(np.hstack(blocks_tr)) if blocks_tr else l2n(EMB_TR)
V_te = l2n(np.hstack(blocks_te)) if blocks_te else l2n(EMB_TE)
log(f"kNN vectors -> TR={V_tr.shape}, TE={V_te.shape}")

# ---- FAISS IVF index (cosine via IP on L2-normalized) ----
def build_ivf_index(vectors, nlist=256, metric=faiss.METRIC_INNER_PRODUCT):
    d = int(vectors.shape[1])
    quant = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFFlat(quant, d, int(nlist), metric)
    index.train(vectors)
    index.add(vectors)
    index.nprobe = max(8, min(64, int(nlist)//4))
    return index

def knn_stats_from_neighbors(nei_idx, nei_sim):
    # map neighbor prices
    vals = y[nei_idx]  # (N,k)
    w = np.maximum(nei_sim, 0.0).astype('float32')
    wsum = w.sum(axis=1, keepdims=True) + 1e-9
    wmean = (w*vals).sum(axis=1, keepdims=True)/wsum
    med = np.median(vals, axis=1, keepdims=True)
    p25 = np.percentile(vals, 25, axis=1, keepdims=True)
    p75 = np.percentile(vals, 75, axis=1, keepdims=True)
    mn  = vals.min(axis=1, keepdims=True)
    mx  = vals.max(axis=1, keepdims=True)
    std = vals.std(axis=1, keepdims=True)
    return np.hstack([wmean, med, p25, p75, mn, mx, std]).astype('float32')

# ---- OOF-safe kNN priors ----
k = 32
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)

knn_tr = np.zeros((len(y), 7), dtype='float32')
log("Building OOF kNN features...")
for f,(tr_idx, va_idx) in enumerate(skf.split(V_tr, bins)):
    base = V_tr[tr_idx].copy(order='C')
    val  = V_tr[va_idx].copy(order='C')
    index = build_ivf_index(base, nlist=256)
    sims, ids_local = index.search(val, k)        # (NV,k)
    nei_global = tr_idx[ids_local]                # map local->global
    knn_tr[va_idx] = knn_stats_from_neighbors(nei_global, sims)
    log(f"  fold {f} done")

# ---- Test priors (index on full train) ----
log("Building test kNN features...")
index_full = build_ivf_index(V_tr.copy(order='C'), nlist=512)
sims_te, ids_te = index_full.search(V_te.copy(order='C'), k)
knn_te = knn_stats_from_neighbors(ids_te, sims_te)
log(f"kNN features -> TR={knn_tr.shape}, TE={knn_te.shape}")

# ---- Append to FE and track columns ----
FE_TR = np.hstack([np.asarray(FE_TR,'float32'), knn_tr]).astype('float32')
FE_TE = np.hstack([np.asarray(FE_TE,'float32'), knn_te]).astype('float32')
knn_cols = ['knn_wmean','knn_median','knn_p25','knn_p75','knn_min','knn_max','knn_std']

# Keep fe_columns.csv in sync (best-effort)
fe_cols_path = "/kaggle/working/fe_columns.csv"
try:
    if os.path.exists(fe_cols_path):
        cols = pd.read_csv(fe_cols_path)['feature'].tolist()
        # avoid duplicate append if rerun
        for c in knn_cols:
            if c not in cols:
                cols.append(c)
    else:
        cols = [f"f{i}" for i in range(FE_TR.shape[1]-len(knn_cols))] + knn_cols
    pd.DataFrame({'feature': cols}).to_csv(fe_cols_path, index=False)
except Exception:
    cols = [f"f{i}" for i in range(FE_TR.shape[1])]

# ---- Monotone constraints (+1 for non-decreasing features) ----
mono = np.zeros(FE_TR.shape[1], dtype=int)

def set_plus(names, col_list):
    for name in names:
        if name in col_list:
            mono[col_list.index(name)] = 1

# Non-decreasing in price (priors + physical totals if present)
set_plus(['knn_wmean','knn_median','knn_p25','knn_p75'], cols)
set_plus(['qty_total_mass_g','qty_total_vol_ml','qty_eff_units'], cols)  # if you added OCR/text qty features



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 57.2 MB/s eta 0:00:00
[14:19:12] kNN vectors -> TR=(75000, 1920), TE=(75000, 1920) | +2.2s
[14:19:12] Building OOF kNN features... | +2.2s
[14:20:25]   fold 0 done | +74.8s
[14:21:38]   fold 1 done | +147.9s
[14:22:51]   fold 2 done | +220.5s
[14:24:05]   fold 3 done | +294.4s
[14:25:19]   fold 4 done | +368.7s
[14:25:19] Building test kNN features... | +368.7s
[14:29:23] kNN features -> TR=(75000, 7), TE=(75000, 7) | +612.4s


In [10]:
# XGBoost expects a parenthesized string: "(0,0,1,...)"
mono_str = "(" + ",".join(map(str, mono.tolist())) + ")"
log(f"Monotone +1 count: {(mono==1).sum()} / {len(mono)}")

# ---- Train monotone XGB on log(price) ----
def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=(d!=0); out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae', objective='reg:absoluteerror',
    max_depth=8, min_child_weight=2, learning_rate=0.03,
    monotone_constraints=mono_str,   # <-- FIXED: string form
)

oof_l = np.zeros_like(y_log, 'float32')
te_l  = np.zeros(FE_TE.shape[0], 'float32')

log("Training monotone XGB with kNN priors...")
for f,(tr_idx, va_idx) in enumerate(skf.split(FE_TR, bins)):
    m = xgb.XGBRegressor(**params)
    m.fit(FE_TR[tr_idx], y_log[tr_idx], eval_set=[(FE_TR[va_idx], y_log[va_idx])], verbose=False)
    oof_l[va_idx] = m.predict(FE_TR[va_idx])
    te_l += m.predict(FE_TE)/skf.n_splits
    print(f"  fold {f} SMAPE={smape(y[va_idx], np.exp(oof_l[va_idx]).clip(0.01)):.3f}%")

cv = smape(y, np.exp(oof_l).clip(0.01))
log(f"[kNN-priors + monotone] CV SMAPE: {cv:.3f}%")

# ---- Save
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'],
              'oof_log': oof_l,
              'oof_price': np.exp(oof_l).clip(0.01)}).to_csv("out/oof_xgb_knn_mono.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],
              'price': np.exp(te_l).clip(0.01).astype(float)}).to_csv("out/test_predictions_xgb_knn_mono.csv", index=False)
log("Saved -> out/oof_xgb_knn_mono.csv, out/test_predictions_xgb_knn_mono.csv")


[14:29:23] Monotone +1 count: 4 / 229 | +612.6s
[14:29:23] Training monotone XGB with kNN priors... | +612.6s
  fold 0 SMAPE=48.852%
  fold 1 SMAPE=48.963%
  fold 2 SMAPE=49.341%
  fold 3 SMAPE=48.448%
  fold 4 SMAPE=48.415%
[14:31:31] [kNN-priors + monotone] CV SMAPE: 48.804% | +740.9s
[14:31:31] Saved -> out/oof_xgb_knn_mono.csv, out/test_predictions_xgb_knn_mono.csv | +741.2s


In [11]:
# =============================================
# Step 1 — Retrain E5-fused head with qty monotone constraints
# + segment-aware blend vs best saved base
# =============================================
import os, glob, time, json, gc, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

# ---- prerequisites
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','oof_stack_l','te_stack_l']
for n in need: 
    assert n in globals(), f"Missing {n}"
use_e5 = ('E5_TR' in globals()) and ('E5_TE' in globals())

def to2(a, dtype='float32'):
    a = np.asarray(a)
    return (a if a.ndim==2 else a.reshape(-1,1)).astype(dtype, copy=False)

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# ---- image B/32 memmap + flags (from checkpoint or working)
CKP = "/kaggle/input/saved-ml-model/checkpoint"
def pick_path(fname):
    for base in [CKP, "/kaggle/working"]:
        p = os.path.join(base, fname)
        if os.path.exists(p): return p
    return None

b32_path = pick_path("img_emb_all.f16")
has_path = pick_path("img_has.npy")
if b32_path and has_path:
    N = len(df_all); b32_dim = (os.path.getsize(b32_path)//2)//N
    MM_B32 = np.memmap(b32_path, dtype='float16', mode='r', shape=(N, b32_dim))
    HAS_ALL = np.load(has_path).astype(np.float32)
    is_tr = (df_all['is_train'].values==1)
    IMG_TR = np.array(MM_B32[is_tr], 'float32')
    IMG_TE = np.array(MM_B32[~is_tr], 'float32')
    HAS_TR = HAS_ALL[is_tr].reshape(-1,1).astype('float32')
    HAS_TE = HAS_ALL[~is_tr].reshape(-1,1).astype('float32')
    log(f"B/32 loaded: dim={b32_dim} | TR={IMG_TR.shape} TE={IMG_TE.shape} | coverage={HAS_ALL.mean()*100:.1f}%")
else:
    # fallback: no images
    IMG_TR = np.zeros((len(train),0), 'float32'); IMG_TE = np.zeros((len(test),0), 'float32')
    HAS_TR = np.zeros((len(train),1), 'float32'); HAS_TE = np.zeros((len(test),1), 'float32')
    log("B/32 not found → proceeding without image vectors.")

# ---- locate qty feature indices inside FE
qty_names = ['qty_has','qty_total_mass_g','qty_total_vol_ml','qty_eff_units']
fe_cols_path = "/kaggle/working/fe_columns.csv"
qty_idx_in_FE = None

if os.path.exists(fe_cols_path):
    fe_names = pd.read_csv(fe_cols_path)['feature'].tolist()
    name_to_idx = {n:i for i,n in enumerate(fe_names)}
    if all(n in name_to_idx for n in qty_names):
        qty_idx_in_FE = [name_to_idx['qty_has'],
                         name_to_idx['qty_total_mass_g'],
                         name_to_idx['qty_total_vol_ml'],
                         name_to_idx['qty_eff_units']]
        log(f"Found qty columns in fe_columns.csv at indices {qty_idx_in_FE}")
        
if qty_idx_in_FE is None:
    # fallback: we appended exactly 4 qty features at the end
    nfe = np.asarray(FE_TR).shape[1]
    qty_idx_in_FE = [nfe-4, nfe-3, nfe-2, nfe-1]
    log(f"Using FE tail as qty indices {qty_idx_in_FE} (last 4 columns)")

# interpret which of those are monotone +1
# (qty_has is binary flag → no monotone; mass/vol/eff_units → +1)
idx_qty_has, idx_mass, idx_vol, idx_eff = qty_idx_in_FE
monotone_in_FE = {idx_mass:1, idx_vol:1, idx_eff:1}  # no constraint on qty_has

# ---- build design matrices: [FE | MiniLM | (E5?) | oof_stack | IMG | HAS]
X_tr = np.hstack([
    to2(FE_TR), to2(EMB_TR),
    (to2(E5_TR) if use_e5 else np.zeros((len(train),0), 'float32')),
    to2(oof_stack_l),
    to2(IMG_TR), to2(HAS_TR)
]).astype('float32')

X_te = np.hstack([
    to2(FE_TE), to2(EMB_TE),
    (to2(E5_TE) if use_e5 else np.zeros((len(test),0), 'float32')),
    to2(te_stack_l),
    to2(IMG_TE), to2(HAS_TE)
]).astype('float32')

log(f"Matrices ready: X_tr={X_tr.shape}, X_te={X_te.shape}")

# ---- build monotone constraint vector aligned to X columns
nFE   = np.asarray(FE_TR).shape[1]
nEMB  = np.asarray(EMB_TR).shape[1]
nE5   = (np.asarray(E5_TR).shape[1] if use_e5 else 0)
nOOF  = 1
nIMG  = IMG_TR.shape[1]
nHAS  = 1

mono = [0]*nFE
for k,v in monotone_in_FE.items():
    if 0 <= k < nFE: mono[k] = v
mono += [0]*nEMB + [0]*nE5 + [0]*nOOF + [0]*nIMG + [0]*nHAS
assert len(mono) == X_tr.shape[1]
mono_str = "(" + ",".join(map(str, mono)) + ")"
log(f"Monotone constraints set on FE indices {sorted(monotone_in_FE.keys())} → vector length {len(mono)}")

# ---- targets / CV
y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 1e-2, None)).astype('float32')
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)

params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae', objective='reg:absoluteerror',
    max_depth=8, min_child_weight=2, learning_rate=0.03,
    monotone_constraints=mono_str
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_mono_l = np.zeros_like(y_log); te_mono_l = np.zeros(X_te.shape[0], dtype='float32')

log("Training E5-fused head with qty monotone constraints...")
for f,(tr,va) in enumerate(skf.split(X_tr, bins)):
    t_fold = time.time()
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr[tr], y_log[tr], eval_set=[(X_tr[va], y_log[va])], verbose=False)
    oof_mono_l[va] = m.predict(X_tr[va])
    te_mono_l += m.predict(X_te)/skf.n_splits
    print(f"  [fold {f}] SMAPE={smape(y[va], np.exp(oof_mono_l[va]).clip(0.01)):.3f}% | time={time.time()-t_fold:.1f}s")

cv_mono = smape(y, np.exp(oof_mono_l).clip(0.01))
log(f"[E5-fused + qty-monotone] CV SMAPE: {cv_mono:.3f}%")

# ---- save this head
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'],
              'oof_log': oof_mono_l,
              'oof_price': np.exp(oof_mono_l).clip(0.01)}).to_csv("out/oof_e5_qty_mono.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],
              'price': np.exp(te_mono_l).clip(0.01).astype(float)}).to_csv("out/test_predictions_e5_qty_mono.csv", index=False)
log("Saved -> out/oof_e5_qty_mono.csv, out/test_predictions_e5_qty_mono.csv")

# ===== Segment-aware blend vs best saved base =====
def find_one(patterns, dirs=("out", os.path.join(CKP,"out"), CKP)):
    for d in dirs:
        for pat in patterns:
            p = os.path.join(d, pat)
            if os.path.exists(p):
                return p
    return None

base_oof_csv  = find_one(["oof_e5_brand_cluster.csv","oof_e5_fused.csv","oof_xgb_best_promoted.csv"])
base_test_csv = find_one(["test_predictions_e5_brand_cluster.csv",
                          "test_predictions_e5_fused.csv",
                          "test_predictions_xgb_best_promoted.csv"])
if base_oof_csv and base_test_csv:
    log(f"Blending vs base:\n  OOF : {base_oof_csv}\n  TEST: {base_test_csv}")
    # load base
    base_oof = pd.read_csv(base_oof_csv).set_index('sample_id')
    col = next(c for c in ["oof_price","oof_log_router","oof_log","oof"] if c in base_oof.columns)
    base_price = (np.exp(base_oof[col]) if "log" in col else base_oof[col]).reindex(train['sample_id']).values.astype('float32')
    base_test  = pd.read_csv(base_test_csv).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

    # segments: has_img ∧ missing_qty  (use qty_has == 0 as improved missing-qty test)
    qty_has_all = np.concatenate([FE_TR[:, idx_qty_has], FE_TE[:, idx_qty_has]]).reshape(-1)
    has_img_all = HAS_ALL.astype(bool) if b32_path and has_path else (df_all['image_link'].fillna('')!='').values
    is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
    SEG_A = has_img_all & (qty_has_all==0)
    A_tr = np.where(is_tr & SEG_A)[0]; B_tr = np.where(is_tr & ~SEG_A)[0]
    A_te = np.where(is_te & SEG_A)[0]; B_te = np.where(is_te & ~SEG_A)[0]
    log(f"Segments | train A={len(A_tr)} B={len(B_tr)} | test A={len(A_te)} B={len(B_te)}")

    mono_price = np.exp(oof_mono_l).clip(0.01)
    def scan(idx):
        best=(1e9,1.0)
        for w in np.linspace(0.0,1.0,51):
            v = smape(y[idx], w*mono_price[idx] + (1-w)*base_price[idx])
            if v<best[0]: best=(v,w)
        return best
    (cvA,wA),(cvB,wB) = scan(A_tr), scan(B_tr)
    blend_tr = np.zeros_like(mono_price)
    blend_tr[A_tr] = wA*mono_price[A_tr] + (1-wA)*base_price[A_tr]
    blend_tr[B_tr] = wB*mono_price[B_tr] + (1-wB)*base_price[B_tr]
    cv_blend = smape(y, blend_tr)
    log(f"Segment A: w={wA:.2f} | seg-CV={cvA:.3f}%")
    log(f"Segment B: w={wB:.2f} | seg-CV={cvB:.3f}%")
    log(f"Overall blended CV SMAPE: {cv_blend:.3f}%")

    # save blended test
    te_mono_p = np.exp(te_mono_l).clip(0.01)
    te_blend = np.zeros_like(base_test)
    te_blend[A_te] = wA*te_mono_p[A_te] + (1-wA)*base_test[A_te]
    te_blend[B_te] = wB*te_mono_p[B_te] + (1-wB)*base_test[B_te]
    sub_path = "out/test_predictions_e5_qty_mono_segblend.csv"
    pd.DataFrame({'sample_id': test['sample_id'], 'price': np.clip(te_blend, 5e-4, None).astype(float)}).to_csv(sub_path, index=False)
    with open("out/e5_qty_mono_segblend_meta.json","w") as f:
        json.dump({'cv_mono': float(cv_mono), 'cv_blend': float(cv_blend),
                   'wA': float(wA), 'wB': float(wB),
                   'qty_idx_in_FE': qty_idx_in_FE}, f, indent=2)
    log(f"Saved -> {sub_path}, out/e5_qty_mono_segblend_meta.json")
else:
    log("Base OOF/TEST not found; skipping blend. (Your monotone head files are saved.)")


[14:31:32] B/32 loaded: dim=512 | TR=(75000, 512) TE=(75000, 512) | coverage=41.7% | +0.4s
[14:31:32] Using FE tail as qty indices [225, 226, 227, 228] (last 4 columns) | +0.4s
[14:31:33] Matrices ready: X_tr=(75000, 2151), X_te=(75000, 2151) | +1.3s
[14:31:33] Monotone constraints set on FE indices [226, 227, 228] → vector length 2151 | +1.3s
[14:31:33] Training E5-fused head with qty monotone constraints... | +1.3s
  [fold 0] SMAPE=47.827% | time=170.5s
  [fold 1] SMAPE=47.849% | time=117.2s
  [fold 2] SMAPE=48.400% | time=133.8s
  [fold 3] SMAPE=47.694% | time=128.5s
  [fold 4] SMAPE=47.319% | time=178.5s
[14:43:41] [E5-fused + qty-monotone] CV SMAPE: 47.818% | +729.9s
[14:43:42] Saved -> out/oof_e5_qty_mono.csv, out/test_predictions_e5_qty_mono.csv | +730.2s
[14:43:42] Blending vs base:
  OOF : /kaggle/input/saved-ml-model/checkpoint/out/oof_e5_brand_cluster.csv
  TEST: /kaggle/input/saved-ml-model/checkpoint/out/test_predictions_e5_brand_cluster.csv | +730.2s
[14:43:42] Segments |

/tmp/ipykernel_37/2815245519.py:24: RuntimeWarning: Mean of empty slice.
  return out.mean()*100
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


IndexError: index 75000 is out of bounds for axis 0 with size 75000

In [12]:
# --- rebuild SEG_A and get TEST-LOCAL indices ---
import numpy as np, os, pandas as pd

is_tr = (df_all['is_train'].values == 1)
is_te = ~is_tr

# has_img flag
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all = np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

# missing_qty flag (prefer qty_has feature if you have it)
qty_idx_in_FE = None
fe_cols_path = "/kaggle/working/fe_columns.csv"
if os.path.exists(fe_cols_path):
    fe_names = pd.read_csv(fe_cols_path)['feature'].tolist()
    if 'qty_has' in fe_names:
        qty_idx_in_FE = fe_names.index('qty_has')

if qty_idx_in_FE is not None:
    qty_has_all = np.concatenate([FE_TR[:, qty_idx_in_FE], FE_TE[:, qty_idx_in_FE]]).reshape(-1)
    missing_qty = (qty_has_all == 0)
else:
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    missing_qty = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b',
                                  regex=True).values

SEG_A = has_img_all & missing_qty

# TEST-LOCAL indices (0..len(test)-1), not global df_all indices
A_te = np.where(SEG_A[is_te])[0]
B_te = np.where((~SEG_A)[is_te])[0]

# sanity
assert A_te.max(initial=0) < len(base_test) and B_te.max(initial=0) < len(base_test), \
    f"Segment indices exceed test size: maxA={A_te.max()}, maxB={B_te.max()}, test={len(base_test)}"


/tmp/ipykernel_37/330385855.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  missing_qty = ~t.str.contains(r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b',


In [13]:
te_mono_p = np.exp(te_mono_l).clip(0.01)
te_blend = np.zeros_like(base_test, dtype=float)
te_blend[A_te] = wA*te_mono_p[A_te] + (1.0-wA)*base_test[A_te]
te_blend[B_te] = wB*te_mono_p[B_te] + (1.0-wB)*base_test[B_te]

sub_path = "out/test_predictions_e5_qty_mono_segblend.csv"
pd.DataFrame({'sample_id': test['sample_id'],
              'price': np.clip(te_blend, 5e-4, None).astype(float)}).to_csv(sub_path, index=False)
print("Saved ->", sub_path)


Saved -> out/test_predictions_e5_qty_mono_segblend.csv


In [14]:
# === Quick segment-aware blend: new kNN+mono vs best base (index-safe) ===
import os, json, time, numpy as np, pandas as pd

t0=time.time(); log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

assert 'train' in globals() and 'test' in globals() and 'df_all' in globals()
y = train['price'].astype(float).values

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# --- find files helper
CKP="/kaggle/input/saved-ml-model/checkpoint"
CANDS=["out", os.path.join(CKP,"out"), CKP]

def find_one(names):
    for d in CANDS:
        for n in names:
            p=os.path.join(d,n)
            if os.path.exists(p): return p
    return None

# --- NEW model (you just trained)
new_oof = find_one(["oof_xgb_knn_mono.csv"])
new_te  = find_one(["test_predictions_xgb_knn_mono.csv"])
assert new_oof and new_te, "Couldn't find the new kNN+mono OOF/TEST."

# --- BASE model to blend against (prefers your strongest saved)
base_oof = find_one([
    "oof_e5_brand_cluster.csv","oof_e5_fused.csv","oof_e5_qty_mono.csv","oof_xgb_best_promoted.csv"
])
base_te  = find_one([
    "test_predictions_e5_brand_cluster.csv","test_predictions_e5_fused.csv",
    "test_predictions_e5_qty_mono.csv","test_predictions_xgb_best_promoted.csv"
])
assert base_oof and base_te, "Couldn't find a base OOF/TEST CSV to blend against."

log(f"NEW OOF : {new_oof}")
log(f"BASE OOF: {base_oof}")

# --- load OOF/TEST prices (auto-handle log columns)
def load_oof_price(path):
    df=pd.read_csv(path).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            v = np.exp(v) if "log" in c else v
            return np.nan_to_num(v, nan=np.nanmedian(v)).astype('float32')
    raise RuntimeError(f"No OOF column in {path}")

def load_test_price(path):
    v = pd.read_csv(path).set_index('sample_id')['price'].reindex(test['sample_id']).values
    return np.nan_to_num(v, nan=np.nanmedian(v)).astype('float32')

p_new_oof = load_oof_price(new_oof)
p_new_te  = load_test_price(new_te)
p_base_oof= load_oof_price(base_oof)
p_base_te = load_test_price(base_te)

# --- build segments: has_img ∧ missing_qty
is_tr = (df_all['is_train'].values==1)
is_te = ~is_tr

# has_img
has_img_path = None
for cand in [os.path.join(CKP,"img_has.npy"), "img_has.npy", "/kaggle/working/img_has.npy"]:
    if os.path.exists(cand): has_img_path=cand; break
if has_img_path:
    has_img_all = np.load(has_img_path).astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

# missing_qty: prefer qty_has from FE if present, else regex fallback
idx_qty_has = None
fe_cols_path="/kaggle/working/fe_columns.csv"
if os.path.exists(fe_cols_path):
    fe_names = pd.read_csv(fe_cols_path)['feature'].tolist()
    if 'qty_has' in fe_names:
        idx_qty_has = fe_names.index('qty_has')

if idx_qty_has is not None and 'FE_TR' in globals() and 'FE_TE' in globals():
    qty_has_all = np.concatenate([FE_TR[:,idx_qty_has], FE_TE[:,idx_qty_has]]).reshape(-1)
    missing_qty = (qty_has_all==0)
else:
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    missing_qty = ~t.str.contains(
        r'\b(\d+(\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b',
        regex=True
    ).values

SEG_A = has_img_all & missing_qty

# ---- IMPORTANT: use LOCAL indices (relative to train/test arrays)
A_tr = np.where(SEG_A[is_tr])[0]
B_tr = np.where((~SEG_A)[is_tr])[0]
A_te = np.where(SEG_A[is_te])[0]
B_te = np.where((~SEG_A)[is_te])[0]
log(f"Segments | train A={len(A_tr)}, B={len(B_tr)} | test A={len(A_te)}, B={len(B_te)}")

# --- scan weights per segment (OOF side)
def best_w(idx):
    best_cv, best_w = 1e9, 0.0
    if len(idx)==0:  # edge safety
        return best_cv, 0.5
    for w in np.linspace(0.0,1.0,51):
        blend = w*p_new_oof[idx] + (1-w)*p_base_oof[idx]
        cv = smape(y[idx], blend)
        if cv < best_cv:
            best_cv, best_w = cv, w
    return best_cv, best_w

cvA, wA = best_w(A_tr)
cvB, wB = best_w(B_tr)

blend_tr = np.zeros_like(p_new_oof)
blend_tr[A_tr] = wA*p_new_oof[A_tr] + (1-wA)*p_base_oof[A_tr]
blend_tr[B_tr] = wB*p_new_oof[B_tr] + (1-wB)*p_base_oof[B_tr]
cv_all = smape(y, blend_tr)

log(f"Segment A: best w={wA:.2f} | seg-CV={cvA:.3f}%")
log(f"Segment B: best w={wB:.2f} | seg-CV={cvB:.3f}%")
log(f"Overall blended CV SMAPE: {cv_all:.3f}%")

# --- make submission (TEST side uses LOCAL indices too)
eps=5e-4
te_blend = np.zeros_like(p_base_te)
if len(A_te): te_blend[A_te] = wA*p_new_te[A_te] + (1-wA)*p_base_te[A_te]
if len(B_te): te_blend[B_te] = wB*p_new_te[B_te] + (1-wB)*p_base_te[B_te]
te_blend = np.clip(te_blend, eps, None)

os.makedirs("out", exist_ok=True)
sub_path = "out/test_predictions_knnmono_vs_base_segblend.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': te_blend.astype(float)}).to_csv(sub_path, index=False)
with open("out/knnmono_vs_base_segblend_meta.json","w") as f:
    json.dump({'new_oof':new_oof,'new_test':new_te,'base_oof':base_oof,'base_test':base_te,
               'wA':float(wA),'wB':float(wB),'cvA':float(cvA),'cvB':float(cvB),'cv_all':float(cv_all)}, f, indent=2)
log(f"Saved -> {sub_path}")


[14:44:31] NEW OOF : out/oof_xgb_knn_mono.csv | +0.0s
[14:44:31] BASE OOF: out/oof_e5_qty_mono.csv | +0.0s


/tmp/ipykernel_37/1528911406.py:88: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  missing_qty = ~t.str.contains(


[14:44:33] Segments | train A=23995, B=51005 | test A=22635, B=52365 | +2.3s
[14:44:33] Segment A: best w=0.52 | seg-CV=52.497% | +2.3s
[14:44:33] Segment B: best w=0.32 | seg-CV=44.811% | +2.3s
[14:44:33] Overall blended CV SMAPE: 47.270% | +2.3s
[14:44:33] Saved -> out/test_predictions_knnmono_vs_base_segblend.csv | +2.5s


In [15]:
# === Calibrate your latest blended submission per segment (A/B) — FIXED INDICES ===
import numpy as np, pandas as pd, json, os

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# ---------- Segments (has_img ∧ missing_qty) ----------
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
pattern = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
missing_qty_all = ~t.str.contains(pattern, regex=True).values

CKP = "/kaggle/input/saved-ml-model/checkpoint"
if os.path.exists(os.path.join(CKP,"img_has.npy")):
    has_img_all = np.load(os.path.join(CKP,"img_has.npy")).astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
n_tr  = len(train); n_te = len(test)

SEG_A = has_img_all & missing_qty_all
A_tr_abs = np.where(is_tr & SEG_A)[0]
B_tr_abs = np.where(is_tr & ~SEG_A)[0]
A_te_abs = np.where(is_te & SEG_A)[0]
B_te_abs = np.where(is_te & ~SEG_A)[0]

# map test indices from absolute -> local
A_te = (A_te_abs - n_tr).astype(int)
B_te = (B_te_abs - n_tr).astype(int)
# safety clamp
A_te = A_te[(A_te >= 0) & (A_te < n_te)]
B_te = B_te[(B_te >= 0) & (B_te < n_te)]

A_tr = A_tr_abs  # train stays absolute for OOF arrays (len = n_tr)
B_tr = B_tr_abs

print(f"Segments | train A={len(A_tr)}, B={len(B_tr)} | test A={len(A_te)}, B={len(B_te)}")

# ---------- Load the two models you blended (OOF & TEST prices) ----------
def load_oof_price(path):
    df=pd.read_csv(path).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"No OOF column in {path}")

def load_test_price(path):
    return pd.read_csv(path).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

p_new_oof = load_oof_price("out/oof_xgb_knn_mono.csv")
p_new_te  = load_test_price("out/test_predictions_xgb_knn_mono.csv")
p_base_oof= load_oof_price("out/oof_e5_qty_mono.csv")
p_base_te = load_test_price("out/test_predictions_e5_qty_mono.csv")

# ---------- Re-scan best weights per segment on OOF ----------
y = train['price'].values.astype(float)

def best_w(idx):
    best=(1e9,0.0)
    for w in np.linspace(0.0,1.0,51):
        cv = smape(y[idx], w*p_new_oof[idx] + (1-w)*p_base_oof[idx])
        if cv<best[0]: best=(cv,w)
    return best

(cvA,wA) = best_w(A_tr)
(cvB,wB) = best_w(B_tr)
print(f"Best weights: segA w={wA:.2f} (CV={cvA:.3f}%), segB w={wB:.2f} (CV={cvB:.3f}%)")

# ---------- Build OOF and TEST blends (TEST uses LOCAL indices) ----------
oo = np.zeros_like(y, dtype=np.float32)
oo[A_tr] = wA*p_new_oof[A_tr] + (1-wA)*p_base_oof[A_tr]
oo[B_tr] = wB*p_new_oof[B_tr] + (1-wB)*p_base_oof[B_tr]
print("Pre-cal CV :", smape(y, oo))

te_blend = np.zeros(n_te, dtype=np.float32)
te_blend[A_te] = wA*p_new_te[A_te] + (1-wA)*p_base_te[A_te]
te_blend[B_te] = wB*p_new_te[B_te] + (1-wB)*p_base_te[B_te]

# ---------- Fit log-linear calibration per segment on OOF ----------
def fit_ab(y_true, y_pred):
    t = np.log(np.clip(y_true, 1e-3, None))
    p = np.log(np.clip(y_pred, 1e-3, None))
    X = np.column_stack([p, np.ones_like(p)])
    a,b = np.linalg.lstsq(X, t, rcond=None)[0]
    return float(a), float(b)

aA,bA = fit_ab(y[A_tr], oo[A_tr]) if len(A_tr) else (1.0,0.0)
aB,bB = fit_ab(y[B_tr], oo[B_tr]) if len(B_tr) else (1.0,0.0)
print(f"Calibration params: segA a={aA:.4f}, b={bA:.4f} | segB a={aB:.4f}, b={bB:.4f}")

def apply_ab(pred, a, b):
    return np.exp(a*np.log(np.clip(pred,1e-3,None)) + b)

oo_cal = oo.copy()
if len(A_tr): oo_cal[A_tr] = apply_ab(oo[A_tr], aA, bA)
if len(B_tr): oo_cal[B_tr] = apply_ab(oo[B_tr], aB, bB)
print("Post-cal CV:", smape(y, oo_cal))

# ---------- Apply calibration to TEST (LOCAL indices) ----------
te_cal = te_blend.copy()
if len(A_te): te_cal[A_te] = apply_ab(te_blend[A_te], aA, bA)
if len(B_te): te_cal[B_te] = apply_ab(te_blend[B_te], aB, bB)
te_cal = np.clip(te_cal, 5e-4, None)

# ---------- Save ----------
os.makedirs("out", exist_ok=True)
sub_path = "out/test_predictions_knnmonobasecalibrated 47.1296.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': te_cal.astype(float)}).to_csv(sub_path, index=False)
with open("out/knnmono_segblend_cal_meta.json","w") as f:
    json.dump({'wA':wA,'wB':wB,'aA':aA,'bA':bA,'aB':aB,'bB':bB}, f, indent=2)
print("Saved ->", sub_path)


Segments | train A=23995, B=51005 | test A=22635, B=52365
Best weights: segA w=0.52 (CV=52.497%), segB w=0.32 (CV=44.811%)
Pre-cal CV : 47.27033757635776
Calibration params: segA a=0.9952, b=0.0008 | segB a=1.0143, b=-0.0387
Post-cal CV: 47.220873775660195
Saved -> out/test_predictions_knnmonobasecalibrated 47.1296.csv


In [16]:
# ===============================
# Segment-aware convex blend (2–3 models) → per-seg log calibration
# Optimizes SMAPE directly, weights ≥0 and sum to 1
# ===============================
import os, numpy as np, pandas as pd, itertools, json

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_price(path):
    df=pd.read_csv(path).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"No OOF column in {path}")

def load_test_price(path):
    return pd.read_csv(path).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

# -------- collect candidates (price space) --------
cand = []
# base
cand.append(("base",    "out/oof_e5_qty_mono.csv",       "out/test_predictions_e5_qty_mono.csv"))
# knn-mono
cand.append(("knnmono", "out/oof_xgb_knn_mono.csv",      "out/test_predictions_xgb_knn_mono.csv"))
# router (optional)
if   os.path.exists("out/oof_router_lowprice.csv") and os.path.exists("out/test_predictions_router_blended.csv"):
    cand.append(("router","out/oof_router_lowprice.csv","out/test_predictions_router_blended.csv"))
elif os.path.exists("out/oof_router.csv") and os.path.exists("out/test_predictions_router.csv"):
    cand.append(("router","out/oof_router.csv","out/test_predictions_router.csv"))

names=[]; OOF=[]; TE=[]
for name,oof_p,te_p in cand:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        names.append(name)
        OOF.append(load_oof_price(oof_p))
        TE.append(load_test_price(te_p))
assert len(names)>=2, "Need at least 2 models' OOF/Test files in /out."

Xtr = np.column_stack(OOF).astype('float32')   # (n_tr, K)
Xte = np.column_stack(TE).astype('float32')    # (n_te, K)
y   = train['price'].astype(float).values
print(f"Models in convex blend: {names} | Xtr={Xtr.shape}, Xte={Xte.shape}")

# -------- segments (A = has_img ∧ missing_qty) with correct TEST mapping --------
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
pattern = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
missing_qty_all = ~t.str.contains(pattern, regex=True).values

CKP = "/kaggle/input/saved-ml-model/checkpoint"
if os.path.exists(os.path.join(CKP,"img_has.npy")):
    has_img_all = np.load(os.path.join(CKP,"img_has.npy")).astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
n_tr, n_te = len(train), len(test)
SEG_A = has_img_all & missing_qty_all

A_tr = np.where(is_tr & SEG_A)[0]
B_tr = np.where(is_tr & ~SEG_A)[0]
A_te = (np.where(is_te & SEG_A)[0] - n_tr).astype(int)
B_te = (np.where(is_te & ~SEG_A)[0] - n_tr).astype(int)
A_te = A_te[(A_te>=0)&(A_te<n_te)]
B_te = B_te[(B_te>=0)&(B_te<n_te)]
print(f"Segments | train A={len(A_tr)}, B={len(B_tr)} | test A={len(A_te)}, B={len(B_te)}")

# -------- convex weight search (SMAPE) for a given index set --------
def convex_weights_for(idx, coarse_step=0.1, fine_halfspan=0.12, fine_step=0.02):
    K = Xtr.shape[1]
    Xs = Xtr[idx]; ys = y[idx]
    best = (1e9, None)
    # coarse simplex grid
    if K == 2:
        for w0 in np.arange(0,1+1e-9, coarse_step):
            w = np.array([w0, 1-w0], dtype=np.float32)
            cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
            if cv < best[0]: best=(cv, w.copy())
        # refine around best
        w0b = best[1][0]
        lo = max(0.0, w0b - fine_halfspan); hi = min(1.0, w0b + fine_halfspan)
        for w0 in np.arange(lo, hi+1e-9, fine_step):
            w = np.array([w0, 1-w0], dtype=np.float32)
            cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
            if cv < best[0]: best=(cv, w.copy())
    else:
        # K == 3 (if router exists)
        # coarse
        grid = np.arange(0,1+1e-9, coarse_step)
        for w0 in grid:
            for w1 in grid:
                w2 = 1 - w0 - w1
                if w2 < -1e-9: continue
                w = np.array([w0, w1, w2], dtype=np.float32)
                cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
                if cv < best[0]: best=(cv, w.copy())
        # refine
        w_b = best[1]; w0c,w1c = w_b[0], w_b[1]
        def clamp01(x): return max(0.0, min(1.0, x))
        w0_lo, w0_hi = clamp01(w0c - fine_halfspan), clamp01(w0c + fine_halfspan)
        w1_lo, w1_hi = clamp01(w1c - fine_halfspan), clamp01(w1c + fine_halfspan)
        for w0 in np.arange(w0_lo, w0_hi+1e-9, fine_step):
            for w1 in np.arange(w1_lo, w1_hi+1e-9, fine_step):
                w2 = 1 - w0 - w1
                if w2 < -1e-9: continue
                w = np.array([w0, w1, w2], dtype=np.float32)
                cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
                if cv < best[0]: best=(cv, w.copy())
    return best  # (cv, w)

cvA,(wA) = convex_weights_for(A_tr)
cvB,(wB) = convex_weights_for(B_tr)
print(f"Best weights: segA w={wA} (CV={cvA:.3f}%), segB w={wB} (CV={cvB:.3f}%)")

# stitch OOF and compute overall CV
oo = np.zeros(n_tr, dtype=np.float32)
oo[A_tr] = np.clip(Xtr[A_tr] @ wA, 5e-4, None)
oo[B_tr] = np.clip(Xtr[B_tr] @ wB, 5e-4, None)
cv_all = smape(y, oo)
print("Pre-cal CV :", cv_all)

# -------- per-seg log calibration (as before) --------
def fit_ab(y_true, y_pred):
    t = np.log(np.clip(y_true, 1e-3, None))
    p = np.log(np.clip(y_pred, 1e-3, None))
    X = np.column_stack([p, np.ones_like(p)])
    a,b = np.linalg.lstsq(X, t, rcond=None)[0]
    return float(a), float(b)

def apply_ab(pred, a, b):
    return np.exp(a*np.log(np.clip(pred,1e-3,None)) + b)

aA,bA = fit_ab(y[A_tr], oo[A_tr]) if len(A_tr) else (1.0,0.0)
aB,bB = fit_ab(y[B_tr], oo[B_tr]) if len(B_tr) else (1.0,0.0)
oo_cal = oo.copy()
if len(A_tr): oo_cal[A_tr] = apply_ab(oo[A_tr], aA, bA)
if len(B_tr): oo_cal[B_tr] = apply_ab(oo[B_tr], aB, bB)
print("Post-cal CV:", smape(y, oo_cal))

# -------- predict test (blend + calibration) --------
te = np.zeros(n_te, dtype=np.float32)
te[A_te] = np.clip(Xte[A_te] @ wA, 5e-4, None)
te[B_te] = np.clip(Xte[B_te] @ wB, 5e-4, None)
te_cal = te.copy()
if len(A_te): te_cal[A_te] = apply_ab(te[A_te], aA, bA)
if len(B_te): te_cal[B_te] = apply_ab(te[B_te], aB, bB)
te_cal = np.clip(te_cal, 5e-4, None)

# -------- save --------
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_cal}).to_csv(
    "out/47.12969759165666_meta_convex_seg_cal.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': te_cal.astype(float)}).to_csv(
    "out/test_predictions_meta_convex_seg_cal.csv", index=False)
with open("out/meta_convex_seg_cal_meta.json","w") as f:
    json.dump({
        'models': names,
        'wA': wA.tolist(), 'wB': wB.tolist(),
        'cv_segA_pre': float(cvA), 'cv_segB_pre': float(cvB), 'cv_all_pre': float(cv_all),
        'aA': aA, 'bA': bA, 'aB': aB, 'bB': bB
    }, f, indent=2)
print("Saved -> out/test_predictions_meta_convex_seg_cal.csv")


Models in convex blend: ['base', 'knnmono'] | Xtr=(75000, 2), Xte=(75000, 2)
Segments | train A=23995, B=51005 | test A=22635, B=52365
Best weights: segA w=[0.48 0.52] (CV=52.497%), segB w=[0.68       0.32000002] (CV=44.811%)
Pre-cal CV : 47.27033761331014
Post-cal CV: 47.22087363441564
Saved -> out/test_predictions_meta_convex_seg_cal.csv


In [17]:
# ===============================
# Segment-aware convex blend (2–3 models) → per-seg log calibration
# Optimizes SMAPE directly, weights ≥0 and sum to 1
# ===============================
import os, numpy as np, pandas as pd, itertools, json

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_price(path):
    df=pd.read_csv(path).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"No OOF column in {path}")

def load_test_price(path):
    return pd.read_csv(path).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

# -------- collect candidates (price space) --------
cand = []
# base
cand.append(("base",    "out/oof_e5_qty_mono.csv",       "out/test_predictions_e5_qty_mono.csv"))
# knn-mono
cand.append(("knnmono", "out/oof_xgb_knn_mono.csv",      "out/test_predictions_xgb_knn_mono.csv"))
# router (optional)
if   os.path.exists("out/oof_router_lowprice.csv") and os.path.exists("out/test_predictions_router_blended.csv"):
    cand.append(("router","out/oof_router_lowprice.csv","out/test_predictions_router_blended.csv"))
elif os.path.exists("out/oof_router.csv") and os.path.exists("out/test_predictions_router.csv"):
    cand.append(("router","out/oof_router.csv","out/test_predictions_router.csv"))

names=[]; OOF=[]; TE=[]
for name,oof_p,te_p in cand:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        names.append(name)
        OOF.append(load_oof_price(oof_p))
        TE.append(load_test_price(te_p))
assert len(names)>=2, "Need at least 2 models' OOF/Test files in /out."

Xtr = np.column_stack(OOF).astype('float32')   # (n_tr, K)
Xte = np.column_stack(TE).astype('float32')    # (n_te, K)
y   = train['price'].astype(float).values
print(f"Models in convex blend: {names} | Xtr={Xtr.shape}, Xte={Xte.shape}")

# -------- segments (A = has_img ∧ missing_qty) with correct TEST mapping --------
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
pattern = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
missing_qty_all = ~t.str.contains(pattern, regex=True).values

CKP = "/kaggle/input/saved-ml-model/checkpoint"
if os.path.exists(os.path.join(CKP,"img_has.npy")):
    has_img_all = np.load(os.path.join(CKP,"img_has.npy")).astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
n_tr, n_te = len(train), len(test)
SEG_A = has_img_all & missing_qty_all

A_tr = np.where(is_tr & SEG_A)[0]
B_tr = np.where(is_tr & ~SEG_A)[0]
A_te = (np.where(is_te & SEG_A)[0] - n_tr).astype(int)
B_te = (np.where(is_te & ~SEG_A)[0] - n_tr).astype(int)
A_te = A_te[(A_te>=0)&(A_te<n_te)]
B_te = B_te[(B_te>=0)&(B_te<n_te)]
print(f"Segments | train A={len(A_tr)}, B={len(B_tr)} | test A={len(A_te)}, B={len(B_te)}")

# -------- convex weight search (SMAPE) for a given index set --------
def convex_weights_for(idx, coarse_step=0.1, fine_halfspan=0.12, fine_step=0.02):
    K = Xtr.shape[1]
    Xs = Xtr[idx]; ys = y[idx]
    best = (1e9, None)
    # coarse simplex grid
    if K == 2:
        for w0 in np.arange(0,1+1e-9, coarse_step):
            w = np.array([w0, 1-w0], dtype=np.float32)
            cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
            if cv < best[0]: best=(cv, w.copy())
        # refine around best
        w0b = best[1][0]
        lo = max(0.0, w0b - fine_halfspan); hi = min(1.0, w0b + fine_halfspan)
        for w0 in np.arange(lo, hi+1e-9, fine_step):
            w = np.array([w0, 1-w0], dtype=np.float32)
            cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
            if cv < best[0]: best=(cv, w.copy())
    else:
        # K == 3 (if router exists)
        # coarse
        grid = np.arange(0,1+1e-9, coarse_step)
        for w0 in grid:
            for w1 in grid:
                w2 = 1 - w0 - w1
                if w2 < -1e-9: continue
                w = np.array([w0, w1, w2], dtype=np.float32)
                cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
                if cv < best[0]: best=(cv, w.copy())
        # refine
        w_b = best[1]; w0c,w1c = w_b[0], w_b[1]
        def clamp01(x): return max(0.0, min(1.0, x))
        w0_lo, w0_hi = clamp01(w0c - fine_halfspan), clamp01(w0c + fine_halfspan)
        w1_lo, w1_hi = clamp01(w1c - fine_halfspan), clamp01(w1c + fine_halfspan)
        for w0 in np.arange(w0_lo, w0_hi+1e-9, fine_step):
            for w1 in np.arange(w1_lo, w1_hi+1e-9, fine_step):
                w2 = 1 - w0 - w1
                if w2 < -1e-9: continue
                w = np.array([w0, w1, w2], dtype=np.float32)
                cv = smape(ys, np.clip(Xs @ w, 5e-4, None))
                if cv < best[0]: best=(cv, w.copy())
    return best  # (cv, w)

cvA,(wA) = convex_weights_for(A_tr)
cvB,(wB) = convex_weights_for(B_tr)
print(f"Best weights: segA w={wA} (CV={cvA:.3f}%), segB w={wB} (CV={cvB:.3f}%)")

# stitch OOF and compute overall CV
oo = np.zeros(n_tr, dtype=np.float32)
oo[A_tr] = np.clip(Xtr[A_tr] @ wA, 5e-4, None)
oo[B_tr] = np.clip(Xtr[B_tr] @ wB, 5e-4, None)
cv_all = smape(y, oo)
print("Pre-cal CV :", cv_all)

# -------- per-seg log calibration (as before) --------
def fit_ab(y_true, y_pred):
    t = np.log(np.clip(y_true, 1e-3, None))
    p = np.log(np.clip(y_pred, 1e-3, None))
    X = np.column_stack([p, np.ones_like(p)])
    a,b = np.linalg.lstsq(X, t, rcond=None)[0]
    return float(a), float(b)

def apply_ab(pred, a, b):
    return np.exp(a*np.log(np.clip(pred,1e-3,None)) + b)

aA,bA = fit_ab(y[A_tr], oo[A_tr]) if len(A_tr) else (1.0,0.0)
aB,bB = fit_ab(y[B_tr], oo[B_tr]) if len(B_tr) else (1.0,0.0)
oo_cal = oo.copy()
if len(A_tr): oo_cal[A_tr] = apply_ab(oo[A_tr], aA, bA)
if len(B_tr): oo_cal[B_tr] = apply_ab(oo[B_tr], aB, bB)
print("Post-cal CV:", smape(y, oo_cal))

# -------- predict test (blend + calibration) --------
te = np.zeros(n_te, dtype=np.float32)
te[A_te] = np.clip(Xte[A_te] @ wA, 5e-4, None)
te[B_te] = np.clip(Xte[B_te] @ wB, 5e-4, None)
te_cal = te.copy()
if len(A_te): te_cal[A_te] = apply_ab(te[A_te], aA, bA)
if len(B_te): te_cal[B_te] = apply_ab(te[B_te], aB, bB)
te_cal = np.clip(te_cal, 5e-4, None)

# -------- save --------
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_cal}).to_csv(
    "out/oof_meta_convex_seg_cal.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': te_cal.astype(float)}).to_csv(
    "out/test_predictions_meta_convex_seg_cal.csv", index=False)
with open("out/meta_convex_seg_cal_meta.json","w") as f:
    json.dump({
        'models': names,
        'wA': wA.tolist(), 'wB': wB.tolist(),
        'cv_segA_pre': float(cvA), 'cv_segB_pre': float(cvB), 'cv_all_pre': float(cv_all),
        'aA': aA, 'bA': bA, 'aB': aB, 'bB': bB
    }, f, indent=2)
print("Saved -> out/test_predictions_meta_convex_seg_cal.csv")


Models in convex blend: ['base', 'knnmono'] | Xtr=(75000, 2), Xte=(75000, 2)
Segments | train A=23995, B=51005 | test A=22635, B=52365
Best weights: segA w=[0.48 0.52] (CV=52.497%), segB w=[0.68       0.32000002] (CV=44.811%)
Pre-cal CV : 47.27033761331014
Post-cal CV: 47.22087363441564
Saved -> out/test_predictions_meta_convex_seg_cal.csv


In [18]:
# ===============================
# Residual booster on top of best blend + optional isotonic cal
# ===============================
import os, json, time, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb

t0=time.time()
log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

# ---------- prereqs ----------
need = ['train','test','df_all','FE_TR','FE_TE']
for n in need: assert n in globals(), f"Missing {n}"
y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 1e-2, None))

# Load current best (what you just saved)
best_oof_path  = "out/oof_meta_convex_seg_cal.csv"
best_test_path = "out/test_predictions_meta_convex_seg_cal.csv"
assert os.path.exists(best_oof_path) and os.path.exists(best_test_path), "Run your convex blend first."

oo_best = pd.read_csv(best_oof_path).set_index('sample_id')['oof_price'].reindex(train['sample_id']).values.astype('float32')
te_best = pd.read_csv(best_test_path).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')
log(f"Loaded best OOF/Test: {best_oof_path} / {best_test_path}")

# Optional: load base & knnmono OOF for a disagreement feature
def load_oof_price(path):
    df=pd.read_csv(path).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"No OOF column in {path}")

oof_base = load_oof_price("out/oof_e5_qty_mono.csv")  if os.path.exists("out/oof_e5_qty_mono.csv")  else None
oof_knnm = load_oof_price("out/oof_xgb_knn_mono.csv") if os.path.exists("out/oof_xgb_knn_mono.csv") else None
spread = (np.log(np.clip(oof_knnm,1e-3,None)) - np.log(np.clip(oof_base,1e-3,None))).astype('float32') if (oof_base is not None and oof_knnm is not None) else None

# ---------- segments (same rule: has_img ∧ missing_qty) ----------
CKP="/kaggle/input/saved-ml-model/checkpoint"
if os.path.exists(os.path.join(CKP,"img_has.npy")):
    has_img_all = np.load(os.path.join(CKP,"img_has.npy")).astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values

t = df_all['catalog_content'].fillna('').astype(str).str.lower()
pattern = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
missing_qty_all = ~t.str.contains(pattern, regex=True).values

is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
n_tr, n_te = len(train), len(test)

SEG_A = has_img_all & missing_qty_all
A_tr = np.where(is_tr & SEG_A)[0]
B_tr = np.where(is_tr & ~SEG_A)[0]
A_te = (np.where(is_te & SEG_A)[0] - n_tr).astype(int)
B_te = (np.where(is_te & ~SEG_A)[0] - n_tr).astype(int)
A_te = A_te[(A_te>=0)&(A_te<n_te)]
B_te = B_te[(B_te>=0)&(B_te<n_te)]
log(f"Segments | train A={len(A_tr)}, B={len(B_tr)} | test A={len(A_te)}, B={len(B_te)}")

# ---------- features for residual model ----------
def to2(a):
    a = np.asarray(a)
    return a if a.ndim==2 else a.reshape(-1,1)

X_tr = np.hstack([
    np.asarray(FE_TR, np.float32),
    to2(np.log(np.clip(oo_best, 1e-3, None)).astype('float32')),
    to2(((A_tr.size+0* np.arange(len(y)))>=0).astype('float32'))[:len(y)]  # dummy column placeholder
]).astype('float32')
# Replace the last column with a proper seg flag per row:
seg_flag_tr = np.zeros(len(y), dtype='float32'); seg_flag_tr[A_tr]=1.0
X_tr[:,-1] = seg_flag_tr

X_te = np.hstack([
    np.asarray(FE_TE, np.float32),
    to2(np.log(np.clip(te_best, 1e-3, None)).astype('float32')),
    to2(np.zeros(n_te, dtype='float32'))
]).astype('float32')
seg_flag_te = np.zeros(n_te, dtype='float32'); seg_flag_te[A_te]=1.0
X_te[:,-1] = seg_flag_te

# Optional spread feature if available
if spread is not None:
    X_tr = np.hstack([X_tr, to2(spread)])
    X_te = np.hstack([X_te, to2(np.zeros(n_te, dtype='float32'))])  # no test spread; set 0

log(f"Residual-feature shapes: X_tr={X_tr.shape}, X_te={X_te.shape}")

# ---------- residual target ----------
res_tr = (y_log - np.log(np.clip(oo_best, 1e-3, None))).astype('float32')

# ---------- CV train ----------
def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = dict(
    objective='reg:squarederror',  # smoother for residuals
    max_depth=7, n_estimators=2000, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    min_child_weight=2, tree_method='hist', device='cuda',
    early_stopping_rounds=150, eval_metric='rmse', random_state=42
)

oof_res = np.zeros_like(res_tr)
te_res  = np.zeros(n_te, dtype='float32')
for f,(tr,va) in enumerate(skf.split(X_tr, bins)):
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr[tr], res_tr[tr], eval_set=[(X_tr[va], res_tr[va])], verbose=False)
    oof_res[va] = m.predict(X_tr[va])
    te_res += m.predict(X_te) / skf.n_splits
    # live CV
    oof_final = np.exp(np.log(np.clip(oo_best[va],1e-3,None)) + oof_res[va]).clip(0.01)
    print(f"  [fold {f}] SMAPE={smape(y[va], oof_final):.3f}%")

# final OOF/TEST after residual boost
oo_boost = np.exp(np.log(np.clip(oo_best,1e-3,None)) + oof_res).clip(0.01)
te_boost = np.exp(np.log(np.clip(te_best,1e-3,None)) + te_res).clip(0.01)

cv_boost = smape(y, oo_boost)
log(f"[Residual booster] CV SMAPE: {cv_boost:.3f}%")

# ---------- optional: per-segment isotonic calibration ----------
USE_ISO = True
if USE_ISO:
    def fit_iso(y_true, pred):
        x = np.log(np.clip(pred,1e-3,None)); y_ = np.log(np.clip(y_true,1e-3,None))
        # sort to avoid duplicates issues
        order = np.argsort(x)
        ir = IsotonicRegression(y_min=y_.min()-1, y_max=y_.max()+1, increasing=True, out_of_bounds='clip')
        ir.fit(x[order], y_[order])
        return ir
    irA = fit_iso(y[A_tr], oo_boost[A_tr]) if len(A_tr) else None
    irB = fit_iso(y[B_tr], oo_boost[B_tr]) if len(B_tr) else None

    oo_iso = oo_boost.copy()
    if irA is not None: oo_iso[A_tr] = np.exp(irA.predict(np.log(np.clip(oo_boost[A_tr],1e-3,None))))
    if irB is not None: oo_iso[B_tr] = np.exp(irB.predict(np.log(np.clip(oo_boost[B_tr],1e-3,None))))
    cv_iso = smape(y, oo_iso)
    log(f"[Residual + isotonic] CV SMAPE: {cv_iso:.3f}%")
    te_iso = te_boost.copy()
    if irA is not None and len(A_te): te_iso[A_te] = np.exp(irA.predict(np.log(np.clip(te_boost[A_te],1e-3,None))))
    if irB is not None and len(B_te): te_iso[B_te] = np.exp(irB.predict(np.log(np.clip(te_boost[B_te],1e-3,None))))
    te_final = np.clip(te_iso, 5e-4, None)
else:
    te_final = np.clip(te_boost, 5e-4, None)

# ---------- save ----------
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': (oo_iso if USE_ISO else oo_boost)}).to_csv(
    "out/oof_residual_boost.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': te_final.astype(float)}).to_csv(
    "out/test_predictions_residual_boost.csv", index=False)
with open("out/residual_boost_meta.json","w") as f:
    json.dump({
        'base_oof': best_oof_path, 'base_test': best_test_path,
        'cv_boost': float(cv_boost),
        'cv_boost_iso': float(cv_iso) if USE_ISO else None
    }, f, indent=2)
log("Saved -> out/test_predictions_residual_boost.csv")


[14:46:02] Loaded best OOF/Test: out/oof_meta_convex_seg_cal.csv / out/test_predictions_meta_convex_seg_cal.csv | +0.0s
[14:46:04] Segments | train A=23995, B=51005 | test A=22635, B=52365 | +2.3s
[14:46:05] Residual-feature shapes: X_tr=(75000, 232), X_te=(75000, 232) | +2.4s
  [fold 0] SMAPE=47.239%
  [fold 1] SMAPE=47.293%
  [fold 2] SMAPE=47.866%
  [fold 3] SMAPE=46.947%
  [fold 4] SMAPE=46.603%
[14:46:17] [Residual booster] CV SMAPE: 47.190% | +15.3s
[14:46:17] [Residual + isotonic] CV SMAPE: 47.087% | +15.3s
[14:46:18] Saved -> out/test_predictions_residual_boost.csv | +15.5s


In [21]:
# === Super-blend: load available OOF/Test predictions and convex-blend them (log-space) ===
import os, glob, json, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    if   'oof_log' in df:         v = df['oof_log'].values
    elif 'oof_log_router' in df:  v = df['oof_log_router'].values
    elif 'oof' in df:             v = df['oof'].values
    elif 'pred_log' in df:        v = df['pred_log'].values
    elif 'oof_price' in df:       v = np.log(np.clip(df['oof_price'].values, 1e-4, None))
    else: raise ValueError(f"Unsupported OOF schema: {path}")
    return pd.Series(v, index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

# candidates we’ll try to load if present
pairs = [
    ("convex_seg_cal",   "out/oof_meta_convex_seg_cal.csv",       "out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost",   "out/oof_residual_boost.csv",            "out/test_predictions_residual_boost.csv"),
    ("ridge_meta",       "out/oof_meta_blend_ridge_quick.csv",    "out/test_predictions_meta_blend_ridge_quick.csv"),
    ("ridge_meta_full",  "out/oof_meta_blend_ridge.csv",          "out/test_predictions_meta_blend_ridge.csv"),
    ("cluster_cal",      "out/oof_meta_after_cluster_cal.csv",    "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",     "out/oof_meta_after_residual_fix.csv",   "out/test_predictions_after_residual_fix.csv"),
    ("decile_blend",     "out/oof_meta_decile_blend.csv",         "out/test_predictions_meta_decile_blend.csv"),
]

names, OOF, TST = [], [], []
for name, oof_p, te_p in pairs:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        try:
            oof_l = load_oof_log(oof_p)
            te_l  = load_te_log(te_p)
            names.append(name); OOF.append(oof_l.reshape(-1,1)); TST.append(te_l.reshape(-1,1))
            print(f"✓ using {name}")
        except Exception as e:
            print(f"skip {name} ->", e)

assert len(names) >= 2, "Need at least two OOF/Test pairs in out/"

Xtr = np.hstack(OOF)   # log-space
Xte = np.hstack(TST)
y   = train['price'].values
print("Super-blend stack:", Xtr.shape, Xte.shape, "| models:", names)

# convex weights (global). grid for K<=3, random Dirichlet otherwise
def best_weights(X, y, coarse=0.1, fine=0.02, shots=3000):
    def score(w): return smape(y, np.exp(np.clip(X@w, -20, 20)).clip(1e-4))
    K = X.shape[1]
    best = (1e9, None)
    if K==2:
        for w0 in np.arange(0,1+1e-9, coarse):
            w = np.array([w0, 1-w0], 'float32'); s = score(w)
            if s<best[0]: best=(s, w.copy())
        lo,hi = max(0,best[1][0]-0.12), min(1,best[1][0]+0.12)
        for w0 in np.arange(lo, hi+1e-9, fine):
            w = np.array([w0, 1-w0], 'float32'); s = score(w)
            if s<best[0]: best=(s, w.copy())
    elif K==3:
        grid = np.arange(0,1+1e-9, coarse)
        for w0 in grid:
            for w1 in grid:
                w2 = 1-w0-w1
                if w2<-1e-9: continue
                w = np.array([w0,w1,w2],'float32'); s = score(w)
                if s<best[0]: best=(s, w.copy())
        # fine around best
        w0c,w1c = best[1][0], best[1][1]
        def clamp(x): return min(1,max(0,x))
        for w0 in np.arange(clamp(w0c-0.12), clamp(w0c+0.12)+1e-9, fine):
            for w1 in np.arange(clamp(w1c-0.12), clamp(w1c+0.12)+1e-9, fine):
                w2 = 1-w0-w1
                if w2<-1e-9: continue
                w = np.array([w0,w1,w2],'float32'); s = score(w)
                if s<best[0]: best=(s, w.copy())
    else:
        for _ in range(shots):
            w = np.random.dirichlet(np.ones(K)).astype('float32'); s = score(w)
            if s<best[0]: best=(s, w.copy())
    return best

cv, w = best_weights(Xtr, y)
print(f"[super-blend] CV={cv:.3f}% | weights={dict(zip(names, np.round(w,3)))}")

oo = np.exp(Xtr@w).clip(1e-4); te = np.exp(Xte@w).clip(1e-4)
cv_all = smape(y, oo); print(f"[super-blend] reconfirm CV={cv_all:.3f}%")

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_superblend_fast.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_superblend_fast.csv", index=False)
print("Saved -> out/test_predictions_superblend_fast.csv")


✓ using convex_seg_cal
✓ using residual_boost
✓ using cluster_cal
✓ using residual_fix
Super-blend stack: (75000, 4) (75000, 4) | models: ['convex_seg_cal', 'residual_boost', 'cluster_cal', 'residual_fix']
[super-blend] CV=47.055% | weights={'convex_seg_cal': 0.03, 'residual_boost': 0.701, 'cluster_cal': 0.265, 'residual_fix': 0.004}
[super-blend] reconfirm CV=47.055%
Saved -> out/test_predictions_superblend_fast.csv


In [22]:
# === Quantile mapping calibration (per-segment) on top of a chosen OOF/Test pair ===
import numpy as np, pandas as pd, os

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# pick a base (prefer last superblend, else your residual_boost, else convex_seg_cal)
candidates = [
    ("out/oof_superblend_fast.csv",          "out/test_predictions_superblend_fast.csv"),
    ("out/oof_residual_boost.csv",           "out/test_predictions_residual_boost.csv"),
    ("out/oof_meta_convex_seg_cal.csv",      "out/test_predictions_meta_convex_seg_cal.csv"),
]
for oof_p, te_p in candidates:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        base_oof, base_te = oof_p, te_p; break
print("Using base for QM:", base_oof, "|", base_te)

oo = pd.read_csv(base_oof).set_index('sample_id')['oof_price'].reindex(train['sample_id']).values.astype('float32')
te = pd.read_csv(base_te).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')
y  = train['price'].values.astype('float32')

# segments (same as before)
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
mmq_all = ~t.str.contains(r'\b(\d+(?:\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all = np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values
is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
SEG_A_all = has_img_all & mmq_all
A_tr = np.where(is_tr & SEG_A_all)[0]; B_tr = np.where(is_tr & ~SEG_A_all)[0]
A_te = (np.where(is_te & SEG_A_all)[0] - len(train)).clip(0, len(test)-1)
B_te = (np.where(is_te & ~SEG_A_all)[0] - len(train)).clip(0, len(test)-1)

def quantile_map_fit(y_true, y_pred, n=101):
    # work in log-space, piecewise-linear mapping pred->true across quantiles
    q = np.linspace(0,1,n)
    xp = np.quantile(np.log(np.clip(y_pred,1e-3,None)), q)
    yp = np.quantile(np.log(np.clip(y_true,1e-3,None)), q)
    return xp, yp

def quantile_map_apply(pred, xp, yp):
    x = np.log(np.clip(pred,1e-3,None))
    y = np.interp(x, xp, yp, left=yp[0], right=yp[-1])
    return np.exp(y)

# fit per segment
xpA, ypA = quantile_map_fit(y[A_tr], oo[A_tr]) if len(A_tr) else (None,None)
xpB, ypB = quantile_map_fit(y[B_tr], oo[B_tr]) if len(B_tr) else (None,None)

oo_qm = oo.copy()
if xpA is not None: oo_qm[A_tr] = quantile_map_apply(oo[A_tr], xpA, ypA)
if xpB is not None: oo_qm[B_tr] = quantile_map_apply(oo[B_tr], xpB, ypB)
cv_before = smape(y, oo); cv_after = smape(y, oo_qm)
print(f"[QM] CV before={cv_before:.3f}%, after={cv_after:.3f}%")

te_qm = te.copy()
if xpA is not None: te_qm[A_te] = quantile_map_apply(te[A_te], xpA, ypA)
if xpB is not None: te_qm[B_te] = quantile_map_apply(te[B_te], xpB, ypB)
te_qm = te_qm.clip(5e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_qm}).to_csv("out/oof_after_quantile_map.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_qm.astype(float)}).to_csv("out/test_predictions_after_quantile_map.csv", index=False)
print("Saved -> out/test_predictions_after_quantile_map.csv")


Using base for QM: out/oof_superblend_fast.csv | out/test_predictions_superblend_fast.csv


/tmp/ipykernel_37/2475984953.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mmq_all = ~t.str.contains(r'\b(\d+(?:\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values


[QM] CV before=47.055%, after=51.047%
Saved -> out/test_predictions_after_quantile_map.csv


In [23]:
# === Linear residual corrector (very light): RidgeCV over FE + log(pred) + seg flag ===
import numpy as np, pandas as pd, os, json
from sklearn.linear_model import RidgeCV

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# choose a base to correct (prefer the latest, else convex_seg_cal)
for oof_p, te_p in [
    ("out/oof_after_quantile_map.csv",      "out/test_predictions_after_quantile_map.csv"),
    ("out/oof_residual_boost.csv",          "out/test_predictions_residual_boost.csv"),
    ("out/oof_meta_convex_seg_cal.csv",     "out/test_predictions_meta_convex_seg_cal.csv"),
]:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        base_oof, base_te = oof_p, te_p; break
print("Correcting base:", base_oof, "|", base_te)

oo = pd.read_csv(base_oof).set_index('sample_id')['oof_price'].reindex(train['sample_id']).values.astype('float32')
te = pd.read_csv(base_te).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')
y  = train['price'].values.astype('float32')

# seg flag
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
mmq_all = ~t.str.contains(r'\b(\d+(?:\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all = np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values
is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
seg = (has_img_all & mmq_all)
seg_tr = seg[is_tr].astype('float32')
seg_te = seg[is_te].astype('float32')

# design: FE only (already in memory), plus log(pred) and interaction
logp_tr = np.log(np.clip(oo, 1e-3, None)).astype('float32')
logp_te = np.log(np.clip(te, 1e-3, None)).astype('float32')

X_tr = np.hstack([FE_TR.astype('float32'),
                  logp_tr.reshape(-1,1),
                  (seg_tr*logp_tr).reshape(-1,1)])
X_te = np.hstack([FE_TE.astype('float32'),
                  logp_te.reshape(-1,1),
                  (seg_te*logp_te).reshape(-1,1)])

res_tr = (np.log(np.clip(y,1e-3,None)) - logp_tr).astype('float32')

m = RidgeCV(alphas=[0.01,0.05,0.1,0.2,0.3,0.5,1.0], fit_intercept=True)
m.fit(X_tr, res_tr)
delta_tr = m.predict(X_tr); delta_te = m.predict(X_te)

oo_lin = np.exp(logp_tr + delta_tr).clip(1e-4)
te_lin = np.exp(logp_te + delta_te).clip(1e-4)
print(f"[linear residual] CV before={smape(y, oo):.3f}%, after={smape(y, oo_lin):.3f}% | alpha*={m.alpha_}")

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_lin}).to_csv("out/oof_after_linear_residual.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_lin.astype(float)}).to_csv("out/test_predictions_after_linear_residual.csv", index=False)
print("Saved -> out/test_predictions_after_linear_residual.csv")


Correcting base: out/oof_after_quantile_map.csv | out/test_predictions_after_quantile_map.csv


/tmp/ipykernel_37/3679916311.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mmq_all = ~t.str.contains(r'\b(\d+(?:\.\d+)?\s*(g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b', regex=True).values


[linear residual] CV before=51.047%, after=47.642% | alpha*=1.0
Saved -> out/test_predictions_after_linear_residual.csv


In [25]:
# Non-capturing groups; also set na=False for safety
import numpy as np, pandas as pd, os

def build_seg_masks(df_all, ckpt_img_has="/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    qty_pat = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
    mmq_all = ~t.str.contains(qty_pat, regex=True, na=False).values  # missing quantity

    if os.path.exists(ckpt_img_has):
        has_img_all = np.load(ckpt_img_has).astype(bool)
    elif os.path.exists("img_has.npy"):
        has_img_all = np.load("img_has.npy").astype(bool)
    else:
        has_img_all = (df_all['image_link'].fillna('')!='').values

    is_tr = (df_all['is_train'].values==1)
    is_te = ~is_tr
    SEG_A = has_img_all & mmq_all

    n_tr = is_tr.sum(); n_te = is_te.sum()
    A_tr = np.where(is_tr & SEG_A)[0]
    B_tr = np.where(is_tr & ~SEG_A)[0]
    A_te = (np.where(is_te & SEG_A)[0] - n_tr).astype(int)
    B_te = (np.where(is_te & ~SEG_A)[0] - n_tr).astype(int)
    A_te = A_te[(A_te>=0)&(A_te<n_te)]
    B_te = B_te[(B_te>=0)&(B_te<n_te)]
    return A_tr, B_tr, A_te, B_te

In [26]:
# Segment-aware super-blend (log-space), K up to 6, fast random-Dirichlet search
import os, numpy as np, pandas as pd, json

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    if   'oof_log' in df:         v = df['oof_log'].values
    elif 'oof_log_router' in df:  v = df['oof_log_router'].values
    elif 'oof' in df:             v = df['oof'].values
    elif 'pred_log' in df:        v = df['pred_log'].values
    elif 'oof_price' in df:       v = np.log(np.clip(df['oof_price'].values, 1e-4, None))
    else: raise ValueError(f"Unsupported OOF schema: {p}")
    return pd.Series(v, index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs = [
    ("convex_seg_cal", "out/oof_meta_convex_seg_cal.csv",    "out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost", "out/oof_residual_boost.csv",         "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",    "out/oof_meta_after_cluster_cal.csv", "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",   "out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
]
names, OOF, TST = [], [], []
for name, oof_p, te_p in pairs:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        names.append(name); OOF.append(load_oof_log(oof_p).reshape(-1,1)); TST.append(load_te_log(te_p).reshape(-1,1))
assert len(names)>=2, "Need at least two pairs in out/"
Xtr = np.hstack(OOF); Xte = np.hstack(TST); y = train['price'].values.astype('float32')

# segment indices
A_tr, B_tr, A_te, B_te = build_seg_masks(df_all)

def best_w(Xs, ys, shots=3000):
    best = (1e9, None)
    K = Xs.shape[1]
    for _ in range(shots):
        w = np.random.dirichlet(np.ones(K)).astype('float32')
        sm = smape(ys, np.exp(np.clip(Xs@w, -20, 20)).clip(1e-4))
        if sm < best[0]: best = (sm, w.copy())
    return best

cvA, wA = best_w(Xtr[A_tr], y[A_tr]) if len(A_tr) else (np.nan, np.full(Xtr.shape[1], 1/Xtr.shape[1],'float32'))
cvB, wB = best_w(Xtr[B_tr], y[B_tr]) if len(B_tr) else (np.nan, np.full(Xtr.shape[1], 1/Xtr.shape[1],'float32'))

oo = np.zeros(len(train), 'float32'); te = np.zeros(len(test), 'float32')
if len(A_tr): oo[A_tr] = np.exp(Xtr[A_tr]@wA).clip(1e-4)
if len(B_tr): oo[B_tr] = np.exp(Xtr[B_tr]@wB).clip(1e-4)
if len(A_te): te[A_te] = np.exp(Xte[A_te]@wA).clip(1e-4)
if len(B_te): te[B_te] = np.exp(Xte[B_te]@wB).clip(1e-4)
cv_all = smape(y, oo)

print(f"[seg-superblend] K={len(names)} | segA CV={cvA:.3f}% segB CV={cvB:.3f}% | overall CV={cv_all:.3f}%")
print("  weights A:", dict(zip(names, np.round(wA,3))))
print("  weights B:", dict(zip(names, np.round(wB,3))))

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_seg_superblend.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_seg_superblend.csv", index=False)

# optional tiny global log-cal (rarely hurts)
t = np.log(np.clip(y,1e-3,None)); p = np.log(np.clip(oo,1e-3,None))
A = np.column_stack([p, np.ones_like(p)])
a,b = np.linalg.lstsq(A, t, rcond=None)[0]
def apply_ab(pred): 
    return np.exp(a*np.log(np.clip(pred,1e-3,None))+b)
oo_cal = apply_ab(oo).clip(1e-4); te_cal = apply_ab(te).clip(1e-4)
cv_after = smape(y, oo_cal)
print(f"[seg-superblend] after global log-cal: CV={cv_after:.3f}%")

pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_cal}).to_csv("out/oof_seg_superblend_cal.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_cal.astype(float)}).to_csv("out/test_predictions_seg_superblend_cal.csv", index=False)
print("Saved -> out/test_predictions_seg_superblend_cal.csv")

[seg-superblend] K=4 | segA CV=52.378% segB CV=44.544% | overall CV=47.050%
  weights A: {'convex_seg_cal': 0.388, 'residual_boost': 0.571, 'cluster_cal': 0.036, 'residual_fix': 0.005}
  weights B: {'convex_seg_cal': 0.022, 'residual_boost': 0.732, 'cluster_cal': 0.221, 'residual_fix': 0.025}
[seg-superblend] after global log-cal: CV=47.035%
Saved -> out/test_predictions_seg_superblend_cal.csv


In [28]:
# ===== A) Build segment masks (no regex warning) =====
import os, numpy as np, pandas as pd

def build_seg_masks(df_all, ckpt_img_has="/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    t = df_all['catalog_content'].fillna('').astype(str).str.lower()
    qty_pat = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
    mmq_all = ~t.str.contains(qty_pat, regex=True, na=False).values  # missing quantity

    if os.path.exists(ckpt_img_has):
        has_img_all = np.load(ckpt_img_has).astype(bool)
    elif os.path.exists("img_has.npy"):
        has_img_all = np.load("img_has.npy").astype(bool)
    else:
        has_img_all = (df_all['image_link'].fillna('')!='').values

    is_tr = (df_all['is_train'].values==1)
    is_te = ~is_tr
    SEG_A = has_img_all & mmq_all

    n_tr = is_tr.sum(); n_te = is_te.sum()
    A_tr = np.where(is_tr & SEG_A)[0]
    B_tr = np.where(is_tr & ~SEG_A)[0]
    A_te = (np.where(is_te & SEG_A)[0] - n_tr).astype(int)
    B_te = (np.where(is_te & ~SEG_A)[0] - n_tr).astype(int)
    A_te = A_te[(A_te>=0)&(A_te<n_te)]
    B_te = B_te[(B_te>=0)&(B_te<n_te)]
    return A_tr, B_tr, A_te, B_te

A_tr, B_tr, A_te, B_te = build_seg_masks(df_all)
print("Segments | train A/B:", len(A_tr), len(B_tr), "| test A/B:", len(A_te), len(B_te))

Segments | train A/B: 23995 51005 | test A/B: 22635 52365


In [29]:
# ===== B) Assemble meta features =====
import os, glob, numpy as np, pandas as pd
from sklearn.decomposition import PCA

def load_oof_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    if   'oof_log' in df:         v = df['oof_log'].values
    elif 'oof_log_router' in df:  v = df['oof_log_router'].values
    elif 'oof' in df:             v = df['oof'].values
    elif 'pred_log' in df:        v = df['pred_log'].values
    elif 'oof_price' in df:       v = np.log(np.clip(df['oof_price'].values, 1e-4, None))
    else: raise ValueError(f"Unsupported OOF schema: {path}")
    return pd.Series(v, index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

# Gather all plausible pairs from out/
pairs = [
    ("convex_seg_cal",   "out/oof_meta_convex_seg_cal.csv",       "out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost",   "out/oof_residual_boost.csv",            "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",      "out/oof_meta_after_cluster_cal.csv",    "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",     "out/oof_meta_after_residual_fix.csv",   "out/test_predictions_after_residual_fix.csv"),
    ("superblend_fast",  "out/oof_superblend_fast.csv",           "out/test_predictions_superblend_fast.csv"),
    ("seg_superblend",   "out/oof_seg_superblend_cal.csv",        "out/test_predictions_seg_superblend_cal.csv"),
    ("huber_meta",       "out/oof_huber_meta.csv",                "out/test_predictions_huber_meta.csv"),
    ("linear_residual",  "out/oof_after_linear_residual.csv",     "out/test_predictions_after_linear_residual.csv"),
    ("ridge_meta_full",  "out/oof_meta_blend_ridge.csv",          "out/test_predictions_meta_blend_ridge.csv"),
    ("ridge_meta_quick", "out/oof_meta_blend_ridge_quick.csv",    "out/test_predictions_meta_blend_ridge_quick.csv"),
]
names, OOF, TST = [], [], []
for name, oof_p, te_p in pairs:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        try:
            oof_l = load_oof_log(oof_p); te_l = load_te_log(te_p)
            names.append(name); OOF.append(oof_l.reshape(-1,1)); TST.append(te_l.reshape(-1,1))
            print("✓ using", name)
        except Exception as e:
            print("skip", name, "->", e)

assert len(names) >= 2, "Need at least two OOF/Test pairs present in out/"
Xtr_log = np.hstack(OOF)  # (n, K) log-preds
Xte_log = np.hstack(TST)
K = Xtr_log.shape[1]
print("meta base (log preds) K=", K)

# Add FE
FE_TR32 = np.asarray(FE_TR, dtype='float32'); FE_TE32 = np.asarray(FE_TE, dtype='float32')

# PCA(64) of MiniLM embeddings (helps semantic clusters)
emb_all = np.vstack([EMB_TR, EMB_TE]).astype('float32', copy=False)
pca = PCA(n_components=64, random_state=42)
Z_all = pca.fit_transform(emb_all).astype('float32')
is_tr = (df_all['is_train'].values==1)
Z_tr = Z_all[is_tr]; Z_te = Z_all[~is_tr]

# Segment flags
seg_tr = np.zeros(len(train), 'float32'); seg_tr[A_tr]=1.0
seg_te = np.zeros(len(test),  'float32'); seg_te[A_te]=1.0

# Final design: [log preds..., seg_flag, FE..., PCA64...]
Xtr_meta = np.hstack([Xtr_log, seg_tr.reshape(-1,1), FE_TR32, Z_tr])
Xte_meta = np.hstack([Xte_log, seg_te.reshape(-1,1), FE_TE32, Z_te])

y   = train['price'].astype('float32').values
y_l = np.log(np.clip(y, 1e-3, None)).astype('float32')

print("Xtr_meta/Xte_meta:", Xtr_meta.shape, Xte_meta.shape, "| features ~", Xtr_meta.shape[1])


✓ using convex_seg_cal
✓ using residual_boost
✓ using cluster_cal
✓ using residual_fix
✓ using superblend_fast
✓ using seg_superblend
✓ using huber_meta
✓ using linear_residual
meta base (log preds) K= 8
Xtr_meta/Xte_meta: (75000, 302) (75000, 302) | features ~ 302


In [30]:
# ===== C) Monotone XGBoost meta (log-space) + seg isotonic =====
import numpy as np, pandas as pd, os, json, time
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb

t0=time.time(); log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# Stratify by price bins
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)

# SMAPE-ish sample weights (downweight big prices a bit)
med = np.median(y); sw = (1.0/(y + med)).astype('float32')
sw /= sw.mean()

# Monotone constraints: +1 for each log-pred column; 0 elsewhere
K = Xtr_log.shape[1]
nfeat = Xtr_meta.shape[1]
mono = [1]*K + [0]*(nfeat-K)
mono_str = "(" + ",".join(str(v) for v in mono) + ")"

params = dict(
    objective='reg:squarederror',
    tree_method='hist',
    device='cuda',          # will fallback below if needed
    max_depth=8,
    n_estimators=2200,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=2,
    eval_metric='mae',
    monotone_constraints=mono_str,
    random_state=42
)

oof_log = np.zeros(len(y), 'float32')
te_log  = np.zeros(len(test), 'float32')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for f,(tr,va) in enumerate(skf.split(Xtr_meta, bins)):
    # GPU->CPU fallback if needed
    try:
        m = xgb.XGBRegressor(**params, early_stopping_rounds=150)
        m.fit(Xtr_meta[tr], y_l[tr],
              sample_weight=sw[tr],
              eval_set=[(Xtr_meta[va], y_l[va])],
              verbose=False)
    except Exception as e:
        p2 = params.copy()
        p2.pop('device', None)
        m = xgb.XGBRegressor(**p2, early_stopping_rounds=150)
        m.fit(Xtr_meta[tr], y_l[tr],
              sample_weight=sw[tr],
              eval_set=[(Xtr_meta[va], y_l[va])],
              verbose=False)

    oof_log[va] = m.predict(Xtr_meta[va])
    te_log     += m.predict(Xte_meta) / skf.n_splits

    fold_cv = smape(y[va], np.exp(oof_log[va]).clip(1e-4))
    log(f"[fold {f}] SMAPE={fold_cv:.3f}% | trees={m.get_booster().best_iteration}")

cv_raw = smape(y, np.exp(oof_log).clip(1e-4))
log(f"[XGB-meta] CV (raw): {cv_raw:.3f}%")

# ----- Per-segment isotonic in log-space -----
is_tr_mask = np.zeros(len(train), bool); is_tr_mask[A_tr]=True
logp = oof_log.copy(); logp_te = te_log.copy()

def fit_iso(y_true, logp_):
    order = np.argsort(logp_)
    ir = IsotonicRegression(increasing=True, out_of_bounds='clip')
    ir.fit(logp_[order], np.log(np.clip(y_true,1e-3,None))[order])
    return ir

irA = fit_iso(y[A_tr], logp[A_tr]) if len(A_tr) else None
irB = fit_iso(y[B_tr], logp[B_tr]) if len(B_tr) else None

oo_iso = np.exp(logp).copy()
if irA is not None: oo_iso[A_tr] = np.exp(irA.predict(logp[A_tr]))
if irB is not None: oo_iso[B_tr] = np.exp(irB.predict(logp[B_tr]))
cv_iso = smape(y, oo_iso.clip(1e-4))
log(f"[XGB-meta] CV after seg isotonic: {cv_iso:.3f}%")

te_iso = np.exp(logp_te).copy()
if irA is not None and len(A_te): te_iso[A_te] = np.exp(irA.predict(logp_te[A_te]))
if irB is not None and len(B_te): te_iso[B_te] = np.exp(irB.predict(logp_te[B_te]))
te_iso = te_iso.clip(1e-4)

# ----- Save -----
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_iso}).to_csv("out/oof_meta_xgb_monotone.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_iso.astype(float)}).to_csv("out/test_predictions_meta_xgb_monotone.csv", index=False)
with open("out/meta_xgb_monotone_meta.json","w") as f:
    json.dump({
        'cv_raw': float(cv_raw),
        'cv_after_iso': float(cv_iso),
        'K_log_preds': int(K),
        'features_total': int(nfeat),
        'models_used': names
    }, f, indent=2)
print("Saved -> out/test_predictions_meta_xgb_monotone.csv")


[15:13:19] [fold 0] SMAPE=48.182% | trees=1454 | +26.3s
[15:13:51] [fold 1] SMAPE=48.583% | trees=1885 | +58.3s
[15:14:19] [fold 2] SMAPE=48.970% | trees=1629 | +86.6s
[15:14:52] [fold 3] SMAPE=48.224% | trees=1851 | +119.1s
[15:15:17] [fold 4] SMAPE=47.699% | trees=1404 | +144.3s
[15:15:17] [XGB-meta] CV (raw): 48.332% | +144.3s
[15:15:17] [XGB-meta] CV after seg isotonic: 48.539% | +144.3s
Saved -> out/test_predictions_meta_xgb_monotone.csv


In [32]:
# === A) LGBM meta on log-preds (+ seg flag), 5-fold CV on y (SMAPE computed) ===
import os, numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# 1) Load the same OOF/Test pairs you’ve been using (log-space features)
def load_oof_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    if   'oof_log' in df:         v = df['oof_log'].values
    elif 'oof_log_router' in df:  v = df['oof_log_router'].values
    elif 'oof' in df:             v = df['oof'].values
    elif 'pred_log' in df:        v = df['pred_log'].values
    elif 'oof_price' in df:       v = np.log(np.clip(df['oof_price'].values, 1e-4, None))
    else: raise ValueError(f"Unsupported OOF schema: {path}")
    return pd.Series(v, index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs = [
    ("convex_seg_cal", "out/oof_meta_convex_seg_cal.csv",    "out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost", "out/oof_residual_boost.csv",         "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",    "out/oof_meta_after_cluster_cal.csv", "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",   "out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
    ("seg_superblend", "out/oof_seg_superblend_cal.csv",     "out/test_predictions_seg_superblend_cal.csv"),
]
names, OOF, TST = [], [], []
for name, oof_p, te_p in pairs:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        names.append(name); OOF.append(load_oof_log(oof_p).reshape(-1,1)); TST.append(load_te_log(te_p).reshape(-1,1))

assert len(names)>=2, "Need at least two OOF/Test pairs in out/"
Xtr_log = np.hstack(OOF)  # (n, K)
Xte_log = np.hstack(TST)
K = Xtr_log.shape[1]
print("Using K log-preds:", K, names)

# 2) Segment flag (warning-free regex)
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all = ~t.str.contains(qty_pat, regex=True, na=False).values
import numpy as np, os
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all = np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values
is_tr=(df_all['is_train'].values==1); is_te=~is_tr
seg_flag_tr = (has_img_all & mmq_all)[is_tr].astype('float32').reshape(-1,1)
seg_flag_te = (has_img_all & mmq_all)[is_te].astype('float32').reshape(-1,1)

# 3) Design: just [log-preds..., seg_flag]  (keep it simple)
Xtr = np.hstack([Xtr_log, seg_flag_tr]).astype('float32')
Xte = np.hstack([Xte_log, seg_flag_te]).astype('float32')
y   = train['price'].astype('float32').values
y_l = np.log(np.clip(y, 1e-3, None)).astype('float32')

# 4) CV LightGBM on log-target, MAE (robust), shallow trees
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
params = dict(
    objective='regression_l1',
    num_leaves=31,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=2500,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=40,
    reg_lambda=1.0,
    verbose=-1
)

oof = np.zeros_like(y_l); te = np.zeros(len(test), 'float32')
for f,(tr,va) in enumerate(skf.split(Xtr, bins)):
    m = lgb.LGBMRegressor(**params)
    m.fit(
    Xtr[tr], y_l[tr],
    eval_set=[(Xtr[va], y_l[va])],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150),
        lgb.log_evaluation(period=0)  # disables verbose output
    ]
)

    oof[va] = m.predict(Xtr[va])
    te += m.predict(Xte) / skf.n_splits
    cv_f = smape(y[va], np.exp(oof[va]).clip(1e-4))
    print(f"[fold {f}] SMAPE={cv_f:.3f}%")

cv_all = smape(y, np.exp(oof).clip(1e-4))
print(f"[LGBM-log-meta] CV: {cv_all:.3f}%")

# 5) Save
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': np.exp(oof).clip(1e-4)}).to_csv("out/oof_meta_lgbm_logpreds.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': np.exp(te).clip(1e-4).astype(float)}).to_csv("out/test_predictions_meta_lgbm_logpreds.csv", index=False)
print("Saved -> out/test_predictions_meta_lgbm_logpreds.csv")


Using K log-preds: 5 ['convex_seg_cal', 'residual_boost', 'cluster_cal', 'residual_fix', 'seg_superblend']
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[267]	valid_0's l1: 0.510196
[fold 0] SMAPE=46.788%
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[343]	valid_0's l1: 0.516231
[fold 1] SMAPE=47.203%
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[108]	valid_0's l1: 0.521157
[fold 2] SMAPE=47.623%
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[251]	valid_0's l1: 0.508822
[fold 3] SMAPE=46.510%
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[243]	valid_0's l1: 0.503975
[fold 4] SMAPE=46.152%
[LGBM-log-meta] CV: 46.855%
Saved -> out/test_predictions_meta_lgbm_logpreds.csv


In [33]:
# === C) Quick re-super-blend adding the new candidates ===
import os, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d, dtype=float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = (np.log(np.clip(df['oof_price'].values, 1e-4, None))
         if 'oof_price' in df else df.filter(like='oof').iloc[:,0].values)
    return pd.Series(v, index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs = []
for name, oo, te in [
    ("meta_lgbm_logpreds",  "out/oof_meta_lgbm_logpreds.csv",      "out/test_predictions_meta_lgbm_logpreds.csv"),
    ("cluster128_cal",      "out/oof_after_cluster128_cal.csv",     "out/test_predictions_after_cluster128_cal.csv"),
    ("seg_superblend",      "out/oof_seg_superblend_cal.csv",       "out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost",      "out/oof_residual_boost.csv",           "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",         "out/oof_meta_after_cluster_cal.csv",   "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",        "out/oof_meta_after_residual_fix.csv",  "out/test_predictions_after_residual_fix.csv"),
    ("convex_seg_cal",      "out/oof_meta_convex_seg_cal.csv",      "out/test_predictions_meta_convex_seg_cal.csv"),
]:
    if os.path.exists(oo) and os.path.exists(te):
        pairs.append((name, oo, te))
        print("✓", name)

assert len(pairs)>=2
names, OOF, TST = [], [], []
for name, oo, te in pairs:
    names.append(name); OOF.append(load_oof_log(oo).reshape(-1,1)); TST.append(load_te_log(te).reshape(-1,1))

Xtr = np.hstack(OOF); Xte = np.hstack(TST); y = train['price'].values.astype('float32')

# quick convex weight search (Dirichlet random)
def best_w(Xs, ys, shots=4000):
    best = (1e9, None)
    K = Xs.shape[1]
    for _ in range(shots):
        w = np.random.dirichlet(np.ones(K)).astype('float32')
        s = smape(ys, np.exp(np.clip(Xs@w, -20, 20)).clip(1e-4))
        if s<best[0]: best=(s, w.copy())
    return best

cv, w = best_w(Xtr, y)
print(f"[re-superblend] CV={cv:.3f}% | weights:", dict(zip(names, np.round(w,3))))
oo = np.exp(Xtr@w).clip(1e-4); te = np.exp(Xte@w).clip(1e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_superblend_round2.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_superblend_round2.csv", index=False)
print("Saved -> out/test_predictions_superblend_round2.csv")


✓ meta_lgbm_logpreds
✓ seg_superblend
✓ residual_boost
✓ cluster_cal
✓ residual_fix
✓ convex_seg_cal
[re-superblend] CV=46.855% | weights: {'meta_lgbm_logpreds': 0.917, 'seg_superblend': 0.025, 'residual_boost': 0.011, 'cluster_cal': 0.036, 'residual_fix': 0.008, 'convex_seg_cal': 0.004}
Saved -> out/test_predictions_superblend_round2.csv


In [41]:
# === LGBM meta on log-preds + pairwise spreads (+ seg flag) [FIXED: no fit(verbose=...)] ===
import os, time, json, itertools
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

# ---------- helpers ----------
def smape(y_true, y_pred):
    d = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m] - y_pred[m]) / d[m]
    return out.mean() * 100

def load_oof_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    if   'oof_log' in df:         v = df['oof_log'].values
    elif 'oof_log_router' in df:  v = df['oof_log_router'].values
    elif 'oof' in df:             v = df['oof'].values
    elif 'pred_log' in df:        v = df['pred_log'].values
    elif 'oof_price' in df:       v = np.log(np.clip(df['oof_price'].values, 1e-4, None))
    else: raise ValueError(f"Unsupported OOF schema: {path}")
    return pd.Series(v, index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(path):
    df = pd.read_csv(path).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

def add_spreads(X):
    # for each pair (i,j), add (log_i - log_j) and |log_i - log_j|
    diffs = []
    K = X.shape[1]
    for i, j in itertools.combinations(range(K), 2):
        d = (X[:, i] - X[:, j]).astype('float32')
        diffs.append(d.reshape(-1,1))
        diffs.append(np.abs(d).reshape(-1,1))
    return (np.hstack(diffs) if diffs else np.empty((X.shape[0], 0), 'float32'))

# ---------- gather OOF/Test pairs (log features) ----------
pairs = [
    ("convex_seg_cal", "out/oof_meta_convex_seg_cal.csv",    "out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost", "out/oof_residual_boost.csv",         "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",    "out/oof_meta_after_cluster_cal.csv", "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",   "out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
    ("seg_superblend", "out/oof_seg_superblend_cal.csv",     "out/test_predictions_seg_superblend_cal.csv"),
]
names, OOF, TST = [], [], []
for name, oof_p, te_p in pairs:
    if os.path.exists(oof_p) and os.path.exists(te_p):
        try:
            OOF.append(load_oof_log(oof_p).reshape(-1,1))
            TST.append(load_te_log(te_p).reshape(-1,1))
            names.append(name)
        except Exception as e:
            print(f"skip {name} -> {e}")

assert len(names) >= 2, "Need at least two OOF/Test pairs."
Xtr_log = np.hstack(OOF).astype('float32')  # (n, K)
Xte_log = np.hstack(TST).astype('float32')
K = Xtr_log.shape[1]
log(f"Using K log-preds: {K} {names}")

# ---------- seg flag (warning-free regex) ----------
t = df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat = r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all = ~t.str.contains(qty_pat, regex=True, na=False).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all = np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all = np.load("img_has.npy").astype(bool)
else:
    has_img_all = (df_all['image_link'].fillna('')!='').values
is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
seg_tr = (has_img_all & mmq_all)[is_tr].astype('float32').reshape(-1,1)
seg_te = (has_img_all & mmq_all)[is_te].astype('float32').reshape(-1,1)

# ---------- build meta features ----------
Xtr_sp = add_spreads(Xtr_log)
Xte_sp = add_spreads(Xte_log)
Xtr = np.hstack([Xtr_log, Xtr_sp, seg_tr]).astype('float32')
Xte = np.hstack([Xte_log, Xte_sp, seg_te]).astype('float32')
y   = train['price'].astype('float32').values
y_l = np.log(np.clip(y, 1e-3, None)).astype('float32')

log(f"Shapes | Xtr={Xtr.shape}, Xte={Xte.shape} (log-preds={K}, spreads={Xtr_sp.shape[1]}, seg=1)")

# ---------- weights & CV setup ----------
med = float(np.median(y))
w = (1.0 / (y + med)).astype('float32'); w /= w.mean()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)

params = dict(
    objective='regression_l1',
    num_leaves=31,
    max_depth=4,
    learning_rate=0.06,
    n_estimators=3000,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_samples=40,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42
)

# ---------- train CV (no fit(verbose=...) to avoid TypeError) ----------
oof_log = np.zeros_like(y_l)
te_log  = np.zeros(len(test), 'float32')
fold_smapes = []

for f,(tr_idx, va_idx) in enumerate(skf.split(Xtr, bins)):
    m = lgb.LGBMRegressor(**params)
    m.fit(
        Xtr[tr_idx], y_l[tr_idx],
        sample_weight=w[tr_idx],
        eval_set=[(Xtr[va_idx], y_l[va_idx])],
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(-1)  # silence per-iteration logs
        ]
    )
    oof_log[va_idx] = m.predict(Xtr[va_idx])
    te_log += m.predict(Xte) / skf.n_splits

    fold_cv = smape(y[va_idx], np.exp(oof_log[va_idx]).clip(1e-4))
    best_iter = getattr(m, "best_iteration_", None)
    log(f"[fold {f}] SMAPE={fold_cv:.3f}% | best_iter={best_iter}")
    fold_smapes.append(fold_cv)

cv_raw = smape(y, np.exp(oof_log).clip(1e-4))
log(f"[LGBM log+spreads] CV (raw): {cv_raw:.3f}% | mean±std per-fold: {np.mean(fold_smapes):.3f} ± {np.std(fold_smapes):.3f}")

# ---------- optional per-segment isotonic in log-space (auto-pick better) ----------
USE_ISO = True
final_oof = np.exp(oof_log).clip(1e-4)
final_te  = np.exp(te_log).clip(1e-4)
cv_iso = np.inf

if USE_ISO:
    A_tr_idx = np.where(seg_tr.ravel()==1)[0]
    B_tr_idx = np.where(seg_tr.ravel()==0)[0]
    A_te_idx = np.where(seg_te.ravel()==1)[0]
    B_te_idx = np.where(seg_te.ravel()==0)[0]

    def fit_iso(y_true, logp):
        if len(y_true) < 5:  # avoid degenerate fits
            return None
        order = np.argsort(logp)
        ir = IsotonicRegression(increasing=True, out_of_bounds='clip')
        ir.fit(logp[order], np.log(np.clip(y_true,1e-3,None))[order])
        return ir

    irA = fit_iso(y[A_tr_idx], oof_log[A_tr_idx]) if len(A_tr_idx) else None
    irB = fit_iso(y[B_tr_idx], oof_log[B_tr_idx]) if len(B_tr_idx) else None

    if (irA is not None) or (irB is not None):
        oof_iso_log = oof_log.copy()
        if irA is not None: oof_iso_log[A_tr_idx] = irA.predict(oof_log[A_tr_idx])
        if irB is not None: oof_iso_log[B_tr_idx] = irB.predict(oof_log[B_tr_idx])
        cv_iso = smape(y, np.exp(oof_iso_log).clip(1e-4))
        log(f"[LGBM log+spreads] CV (after seg isotonic): {cv_iso:.3f}%")

        if cv_iso < cv_raw:
            # apply same iso to test
            if irA is not None and len(A_te_idx):
                te_log[A_te_idx] = irA.predict(te_log[A_te_idx])
            if irB is not None and len(B_te_idx):
                te_log[B_te_idx] = irB.predict(te_log[B_te_idx])
            final_oof = np.exp(oof_iso_log).clip(1e-4)
            final_te  = np.exp(te_log).clip(1e-4)
            log("[isotonic] Using isotonic-calibrated predictions (better CV).")

# ---------- save ----------
os.makedirs("out", exist_ok=True)
oof_path = "out/oof_meta_lgbm_spreads.csv"
te_path  = "out/test_predictions_meta_lgbm_spreads.csv"
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': final_oof}).to_csv(oof_path, index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': final_te.astype(float)}).to_csv(te_path, index=False)

with open("out/meta_lgbm_spreads_meta.json","w") as f:
    json.dump({
        'models_used': names,
        'K_log_preds': int(K),
        'spreads_features': int(Xtr_sp.shape[1]),
        'cv_raw': float(cv_raw),
        'cv_folds_mean': float(np.mean(fold_smapes)),
        'cv_folds_std': float(np.std(fold_smapes)),
        'used_isotonic': bool(cv_iso < cv_raw),
        'cv_after_iso': (None if np.isinf(cv_iso) else float(cv_iso)),
        'params': {k:(int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v,(np.floating,)) else v) for k,v in params.items()}
    }, f, indent=2)

log(f"Saved -> {te_path}")


[15:33:39] Using K log-preds: 5 ['convex_seg_cal', 'residual_boost', 'cluster_cal', 'residual_fix', 'seg_superblend'] | +0.2s
[15:33:41] Shapes | Xtr=(75000, 26), Xte=(75000, 26) (log-preds=5, spreads=20, seg=1) | +2.3s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[128]	valid_0's l1: 0.533982
[15:33:46] [fold 0] SMAPE=48.611% | best_iter=128 | +7.1s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[96]	valid_0's l1: 0.53709
[15:33:50] [fold 1] SMAPE=48.810% | best_iter=96 | +11.3s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[107]	valid_0's l1: 0.548929
[15:33:54] [fold 2] SMAPE=49.754% | best_iter=107 | +15.7s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[109]	valid_0's l1: 0.5371
[15:33:59] [fold 3] SMAPE=48.783% | best_iter=109 | +20.2s
Training until validation scores don't improve

In [42]:
def load_te_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = np.log(np.clip(df['price'].values, 1e-4, None))
    return pd.Series(v, index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs = []
for name, oo, te in [
    ("meta_lgbm_spreads",         "out/oof_meta_lgbm_spreads.csv",                  "out/test_predictions_meta_lgbm_spreads.csv"),
    ("meta_lgbm_spreads_cl128",   "out/oof_meta_lgbm_spreads_cluster128.csv",       "out/test_predictions_meta_lgbm_spreads_cluster128.csv"),
    ("meta_lgbm_logpreds",        "out/oof_meta_lgbm_logpreds.csv",                 "out/test_predictions_meta_lgbm_logpreds.csv"),
    ("seg_superblend",            "out/oof_seg_superblend_cal.csv",                  "out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost",            "out/oof_residual_boost.csv",                      "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",               "out/oof_meta_after_cluster_cal.csv",              "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",              "out/oof_meta_after_residual_fix.csv",             "out/test_predictions_after_residual_fix.csv"),
    ("convex_seg_cal",            "out/oof_meta_convex_seg_cal.csv",                 "out/test_predictions_meta_convex_seg_cal.csv"),
]:
    if os.path.exists(oo) and os.path.exists(te):
        pairs.append((name, oo, te)); print("✓", name)

assert len(pairs)>=2
names, OOF, TST = [], [], []
for name, oo, te in pairs:
    names.append(name); OOF.append(load_oof_log(oo).reshape(-1,1)); TST.append(load_te_log(te).reshape(-1,1))

Xtr = np.hstack(OOF); Xte = np.hstack(TST); y = train['price'].values.astype('float32')

def best_w(Xs, ys, shots=5000):
    best = (1e9, None)
    K = Xs.shape[1]
    for _ in range(shots):
        w = np.random.dirichlet(np.ones(K)).astype('float32')
        s = smape(ys, np.exp(np.clip(Xs@w, -20, 20)).clip(1e-4))
        if s<best[0]: best=(s, w.copy())
    return best

cv, w = best_w(Xtr, y)
print(f"[final superblend] CV={cv:.3f}% | weights:", dict(zip(names, np.round(w,3))))
oo = np.exp(Xtr@w).clip(1e-4); te = np.exp(Xte@w).clip(1e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_superblend_final.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_superblend_final.csv", index=False)
print("Saved -> out/test_predictions_superblend_final.csv")


✓ meta_lgbm_spreads
✓ meta_lgbm_spreads_cl128
✓ meta_lgbm_logpreds
✓ seg_superblend
✓ residual_boost
✓ cluster_cal
✓ residual_fix
✓ convex_seg_cal
[final superblend] CV=46.859% | weights: {'meta_lgbm_spreads': 0.004, 'meta_lgbm_spreads_cl128': 0.097, 'meta_lgbm_logpreds': 0.744, 'seg_superblend': 0.072, 'residual_boost': 0.011, 'cluster_cal': 0.011, 'residual_fix': 0.05, 'convex_seg_cal': 0.011}
Saved -> out/test_predictions_superblend_final.csv


In [43]:
# === Cluster-128 calibration over meta_lgbm_logpreds (MiniLM -> PCA64 -> KMeans128) ===
import os, numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

base_oof = "out/oof_meta_lgbm_logpreds.csv"
base_te  = "out/test_predictions_meta_lgbm_logpreds.csv"
assert os.path.exists(base_oof) and os.path.exists(base_te), "Run meta_lgbm_logpreds first."

oo = pd.read_csv(base_oof).set_index('sample_id')['oof_price'].reindex(train['sample_id']).values.astype('float32')
te = pd.read_csv(base_te ).set_index('sample_id')['price'    ].reindex(test['sample_id']).values.astype('float32')
y  = train['price'].values.astype('float32')

# PCA64 of MiniLM, then KMeans128
emb_all = np.vstack([EMB_TR, EMB_TE]).astype('float32', copy=False)
Z = PCA(n_components=64, random_state=42).fit_transform(emb_all).astype('float32')
is_tr = (df_all['is_train'].values==1)
lab = KMeans(n_clusters=128, random_state=42, n_init='auto').fit_predict(Z)
lab_tr, lab_te = lab[is_tr], lab[~is_tr]

def fit_ab(y_true, y_pred):
    t = np.log(np.clip(y_true,1e-3,None))
    p = np.log(np.clip(y_pred,1e-3,None))
    X = np.column_stack([p, np.ones_like(p)])
    a,b = np.linalg.lstsq(X, t, rcond=None)[0]
    return float(a), float(b)

a_map, b_map = {}, {}
for k in range(128):
    idx = np.where(lab_tr==k)[0]
    if idx.size >= 30: a,b = fit_ab(y[idx], oo[idx])
    else:              a,b = 1.0, 0.0
    a_map[k], b_map[k] = a,b

def apply_ab(pred, a, b):
    return np.exp(a*np.log(np.clip(pred,1e-3,None)) + b)

oo_c = oo.copy()
for k in range(128):
    idx = np.where(lab_tr==k)[0]
    if idx.size: oo_c[idx] = apply_ab(oo[idx], a_map[k], b_map[k])

te_c = te.copy()
for k in range(128):
    idx = np.where(lab_te==k)[0]
    if idx.size: te_c[idx] = apply_ab(te[idx], a_map[k], b_map[k])
te_c = te_c.clip(1e-4)

print(f"[clust128] CV before={smape(y,oo):.3f}%, after={smape(y,oo_c):.3f}%")
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_c}).to_csv("out/oof_meta_lgbm_logpreds_cl128.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_c.astype(float)}).to_csv("out/test_predictions_meta_lgbm_logpreds_cl128.csv", index=False)
print("Saved -> out/test_predictions_meta_lgbm_logpreds_cl128.csv")


[clust128] CV before=46.855%, after=47.050%
Saved -> out/test_predictions_meta_lgbm_logpreds_cl128.csv


In [44]:
# === Price-bin convex blend (bins by baseline predictions) ===
import os, json, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_price(p):
    df=pd.read_csv(p).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"No OOF column in {p}")

def load_te_price(p):
    return pd.read_csv(p).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

# candidates: use the best ones you have
cand_files = [
    ("meta_lgbm_logpreds_cl128","out/oof_meta_lgbm_logpreds_cl128.csv","out/test_predictions_meta_lgbm_logpreds_cl128.csv"),
    ("meta_lgbm_logpreds",     "out/oof_meta_lgbm_logpreds.csv",      "out/test_predictions_meta_lgbm_logpreds.csv"),
    ("seg_superblend",         "out/oof_seg_superblend_cal.csv",      "out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost",         "out/oof_residual_boost.csv",          "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",            "out/oof_meta_after_cluster_cal.csv",  "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",           "out/oof_meta_after_residual_fix.csv", "out/test_predictions_after_residual_fix.csv"),
]
names, OOF, TST = [], [], []
for name, oo, te in cand_files:
    if os.path.exists(oo) and os.path.exists(te):
        try:
            OOF.append(load_oof_price(oo).reshape(-1,1))
            TST.append(load_te_price(te).reshape(-1,1))
            names.append(name)
            print("✓ using", name)
        except Exception as e:
            print("skip", name, "->", e)

assert len(names)>=2, "Need at least 2 candidates."
Xtr = np.hstack(OOF).astype('float32')      # price space
Xte = np.hstack(TST).astype('float32')
y   = train['price'].values.astype('float32')
K   = Xtr.shape[1]

# baseline for binning = the best available (prefer cl128 else base meta)
if "meta_lgbm_logpreds_cl128" in names:
    b_idx = names.index("meta_lgbm_logpreds_cl128")
else:
    b_idx = names.index("meta_lgbm_logpreds")
b_tr = Xtr[:, b_idx]
b_te = Xte[:, b_idx]

# build 10 bins by baseline predicted price
q = np.quantile(b_tr, np.linspace(0,1,11))
q[0] -= 1e-6; q[-1] += 1e-6
bin_id_tr = np.digitize(b_tr, q[1:-1], right=True)  # 0..9
bin_id_te = np.digitize(b_te, q[1:-1], right=True)  # 0..9

def best_w_bin(Xs, ys, shots=2500):
    best=(1e9,None)
    for _ in range(shots):
        w = np.random.dirichlet(np.ones(K)).astype('float32')
        pred = Xs @ w
        s = smape(ys, np.clip(pred, 1e-4, None))
        if s < best[0]: best=(s, w.copy())
    return best

# learn weights per bin (fallback to global if too small)
global_cv, global_w = best_w_bin(Xtr, y, shots=4000)
print(f"[global] CV={global_cv:.3f}%")

min_bin = 1500  # require enough samples in a bin to avoid noise
w_bins = np.tile(global_w, (10,1))
cv_bins = np.full(10, np.nan)
for b in range(10):
    idx = np.where(bin_id_tr==b)[0]
    if idx.size >= min_bin:
        cv, w = best_w_bin(Xtr[idx], y[idx], shots=3000)
        w_bins[b] = w
        cv_bins[b] = cv
        print(f"[bin {b}] n={idx.size} CV={cv:.3f}%")
    else:
        print(f"[bin {b}] n={idx.size} -> using global weights")

# smooth weights across bins (Gaussian on bin index)
from math import exp
def smooth_rows(W, sigma=1.0):
    B, K = W.shape
    W_s = np.zeros_like(W)
    for b in range(B):
        num = np.zeros(K, 'float64'); den = 0.0
        for j in range(B):
            g = exp(-0.5*((b-j)/sigma)**2)
            num += g * W[j]
            den += g
        W_s[b] = (num/den)
        W_s[b] /= W_s[b].sum()
    return W_s.astype('float32')

w_bins_s = smooth_rows(w_bins, sigma=1.0)

# stitch OOF/TEST
oo = np.zeros_like(y)
te = np.zeros(len(test), 'float32')
for b in range(10):
    wi = w_bins_s[b]
    idx_tr = np.where(bin_id_tr==b)[0]
    idx_te = np.where(bin_id_te==b)[0]
    if idx_tr.size: oo[idx_tr] = np.clip(Xtr[idx_tr] @ wi, 1e-4, None)
    if idx_te.size: te[idx_te] = np.clip(Xte[idx_te] @ wi, 1e-4, None)

cv_all = smape(y, oo)
print(f"[price-bin blend] CV={cv_all:.3f}% (global {global_cv:.3f}%)")

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_pricebin_blend.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_pricebin_blend.csv", index=False)
with open("out/pricebin_blend_meta.json","w") as f:
    json.dump({
        'names': names, 'weights_per_bin': w_bins_s.tolist(),
        'global_cv': float(global_cv), 'cv_bins': [None if np.isnan(x) else float(x) for x in cv_bins],
        'cv_all': float(cv_all), 'quantiles': q.tolist(), 'baseline': names[b_idx]
    }, f, indent=2)
print("Saved -> out/test_predictions_pricebin_blend.csv")


✓ using meta_lgbm_logpreds_cl128
✓ using meta_lgbm_logpreds
✓ using seg_superblend
✓ using residual_boost
✓ using cluster_cal
✓ using residual_fix
[global] CV=46.858%
[bin 0] n=7500 CV=47.848%
[bin 1] n=7500 CV=50.681%
[bin 2] n=7500 CV=52.422%
[bin 3] n=7500 CV=49.872%
[bin 4] n=7500 CV=49.591%
[bin 5] n=7500 CV=49.652%
[bin 6] n=7500 CV=48.087%
[bin 7] n=7500 CV=45.106%
[bin 8] n=7501 CV=41.439%
[bin 9] n=7499 CV=33.622%
[price-bin blend] CV=46.842% (global 46.858%)
Saved -> out/test_predictions_pricebin_blend.csv


In [45]:
# === Final quick super-blend including new models ===
import os, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = (np.log(np.clip(df['oof_price'].values,1e-4,None))
         if 'oof_price' in df else df.filter(like='oof').iloc[:,0].values)
    return pd.Series(v,index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = np.log(np.clip(df['price'].values,1e-4,None))
    return pd.Series(v,index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs=[]
for name, oo, te in [
    ("meta_lgbm_logpreds_cl128","out/oof_meta_lgbm_logpreds_cl128.csv","out/test_predictions_meta_lgbm_logpreds_cl128.csv"),
    ("pricebin_blend",         "out/oof_pricebin_blend.csv",          "out/test_predictions_pricebin_blend.csv"),
    ("meta_lgbm_logpreds",     "out/oof_meta_lgbm_logpreds.csv",      "out/test_predictions_meta_lgbm_logpreds.csv"),
    ("seg_superblend",         "out/oof_seg_superblend_cal.csv",      "out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost",         "out/oof_residual_boost.csv",          "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",            "out/oof_meta_after_cluster_cal.csv",  "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",           "out/oof_meta_after_residual_fix.csv", "out/test_predictions_after_residual_fix.csv"),
]:
    if os.path.exists(oo) and os.path.exists(te):
        pairs.append((name, oo, te)); print("✓", name)

assert len(pairs)>=2
names, OOF, TST = [], [], []
for name, oo, te in pairs:
    names.append(name); OOF.append(load_oof_log(oo).reshape(-1,1)); TST.append(load_te_log(te).reshape(-1,1))

Xtr=np.hstack(OOF); Xte=np.hstack(TST); y=train['price'].values.astype('float32')

def best_w(Xs, ys, shots=6000):
    best=(1e9,None)
    K=Xs.shape[1]
    for _ in range(shots):
        w=np.random.dirichlet(np.ones(K)).astype('float32')
        s=smape(ys, np.exp(np.clip(Xs@w, -20, 20)).clip(1e-4))
        if s<best[0]: best=(s, w.copy())
    return best

cv,w = best_w(Xtr,y)
print(f"[superblend final] CV={cv:.3f}% | weights:", dict(zip(names, np.round(w,3))))
oo=np.exp(Xtr@w).clip(1e-4); te=np.exp(Xte@w).clip(1e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_superblend_final.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_superblend_final.csv", index=False)
print("Saved -> out/test_predictions_superblend_final.csv")


✓ meta_lgbm_logpreds_cl128
✓ pricebin_blend
✓ meta_lgbm_logpreds
✓ seg_superblend
✓ residual_boost
✓ cluster_cal
✓ residual_fix
[superblend final] CV=46.850% | weights: {'meta_lgbm_logpreds_cl128': 0.076, 'pricebin_blend': 0.34, 'meta_lgbm_logpreds': 0.492, 'seg_superblend': 0.04, 'residual_boost': 0.001, 'cluster_cal': 0.011, 'residual_fix': 0.04}
Saved -> out/test_predictions_superblend_final.csv


In [46]:
# === Price–bin ridge calibration over meta_lgbm_logpreds (20 bins, smoothed) ===
import os, json, numpy as np, pandas as pd
from math import exp

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

base_oof = "out/oof_meta_lgbm_logpreds.csv"
base_te  = "out/test_predictions_meta_lgbm_logpreds.csv"
assert os.path.exists(base_oof) and os.path.exists(base_te), "Run meta_lgbm_logpreds first."

oo = pd.read_csv(base_oof).set_index('sample_id')['oof_price'].reindex(train['sample_id']).values.astype('float32')
te = pd.read_csv(base_te ).set_index('sample_id')['price'    ].reindex(test['sample_id']).values.astype('float32')
y  = train['price'].values.astype('float32')

# Work in log-space with floors
eps = 1e-4
p_tr = np.log(np.clip(oo, eps, None))
t_tr = np.log(np.clip(y,  eps, None))
p_te = np.log(np.clip(te, eps, None))

# Bin by *predicted* price (train), map test via same cutpoints
nbins = 20
cuts = np.quantile(oo, np.linspace(0,1,nbins+1))
cuts[0]  -= 1e-6
cuts[-1] += 1e-6
bin_tr = np.digitize(oo, cuts[1:-1], right=True)  # 0..nbins-1
bin_te = np.digitize(te, cuts[1:-1], right=True)

# Closed-form ridge with prior toward (a=1, b=0):
# minimize  ||Xθ - t||^2 + λ * (θ - θ0)^T I (θ - θ0), with θ0=[1,0]
def ridge_ab(p, t, lam=50.0):
    # Sufficient stats
    s_pp = float((p*p).sum())
    s_p  = float(p.sum())
    s_tt = float((t*t).sum())     # not used directly
    s_pt = float((p*t).sum())
    n    = float(len(p))
    # Normal eq: (X^T X + λI) θ = X^T t + λ θ0
    XtX = np.array([[s_pp, s_p],
                    [s_p,  n  ]], dtype='float64')
    Xty = np.array([s_pt, t.sum()], dtype='float64')
    Theta0 = np.array([1.0, 0.0], dtype='float64')
    A = XtX + lam*np.eye(2, dtype='float64')
    b = Xty + lam*Theta0
    a,b0 = np.linalg.solve(A,b)
    # Clamp to sane ranges
    a  = float(np.clip(a, 0.7, 1.3))
    b0 = float(np.clip(b0, -1.0, 1.0))
    return a,b0

# Fit per-bin with size-aware λ (smaller bins => stronger regularization)
counts = np.bincount(bin_tr, minlength=nbins)
a_raw  = np.zeros(nbins, dtype='float64')
b_raw  = np.zeros(nbins, dtype='float64')
global_a, global_b = ridge_ab(p_tr, t_tr, lam=5.0)  # very mild global reg
for b in range(nbins):
    idx = np.where(bin_tr==b)[0]
    if idx.size >= 200:
        lam = 50.0 * (300.0 / idx.size)  # scale λ down with bin size
        a_raw[b], b_raw[b] = ridge_ab(p_tr[idx], t_tr[idx], lam=lam)
    else:
        a_raw[b], b_raw[b] = global_a, global_b

# Smooth (a,b) over bins with Gaussian kernel
def smooth_vec(v, sigma=1.25):
    out = np.zeros_like(v, dtype='float64')
    for i in range(len(v)):
        num = 0.0; den = 0.0
        for j in range(len(v)):
            g = exp(-0.5*((i-j)/sigma)**2)
            num += g * v[j]; den += g
        out[i] = num/den
    return out

a_sm = smooth_vec(a_raw, sigma=1.25)
b_sm = smooth_vec(b_raw, sigma=1.25)

# Apply calibration per-bin: y_hat = exp(a*log(p) + b)
def apply_binwise(p_log, bins, a, b):
    out = np.empty_like(p_log, dtype='float64')
    for bb in range(nbins):
        idc = (bins==bb)
        if np.any(idc):
            out[idc] = np.exp(a[bb]*p_log[idc] + b[bb])
    return out

oo_cal = apply_binwise(p_tr, bin_tr, a_sm, b_sm).astype('float32').clip(eps)
te_cal = apply_binwise(p_te, bin_te, a_sm, b_sm).astype('float32').clip(eps)

cv_before = smape(y, oo)
cv_after  = smape(y, oo_cal)
print(f"[price-bin ridge cal] CV before={cv_before:.3f}%, after={cv_after:.3f}%")

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_cal}).to_csv(
    "out/oof_meta_lgbm_logpreds_pricebin_cal.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_cal.astype(float)}).to_csv(
    "out/test_predictions_meta_lgbm_logpreds_pricebin_cal.csv", index=False)

with open("out/pricebin_cal_meta.json","w") as f:
    json.dump({
        'nbins': nbins,
        'cuts': cuts.tolist(),
        'cv_before': float(cv_before),
        'cv_after': float(cv_after),
        'a_raw': a_raw.tolist(),
        'b_raw': b_raw.tolist(),
        'a_sm': a_sm.tolist(),
        'b_sm': b_sm.tolist(),
        'global_a': float(global_a),
        'global_b': float(global_b)
    }, f, indent=2)

print("Saved -> out/test_predictions_meta_lgbm_logpreds_pricebin_cal.csv")


[price-bin ridge cal] CV before=46.855%, after=47.030%
Saved -> out/test_predictions_meta_lgbm_logpreds_pricebin_cal.csv


In [47]:
# === Final quick super-blend including price-bin calibrated meta ===
import os, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = (np.log(np.clip(df['oof_price'].values,1e-4,None))
         if 'oof_price' in df else df.filter(like='oof').iloc[:,0].values)
    return pd.Series(v,index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df = pd.read_csv(p).set_index('sample_id')
    v = np.log(np.clip(df['price'].values,1e-4,None))
    return pd.Series(v,index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs=[]
for name, oo, te in [
    ("meta_lgbm_logpreds_pricebin_cal","out/oof_meta_lgbm_logpreds_pricebin_cal.csv","out/test_predictions_meta_lgbm_logpreds_pricebin_cal.csv"),
    ("pricebin_blend",                "out/oof_pricebin_blend.csv",                  "out/test_predictions_pricebin_blend.csv"),
    ("meta_lgbm_logpreds_cl128",      "out/oof_meta_lgbm_logpreds_cl128.csv",        "out/test_predictions_meta_lgbm_logpreds_cl128.csv"),
    ("meta_lgbm_logpreds",            "out/oof_meta_lgbm_logpreds.csv",              "out/test_predictions_meta_lgbm_logpreds.csv"),
    ("seg_superblend",                "out/oof_seg_superblend_cal.csv",               "out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost",                "out/oof_residual_boost.csv",                   "out/test_predictions_residual_boost.csv"),
    ("cluster_cal",                   "out/oof_meta_after_cluster_cal.csv",           "out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix",                  "out/oof_meta_after_residual_fix.csv",          "out/test_predictions_after_residual_fix.csv"),
]:
    if os.path.exists(oo) and os.path.exists(te):
        pairs.append((name, oo, te)); print("✓", name)

assert len(pairs)>=2
names, OOF, TST = [], [], []
for name, oo, te in pairs:
    names.append(name); OOF.append(load_oof_log(oo).reshape(-1,1)); TST.append(load_te_log(te).reshape(-1,1))

Xtr=np.hstack(OOF); Xte=np.hstack(TST); y=train['price'].values.astype('float32')

def best_w(Xs, ys, shots=6000):
    best=(1e9,None)
    K=Xs.shape[1]
    for _ in range(shots):
        w=np.random.dirichlet(np.ones(K)).astype('float32')
        s=smape(ys, np.exp(np.clip(Xs@w, -20, 20)).clip(1e-4))
        if s<best[0]: best=(s, w.copy())
    return best

cv,w = best_w(Xtr,y)
print(f"[superblend w/ pricebin-cal] CV={cv:.3f}% | weights:", dict(zip(names, np.round(w,3))))
oo=np.exp(Xtr@w).clip(1e-4); te=np.exp(Xte@w).clip(1e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_superblend_with_pricebincal.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_superblend_with_pricebincal.csv", index=False)
print("Saved -> out/test_predictions_superblend_with_pricebincal.csv")


✓ meta_lgbm_logpreds_pricebin_cal
✓ pricebin_blend
✓ meta_lgbm_logpreds_cl128
✓ meta_lgbm_logpreds
✓ seg_superblend
✓ residual_boost
✓ cluster_cal
✓ residual_fix
[superblend w/ pricebin-cal] CV=46.856% | weights: {'meta_lgbm_logpreds_pricebin_cal': 0.021, 'pricebin_blend': 0.153, 'meta_lgbm_logpreds_cl128': 0.027, 'meta_lgbm_logpreds': 0.643, 'seg_superblend': 0.017, 'residual_boost': 0.083, 'cluster_cal': 0.055, 'residual_fix': 0.001}
Saved -> out/test_predictions_superblend_with_pricebincal.csv


In [48]:
# === A) Seed-bagged LGBM meta (log-preds + seg flag) ===
import os, time, json
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression

t0=time.time(); log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    if   'oof_log' in df: v=df['oof_log'].values
    elif 'oof_log_router' in df: v=df['oof_log_router'].values
    elif 'oof' in df: v=df['oof'].values
    elif 'pred_log' in df: v=df['pred_log'].values
    elif 'oof_price' in df: v=np.log(np.clip(df['oof_price'].values,1e-4,None))
    else: raise ValueError(f"Unsupported OOF schema: {p}")
    return pd.Series(v,index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    v=np.log(np.clip(df['price'].values,1e-4,None))
    return pd.Series(v,index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs=[
    ("convex_seg_cal","out/oof_meta_convex_seg_cal.csv","out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost","out/oof_residual_boost.csv","out/test_predictions_residual_boost.csv"),
    ("cluster_cal","out/oof_meta_after_cluster_cal.csv","out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix","out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
    ("seg_superblend","out/oof_seg_superblend_cal.csv","out/test_predictions_seg_superblend_cal.csv"),
]
names,OOF,TST=[],[],[]
for name,o,t in pairs:
    if os.path.exists(o) and os.path.exists(t):
        OOF.append(load_oof_log(o).reshape(-1,1))
        TST.append(load_te_log(t).reshape(-1,1))
        names.append(name)
assert len(names)>=2
Xtr_log=np.hstack(OOF).astype('float32'); Xte_log=np.hstack(TST).astype('float32')
K=Xtr_log.shape[1]; log(f"K={K} models: {names}")

# seg flag (same rule as before; warning-free)
tcol=df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat=r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all=~tcol.str.contains(qty_pat,regex=True,na=False).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all=np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all=np.load("img_has.npy").astype(bool)
else:
    has_img_all=(df_all['image_link'].fillna('')!='').values
is_tr=(df_all['is_train'].values==1); is_te=~is_tr
seg_tr=(has_img_all & mmq_all)[is_tr].astype('float32').reshape(-1,1)
seg_te=(has_img_all & mmq_all)[is_te].astype('float32').reshape(-1,1)

# design: [log-preds..., seg_flag]  (simple & strong)
Xtr=np.hstack([Xtr_log, seg_tr]).astype('float32')
Xte=np.hstack([Xte_log, seg_te]).astype('float32')
y=train['price'].values.astype('float32')
y_l=np.log(np.clip(y,1e-3,None)).astype('float32')

# weighted for SMAPE-ish
med=float(np.median(y)); w=(1.0/(y+med)).astype('float32'); w/=w.mean()
skf=StratifiedKFold(5,shuffle=True,random_state=42)
bins=pd.qcut(y,q=20,duplicates='drop').astype(str)

base_params=dict(
    objective='regression_l1',
    num_leaves=31, max_depth=4, learning_rate=0.06,
    n_estimators=2600, subsample=0.85, colsample_bytree=0.85,
    min_child_samples=40, reg_lambda=1.0, n_jobs=-1
)
seeds=[13,27,42,99,202,777,1201]

oof_sum=np.zeros_like(y_l); te_sum=np.zeros(len(test),'float32'); fold_smapes=[]
for s in seeds:
    oof=np.zeros_like(y_l); te=np.zeros(len(test),'float32')
    for f,(tr,va) in enumerate(skf.split(Xtr,bins)):
        m=lgb.LGBMRegressor(random_state=s, **base_params)
        m.fit(Xtr[tr],y_l[tr], sample_weight=w[tr],
              eval_set=[(Xtr[va],y_l[va])],
              callbacks=[lgb.early_stopping(200), lgb.log_evaluation(-1)])
        oof[va]=m.predict(Xtr[va]); te+=m.predict(Xte)/skf.n_splits
    oof_sum+=oof; te_sum+=te
    fold_smapes.append(smape(y,np.exp(oof).clip(1e-4)))
    log(f"[seed {s}] OOF SMAPE={fold_smapes[-1]:.3f}%")

oof_avg=oof_sum/len(seeds); te_avg=te_sum/len(seeds)
cv_raw=smape(y,np.exp(oof_avg).clip(1e-4))
log(f"[bagged LGBM] CV (raw): {cv_raw:.3f}% | seeds={len(seeds)}")

# seg-isotonic (auto-use if better)
A=np.where(seg_tr.ravel()==1)[0]; B=np.where(seg_tr.ravel()==0)[0]
def fit_iso(y_true, logp):
    if len(y_true)<5: return None
    order=np.argsort(logp)
    ir=IsotonicRegression(increasing=True,out_of_bounds='clip')
    ir.fit(logp[order], np.log(np.clip(y_true,1e-3,None))[order]); return ir

irA=fit_iso(y[A],oof_avg[A]); irB=fit_iso(y[B],oof_avg[B])
oof_iso=oof_avg.copy()
if irA is not None: oof_iso[A]=irA.predict(oof_avg[A])
if irB is not None: oof_iso[B]=irB.predict(oof_avg[B])
cv_iso=smape(y,np.exp(oof_iso).clip(1e-4))
use_iso=cv_iso<cv_raw; log(f"[bagged LGBM] CV after seg-iso: {cv_iso:.3f}% (better={use_iso})")

te_final=te_avg.copy()
if use_iso:
    A_te=np.where(seg_te.ravel()==1)[0]; B_te=np.where(seg_te.ravel()==0)[0]
    if irA is not None and len(A_te): te_final[A_te]=irA.predict(te_avg[A_te])
    if irB is not None and len(B_te): te_final[B_te]=irB.predict(te_avg[B_te])

oo_out=np.exp(oof_iso if use_iso else oof_avg).clip(1e-4)
te_out=np.exp(te_final).clip(1e-4)

os.makedirs("out",exist_ok=True)
pd.DataFrame({'sample_id':train['sample_id'],'oof_price':oo_out}).to_csv("out/oof_meta_lgbm_bag.csv",index=False)
pd.DataFrame({'sample_id':test['sample_id'],'price':te_out.astype(float)}).to_csv("out/test_predictions_meta_lgbm_bag.csv",index=False)
print("Saved -> out/test_predictions_meta_lgbm_bag.csv")


[15:49:46] K=5 models: ['convex_seg_cal', 'residual_boost', 'cluster_cal', 'residual_fix', 'seg_superblend'] | +0.2s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[159]	valid_0's l1: 0.527491
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[293]	valid_0's l1: 0.532012
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[113]	valid_0's l1: 0.540957
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[179]	valid_0's l1: 0.529197
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[157]	valid_0's l1: 0.522713
[15:50:14] [seed 13] OOF SMAPE=48.397% | +28.1s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[169]	valid_0's l1: 0.527525
Training until validation scores don't improve for 200 rounds
Early stopping, b

In [49]:
# === B) Segment-aware coordinate-descent blend (SMAPE; simplex weights) ===
import os, numpy as np, pandas as pd, json, random

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_price(p):
    df=pd.read_csv(p).set_index('sample_id')
    for c in ["oof_price","oof_log","oof_log_router","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"no OOF col in {p}")

def load_te_price(p):
    return pd.read_csv(p).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

cand = []
for name,oo,te in [
    ("meta_lgbm_logpreds","out/oof_meta_lgbm_logpreds.csv","out/test_predictions_meta_lgbm_logpreds.csv"),
    ("pricebin_blend","out/oof_pricebin_blend.csv","out/test_predictions_pricebin_blend.csv"),
    ("seg_superblend","out/oof_seg_superblend_cal.csv","out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost","out/oof_residual_boost.csv","out/test_predictions_residual_boost.csv"),
    ("cluster_cal","out/oof_meta_after_cluster_cal.csv","out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix","out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
]:
    if os.path.exists(oo) and os.path.exists(te):
        cand.append((name, load_oof_price(oo), load_te_price(te)))
names=[c[0] for c in cand]; assert len(names)>=2
Xtr=np.column_stack([c[1] for c in cand]).astype('float32')
Xte=np.column_stack([c[2] for c in cand]).astype('float32')
y=train['price'].values.astype('float32'); K=Xtr.shape[1]
print("Blend K:",K,names)

# segments
t=df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat=r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all=~t.str.contains(qty_pat,regex=True,na=False).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all=np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all=np.load("img_has.npy").astype(bool)
else:
    has_img_all=(df_all['image_link'].fillna('')!='').values
is_tr=(df_all['is_train'].values==1); is_te=~is_tr
A_tr=np.where((has_img_all & mmq_all)[is_tr])[0]; B_tr=np.where(~(has_img_all & mmq_all)[is_tr])[0]
A_te=np.where((has_img_all & mmq_all)[is_te])[0]; B_te=np.where(~(has_img_all & mmq_all)[is_te])[0]

def proj_simplex(w):
    # Euclidean projection to simplex
    u=np.sort(w)[::-1]; css=np.cumsum(u); rho=np.where(u - (css-1)/ (np.arange(len(u))+1) > 0)[0][-1]
    theta=(css[rho]-1)/(rho+1); return np.maximum(w-theta,0.0)

def cd_opt(X, y, idx, shots=3000, iters=200, step=0.05, seed=42):
    rng=np.random.default_rng(seed)
    best_s=1e9; best_w=None
    # multistart (Dirichlet)
    for _ in range(shots):
        w=rng.dirichlet(np.ones(K)).astype('float32')
        s=smape(y[idx], np.clip(X[idx]@w,1e-4,None))
        if s<best_s: best_s, best_w = s, w.copy()
    # local coordinate descent with pairwise transfers
    w=best_w.copy()
    for _ in range(iters):
        i,j = rng.integers(0,K), rng.integers(0,K)
        if i==j: continue
        delta = step * (rng.random()*2-1)  # [-step, step]
        w_try = w.copy(); w_try[i]+=delta; w_try[j]-=delta; w_try=proj_simplex(w_try)
        s_try = smape(y[idx], np.clip(X[idx]@w_try,1e-4,None))
        if s_try < best_s: best_s, w = s_try, w_try
        step *= 0.995
    return w, best_s

wA, cvA = cd_opt(Xtr,y,A_tr,shots=2500,iters=250,step=0.07,seed=1)
wB, cvB = cd_opt(Xtr,y,B_tr,shots=2500,iters=250,step=0.07,seed=2)
print("segA:",cvA,"segB:",cvB)

oo=np.zeros_like(y)
oo[A_tr]=np.clip(Xtr[A_tr]@wA,1e-4,None)
oo[B_tr]=np.clip(Xtr[B_tr]@wB,1e-4,None)
cv_all=smape(y,oo); print("CV all:",cv_all)

te=np.zeros(len(test),'float32')
te[A_te]=np.clip(Xte[A_te]@wA,1e-4,None)
te[B_te]=np.clip(Xte[B_te]@wB,1e-4,None)

os.makedirs("out",exist_ok=True)
pd.DataFrame({'sample_id':train['sample_id'],'oof_price':oo}).to_csv("out/oof_seg_cd_blend.csv",index=False)
pd.DataFrame({'sample_id':test['sample_id'],'price':te.astype(float)}).to_csv("out/test_predictions_seg_cd_blend.csv",index=False)
with open("out/seg_cd_blend_meta.json","w") as f:
    json.dump({'names':names,'wA':wA.tolist(),'wB':wB.tolist(),'cvA':float(cvA),'cvB':float(cvB),'cv_all':float(cv_all)},f,indent=2)
print("Saved -> out/test_predictions_seg_cd_blend.csv")


Blend K: 6 ['meta_lgbm_logpreds', 'pricebin_blend', 'seg_superblend', 'residual_boost', 'cluster_cal', 'residual_fix']
segA: 52.02556205524955 segB: 44.39486993804243
CV all: 46.836182702740885
Saved -> out/test_predictions_seg_cd_blend.csv


In [50]:
# === C) Quadratic Ridge meta on log-preds (+ seg) ===
import os, json, numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold

def smape(y_true,y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    if   'oof_log' in df: v=df['oof_log'].values
    elif 'oof_log_router' in df: v=df['oof_log_router'].values
    elif 'oof' in df: v=df['oof'].values
    elif 'pred_log' in df: v=df['pred_log'].values
    elif 'oof_price' in df: v=np.log(np.clip(df['oof_price'].values,1e-4,None))
    else: raise ValueError
    return pd.Series(v,index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    v=np.log(np.clip(df['price'].values,1e-4,None))
    return pd.Series(v,index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs=[
    ("convex_seg_cal","out/oof_meta_convex_seg_cal.csv","out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost","out/oof_residual_boost.csv","out/test_predictions_residual_boost.csv"),
    ("cluster_cal","out/oof_meta_after_cluster_cal.csv","out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix","out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
    ("seg_superblend","out/oof_seg_superblend_cal.csv","out/test_predictions_seg_superblend_cal.csv"),
]
OOF=[]; TST=[]; names=[]
for name,o,t in pairs:
    if os.path.exists(o) and os.path.exists(t):
        OOF.append(load_oof_log(o).reshape(-1,1))
        TST.append(load_te_log(t).reshape(-1,1))
        names.append(name)
assert len(names)>=2
Ltr=np.hstack(OOF); Lte=np.hstack(TST); K=Ltr.shape[1]

# seg flag
tcol=df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat=r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all=~tcol.str.contains(qty_pat,regex=True,na=False).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all=np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all=np.load("img_has.npy").astype(bool)
else:
    has_img_all=(df_all['image_link'].fillna('')!='').values
is_tr=(df_all['is_train'].values==1); is_te=~is_tr
seg_tr=(has_img_all & mmq_all)[is_tr].astype('float32').reshape(-1,1)
seg_te=(has_img_all & mmq_all)[is_te].astype('float32').reshape(-1,1)

# quadratic features: [L, L^2, pairwise L_i*L_j, seg, seg*L]
def quad_feats(L, seg):
    n,K=L.shape
    parts=[L]
    parts.append(L**2)
    prods=[]
    for i in range(K):
        for j in range(i+1,K):
            prods.append((L[:,i]*L[:,j]).reshape(-1,1))
    parts.append(np.hstack(prods) if prods else np.empty((n,0),'float32'))
    parts.append(seg)
    parts.append(L*seg)
    return np.hstack(parts).astype('float32')

Xtr=quad_feats(Ltr, seg_tr); Xte=quad_feats(Lte, seg_te)
y=train['price'].values.astype('float32'); y_log=np.log(np.clip(y,1e-3,None)).astype('float32')

skf=StratifiedKFold(5,shuffle=True,random_state=42)
bins=pd.qcut(y,q=20,duplicates='drop').astype(str)
alphas=[0.02,0.05,0.1,0.2,0.3,0.5]
best={'cv':1e9}
for a in alphas:
    oof=np.zeros_like(y_log); te=np.zeros(len(test),'float32')
    for tr,va in skf.split(Xtr,bins):
        m=Ridge(alpha=a, random_state=42)
        m.fit(Xtr[tr], y_log[tr])
        oof[va]=m.predict(Xtr[va]); te+=m.predict(Xte)/skf.n_splits
    cv=smape(y,np.exp(oof).clip(1e-4))
    print(f"[alpha={a}] CV={cv:.3f}%")
    if cv<best['cv']: best={'alpha':a,'cv':cv,'oof':oof.copy(),'te':te.copy()}

print(f"[ridge-quad] best alpha={best['alpha']} | CV={best['cv']:.3f}%")
oo=np.exp(best['oof']).clip(1e-4); te=np.exp(best['te']).clip(1e-4)

os.makedirs("out",exist_ok=True)
pd.DataFrame({'sample_id':train['sample_id'],'oof_price':oo}).to_csv("out/oof_meta_ridge_quad.csv",index=False)
pd.DataFrame({'sample_id':test['sample_id'],'price':te.astype(float)}).to_csv("out/test_predictions_meta_ridge_quad.csv",index=False)
print("Saved -> out/test_predictions_meta_ridge_quad.csv")


[alpha=0.02] CV=47.155%
[alpha=0.05] CV=47.154%
[alpha=0.1] CV=47.153%
[alpha=0.2] CV=47.152%
[alpha=0.3] CV=47.151%


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.26012e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


[alpha=0.5] CV=47.153%
[ridge-quad] best alpha=0.3 | CV=47.151%
Saved -> out/test_predictions_meta_ridge_quad.csv


In [51]:
# === Per-segment LGBM meta (log-preds only) + seed bagging + seg isotonic ===
import os, time, numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression

t0=time.time(); log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    if   'oof_log' in df: v=df['oof_log'].values
    elif 'oof_log_router' in df: v=df['oof_log_router'].values
    elif 'oof' in df: v=df['oof'].values
    elif 'pred_log' in df: v=df['pred_log'].values
    elif 'oof_price' in df: v=np.log(np.clip(df['oof_price'].values,1e-4,None))
    else: raise ValueError(f"Unsupported OOF schema: {p}")
    return pd.Series(v,index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    v=np.log(np.clip(df['price'].values,1e-4,None))
    return pd.Series(v,index=df.index).reindex(test['sample_id']).values.astype('float32')

# Use the same 5 strong models (log features only)
pairs=[
    ("convex_seg_cal","out/oof_meta_convex_seg_cal.csv","out/test_predictions_meta_convex_seg_cal.csv"),
    ("residual_boost","out/oof_residual_boost.csv","out/test_predictions_residual_boost.csv"),
    ("cluster_cal","out/oof_meta_after_cluster_cal.csv","out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix","out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
    ("seg_superblend","out/oof_seg_superblend_cal.csv","out/test_predictions_seg_superblend_cal.csv"),
]
names,OOF,TST=[],[],[]
for name,o,t in pairs:
    if os.path.exists(o) and os.path.exists(t):
        OOF.append(load_oof_log(o).reshape(-1,1))
        TST.append(load_te_log(t).reshape(-1,1))
        names.append(name)
assert len(names)>=2, "Need at least two OOF/Test pairs."
Ltr=np.hstack(OOF).astype('float32'); Lte=np.hstack(TST).astype('float32')
K=Ltr.shape[1]; log(f"K log-preds={K} | {names}")

# Segment A = has_img ∧ missing_qty (same rule; warning-free regex)
tcol=df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat=r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all=~tcol.str.contains(qty_pat,regex=True,na=False).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all=np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all=np.load("img_has.npy").astype(bool)
else:
    has_img_all=(df_all['image_link'].fillna('')!='').values

is_tr=(df_all['is_train'].values==1); is_te=~is_tr
segA_tr=(has_img_all & mmq_all)[is_tr]
segA_te=(has_img_all & mmq_all)[is_te]

A_tr=np.where(segA_tr)[0]; B_tr=np.where(~segA_tr)[0]
A_te=np.where(segA_te)[0]; B_te=np.where(~segA_te)[0]
y=train['price'].values.astype('float32'); y_log=np.log(np.clip(y,1e-3,None)).astype('float32')

# SMAPE-ish weights
med=float(np.median(y)); w=(1.0/(y+med)).astype('float32'); w/=w.mean()

# Per-segment CV splits (stratify on full y, then mask inside loop)
skf=StratifiedKFold(5,shuffle=True,random_state=42)
bins=pd.qcut(y,q=20,duplicates='drop').astype(str)

# Model params
base_params=dict(
    objective='regression_l1',
    num_leaves=31,
    max_depth=4,
    learning_rate=0.06,
    n_estimators=2600,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_samples=40,
    reg_lambda=1.0,
    n_jobs=-1,
)

seeds=[11,23,42,77,101]  # lighter bag

def fit_per_segment(Ltr, y_log, w, A_tr, B_tr, Lte, A_te, B_te, seeds):
    oof=np.zeros_like(y_log); te=np.zeros(len(test),'float32')
    for s in seeds:
        oof_s=np.zeros_like(y_log); te_s=np.zeros(len(test),'float32')
        for f,(tr,va) in enumerate(skf.split(Ltr,bins)):
            # --- SEG A ---
            trA=np.intersect1d(tr, A_tr, assume_unique=False); vaA=np.intersect1d(va, A_tr, assume_unique=False)
            if trA.size>=500 and vaA.size>0:
                mA=lgb.LGBMRegressor(random_state=s, **base_params)
                mA.fit(Ltr[trA], y_log[trA], sample_weight=w[trA],
                       eval_set=[(Ltr[vaA], y_log[vaA])],
                       callbacks=[lgb.early_stopping(200), lgb.log_evaluation(-1)])
                oof_s[vaA]=mA.predict(Ltr[vaA])
                te_s[A_te]+=mA.predict(Lte[A_te])/ (skf.n_splits)
            # --- SEG B ---
            trB=np.intersect1d(tr, B_tr, assume_unique=False); vaB=np.intersect1d(va, B_tr, assume_unique=False)
            if trB.size>=500 and vaB.size>0:
                mB=lgb.LGBMRegressor(random_state=s, **base_params)
                mB.fit(Ltr[trB], y_log[trB], sample_weight=w[trB],
                       eval_set=[(Ltr[vaB], y_log[vaB])],
                       callbacks=[lgb.early_stopping(200), lgb.log_evaluation(-1)])
                oof_s[vaB]=mB.predict(Ltr[vaB])
                te_s[B_te]+=mB.predict(Lte[B_te])/ (skf.n_splits)
        oof += oof_s/len(seeds); te += te_s/len(seeds)
        log(f"[seed {s}] OOF SMAPE={smape(y, np.exp(oof_s).clip(1e-4)):.3f}%")
    return oof, te

oof_log, te_log = fit_per_segment(Ltr, y_log, w, A_tr, B_tr, Lte, A_te, B_te, seeds)
cv_raw = smape(y, np.exp(oof_log).clip(1e-4))
log(f"[per-seg LGBM] CV(raw)={cv_raw:.3f}%")

# Seg-wise isotonic (in log-space)
def fit_iso(y_true, logp):
    if len(y_true)<5: return None
    order=np.argsort(logp)
    ir=IsotonicRegression(increasing=True, out_of_bounds='clip')
    ir.fit(logp[order], np.log(np.clip(y_true,1e-3,None))[order])
    return ir

irA=fit_iso(y[A_tr], oof_log[A_tr]); irB=fit_iso(y[B_tr], oof_log[B_tr])

oof_iso=oof_log.copy()
if irA is not None: oof_iso[A_tr]=irA.predict(oof_log[A_tr])
if irB is not None: oof_iso[B_tr]=irB.predict(oof_log[B_tr])

cv_iso = smape(y, np.exp(oof_iso).clip(1e-4))
use_iso = cv_iso < cv_raw
log(f"[per-seg LGBM] CV(after seg-iso)={cv_iso:.3f}%  (better={use_iso})")

# Apply iso to test if better
te_final = te_log.copy()
if use_iso:
    if irA is not None and len(A_te): te_final[A_te]=irA.predict(te_log[A_te])
    if irB is not None and len(B_te): te_final[B_te]=irB.predict(te_log[B_te])

oo = np.exp(oof_iso if use_iso else oof_log).clip(1e-4)
te = np.exp(te_final).clip(1e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_meta_lgbm_perseg.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_meta_lgbm_perseg.csv", index=False)
print("Saved -> out/test_predictions_meta_lgbm_perseg.csv")


[15:57:22] K log-preds=5 | ['convex_seg_cal', 'residual_boost', 'cluster_cal', 'residual_fix', 'seg_superblend'] | +0.2s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[239]	valid_0's l1: 0.595918
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[165]	valid_0's l1: 0.495951
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[130]	valid_0's l1: 0.607218
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[493]	valid_0's l1: 0.499765
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[110]	valid_0's l1: 0.625282
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[90]	valid_0's l1: 0.504226
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[125]	valid_0's l1: 0.60984

In [52]:
# === Re-super-blend including per-seg meta ===
import os, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    v=(np.log(np.clip(df['oof_price'].values,1e-4,None))
       if 'oof_price' in df else df.filter(like='oof').iloc[:,0].values)
    return pd.Series(v,index=df.index).reindex(train['sample_id']).values.astype('float32')

def load_te_log(p):
    df=pd.read_csv(p).set_index('sample_id')
    v=np.log(np.clip(df['price'].values,1e-4,None))
    return pd.Series(v,index=df.index).reindex(test['sample_id']).values.astype('float32')

pairs=[]
for name,oo,te in [
    ("meta_lgbm_perseg","out/oof_meta_lgbm_perseg.csv","out/test_predictions_meta_lgbm_perseg.csv"),
    ("meta_lgbm_logpreds","out/oof_meta_lgbm_logpreds.csv","out/test_predictions_meta_lgbm_logpreds.csv"),
    ("seg_cd_blend","out/oof_seg_cd_blend.csv","out/test_predictions_seg_cd_blend.csv"),
    ("pricebin_blend","out/oof_pricebin_blend.csv","out/test_predictions_pricebin_blend.csv"),
    ("seg_superblend","out/oof_seg_superblend_cal.csv","out/test_predictions_seg_superblend_cal.csv"),
    ("residual_boost","out/oof_residual_boost.csv","out/test_predictions_residual_boost.csv"),
    ("cluster_cal","out/oof_meta_after_cluster_cal.csv","out/test_predictions_after_cluster_cal.csv"),
    ("residual_fix","out/oof_meta_after_residual_fix.csv","out/test_predictions_after_residual_fix.csv"),
]:
    if os.path.exists(oo) and os.path.exists(te):
        pairs.append((name,oo,te)); print("✓", name)

assert len(pairs)>=2
names,OOF,TST=[],[],[]
for name,oo,te in pairs:
    names.append(name); OOF.append(load_oof_log(oo).reshape(-1,1)); TST.append(load_te_log(te).reshape(-1,1))

Xtr=np.hstack(OOF); Xte=np.hstack(TST); y=train['price'].values.astype('float32')

def best_w(Xs, ys, shots=7000):
    best=(1e9,None); K=Xs.shape[1]
    rng=np.random.default_rng(42)
    for _ in range(shots):
        w=rng.dirichlet(np.ones(K)).astype('float32')
        s=smape(ys, np.exp(np.clip(Xs@w, -20, 20)).clip(1e-4))
        if s<best[0]: best=(s, w.copy())
    return best

cv,w = best_w(Xtr,y)
print(f"[re-superblend] CV={cv:.3f}% | weights:", dict(zip(names, np.round(w,3))))
oo=np.exp(Xtr@w).clip(1e-4); te=np.exp(Xte@w).clip(1e-4)

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo}).to_csv("out/oof_superblend_round3.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te.astype(float)}).to_csv("out/test_predictions_superblend_round3.csv", index=False)
print("Saved -> out/test_predictions_superblend_round3.csv")


✓ meta_lgbm_perseg
✓ meta_lgbm_logpreds
✓ seg_cd_blend
✓ pricebin_blend
✓ seg_superblend
✓ residual_boost
✓ cluster_cal
✓ residual_fix
[re-superblend] CV=46.846% | weights: {'meta_lgbm_perseg': 0.006, 'meta_lgbm_logpreds': 0.062, 'seg_cd_blend': 0.71, 'pricebin_blend': 0.114, 'seg_superblend': 0.058, 'residual_boost': 0.022, 'cluster_cal': 0.002, 'residual_fix': 0.025}
Saved -> out/test_predictions_superblend_round3.csv


In [53]:
# === Per-segment logistic gate: seg_cd_blend ↔ meta_lgbm_logpreds ===
import os, json, numpy as np, pandas as pd, time
t0=time.time(); log=lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def load_oof_price(path):
    df=pd.read_csv(path).set_index('sample_id')
    for c in ["oof_price","oof_log_router","oof_log","oof"]:
        if c in df.columns:
            v=df[c].reindex(train['sample_id']).values
            return (np.exp(v) if "log" in c else v).astype('float32')
    raise RuntimeError(f"No OOF column in {path}")

def load_te_price(path):
    return pd.read_csv(path).set_index('sample_id')['price'].reindex(test['sample_id']).values.astype('float32')

# --- sources
A_name, A_oof_p, A_te_p = "seg_cd_blend", "out/oof_seg_cd_blend.csv", "out/test_predictions_seg_cd_blend.csv"
B_name, B_oof_p, B_te_p = "meta_lgbm_logpreds", "out/oof_meta_lgbm_logpreds.csv", "out/test_predictions_meta_lgbm_logpreds.csv"
assert os.path.exists(A_oof_p) and os.path.exists(A_te_p) and os.path.exists(B_oof_p) and os.path.exists(B_te_p)

y  = train['price'].values.astype('float32')
A_oo, A_te = load_oof_price(A_oof_p), load_te_price(A_te_p)
B_oo, B_te = load_oof_price(B_oof_p), load_te_price(B_te_p)

# --- segment masks: A = has_img & missing_qty (warning-free regex)
tcol=df_all['catalog_content'].fillna('').astype(str).str.lower()
qty_pat=r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
mmq_all=~tcol.strcontains(qty_pat,regex=True,na=False).values if hasattr(pd.Series.str,'strcontains') else ~tcol.str.contains(qty_pat,regex=True,na=False).values
if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
    has_img_all=np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
elif os.path.exists("img_has.npy"):
    has_img_all=np.load("img_has.npy").astype(bool)
else:
    has_img_all=(df_all['image_link'].fillna('')!='').values

is_tr=(df_all['is_train'].values==1); is_te=(~is_tr)
segA_all=(has_img_all & mmq_all)
A_tr=np.where(is_tr & segA_all)[0]
B_tr=np.where(is_tr & ~segA_all)[0]
A_te_idx=(np.where(is_te & segA_all)[0]-len(train)).astype(int)
B_te_idx=(np.where(is_te & ~segA_all)[0]-len(train)).astype(int)
A_te_idx=A_te_idx[(A_te_idx>=0)&(A_te_idx<len(test))]
B_te_idx=B_te_idx[(B_te_idx>=0)&(B_te_idx<len(test))]

# --- logistic gate per segment: w = 1/(1+exp(-k*(spread - t)))
def best_gate_params(y_true, P1, P2, idx, k_grid=(0.5,1,1.5,2,3,4,6,8), t_q=(0.2,0.35,0.5,0.65,0.8)):
    eps=1e-4
    p1=np.log(np.clip(P1[idx],eps,None)); p2=np.log(np.clip(P2[idx],eps,None))
    spread=p1-p2
    thr_list=np.quantile(spread, t_q)
    best=(1e9, None)
    for k in k_grid:
        for t in thr_list:
            w=1.0/(1.0+np.exp(-k*(spread - t)))
            pred=np.exp(w*p1 + (1-w)*p2).clip(eps)
            s=smape(y_true[idx], pred)
            if s<best[0]: best=(s, (k, float(t)))
    return best  # (cv, (k, t))

# --- fit params per segment
cvA,(kA,tA)=best_gate_params(y, A_oo, B_oo, A_tr)
cvB,(kB,tB)=best_gate_params(y, A_oo, B_oo, B_tr)
log(f"[gate params] segA cv={cvA:.3f}% k={kA} t={tA:.4f} | segB cv={cvB:.3f}% k={kB} t={tB:.4f}")

# --- apply gate to build OOF
eps=1e-4
logA_ooA, logA_ooB = np.log(np.clip(A_oo,eps,None)), np.log(np.clip(B_oo,eps,None))
spreadA = logA_ooA - logA_ooB
w_tr = np.zeros_like(y, dtype='float32')
w_tr[A_tr] = 1/(1+np.exp(-kA*(spreadA[A_tr]-tA)))
w_tr[B_tr] = 1/(1+np.exp(-kB*(spreadA[B_tr]-tB)))
oo_gate = np.exp(w_tr*logA_ooA + (1-w_tr)*logA_ooB).clip(eps)

cv_gate = smape(y, oo_gate)
log(f"[gated blend] CV (pre-cal): {cv_gate:.3f}%")

# --- tiny per-seg log calibration (LS) on top of gated
def fit_ab(y_true, p):
    t=np.log(np.clip(y_true,eps,None)); x=np.log(np.clip(p,eps,None))
    X=np.column_stack([x, np.ones_like(x)])
    a,b=np.linalg.lstsq(X,t,rcond=None)[0]
    return float(a), float(b)

def apply_ab(p,a,b): return np.exp(a*np.log(np.clip(p,eps,None))+b)

aA,bA=fit_ab(y[A_tr], oo_gate[A_tr]) if len(A_tr) else (1.0,0.0)
aB,bB=fit_ab(y[B_tr], oo_gate[B_tr]) if len(B_tr) else (1.0,0.0)

oo_cal=oo_gate.copy()
if len(A_tr): oo_cal[A_tr]=apply_ab(oo_cal[A_tr],aA,bA)
if len(B_tr): oo_cal[B_tr]=apply_ab(oo_cal[B_tr],aB,bB)
cv_cal=smape(y, oo_cal); log(f"[gated blend] CV (post-cal): {cv_cal:.3f}%")

# --- predict TEST
logA_teA, logA_teB = np.log(np.clip(A_te,eps,None)), np.log(np.clip(B_te,eps,None))
spreadT = logA_teA - logA_teB
w_te = np.zeros(len(test), dtype='float32')
w_te[A_te_idx] = 1/(1+np.exp(-kA*(spreadT[A_te_idx]-tA)))
w_te[B_te_idx] = 1/(1+np.exp(-kB*(spreadT[B_te_idx]-tB)))
te_gate = np.exp(w_te*logA_teA + (1-w_te)*logA_teB).clip(eps)

te_cal = te_gate.copy()
if len(A_te_idx): te_cal[A_te_idx]=apply_ab(te_gate[A_te_idx],aA,bA)
if len(B_te_idx): te_cal[B_te_idx]=apply_ab(te_gate[B_te_idx],aB,bB)
te_cal = te_cal.clip(eps)

# --- save
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': oo_cal}).to_csv("out/oof_meta_gated_seg.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': te_cal.astype(float)}).to_csv("out/test_predictions_meta_gated_seg.csv", index=False)
with open("out/meta_gated_seg_meta.json","w") as f:
    json.dump({'A':A_name,'B':B_name,'cv_gate':float(cv_gate),'cv_cal':float(cv_cal),
               'kA':kA,'tA':float(tA),'kB':kB,'tB':float(tB),
               'aA':aA,'bA':bA,'aB':aB,'bB':bB}, f, indent=2)
log("Saved -> out/test_predictions_meta_gated_seg.csv")


[16:04:40] [gate params] segA cv=52.028% k=0.5 t=-0.0046 | segB cv=44.401% k=3 t=-0.0121 | +2.3s
[16:04:40] [gated blend] CV (pre-cal): 46.841% | +2.3s
[16:04:41] [gated blend] CV (post-cal): 47.024% | +2.3s
[16:04:41] Saved -> out/test_predictions_meta_gated_seg.csv | +2.6s


In [54]:
# === Optional: per-segment price-bin ridge calibration on gated output ===
import os, json, numpy as np, pandas as pd
from math import exp

def smape(y_true, y_pred):
    d=(np.abs(y_true)+np.abs(y_pred))/2.0
    m=d!=0; out=np.zeros_like(d,float); out[m]=np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

base_oof = "out/oof_meta_gated_seg.csv"
base_te  = "out/test_predictions_meta_gated_seg.csv"
assert os.path.exists(base_oof) and os.path.exists(base_te), "Run gated_seg first."

oo = pd.read_csv(base_oof).set_index('sample_id')['oof_price'].reindex(train['sample_id']).values.astype('float32')
te = pd.read_csv(base_te ).set_index('sample_id')['price'    ].reindex(test['sample_id']).values.astype('float32')
y  = train['price'].values.astype('float32')

# same seg masks from earlier cell
# A_tr, B_tr, A_te_idx, B_te_idx already computed in kernel; recompute if not
try:
    A_tr, B_tr, A_te_idx, B_te_idx
except NameError:
    tcol=df_all['catalog_content'].fillna('').astype(str).str.lower()
    qty_pat=r'\b(?:\d+(?:\.\d+)?\s*(?:g|gram|kg|ml|l|litre|liter|oz|pack|pcs|piece|count))\b'
    mmq_all=~tcol.str.contains(qty_pat,regex=True,na=False).values
    if os.path.exists("/kaggle/input/saved-ml-model/checkpoint/img_has.npy"):
        has_img_all=np.load("/kaggle/input/saved-ml-model/checkpoint/img_has.npy").astype(bool)
    elif os.path.exists("img_has.npy"):
        has_img_all=np.load("img_has.npy").astype(bool)
    else:
        has_img_all=(df_all['image_link'].fillna('')!='').values
    is_tr=(df_all['is_train'].values==1); is_te=(~is_tr)
    segA_all=(has_img_all & mmq_all)
    A_tr=np.where(is_tr & segA_all)[0]; B_tr=np.where(is_tr & ~segA_all)[0]
    A_te_idx=(np.where(is_te & segA_all)[0]-len(train)).astype(int)
    B_te_idx=(np.where(is_te & ~segA_all)[0]-len(train)).astype(int)
    A_te_idx=A_te_idx[(A_te_idx>=0)&(A_te_idx<len(test))]
    B_te_idx=B_te_idx[(B_te_idx>=0)&(B_te_idx<len(test))]

def ridge_ab(p_log, t_log, lam=40.0):
    s_pp=float((p_log*p_log).sum()); s_p=float(p_log.sum()); s_pt=float((p_log*t_log).sum()); n=float(len(p_log))
    XtX=np.array([[s_pp, s_p],[s_p, n]],dtype='float64'); Xty=np.array([s_pt, t_log.sum()],dtype='float64')
    A=XtX + lam*np.eye(2); b=Xty + lam*np.array([1.0,0.0])  # prior (a=1,b=0)
    a,b0=np.linalg.solve(A,b); a=float(np.clip(a,0.7,1.3)); b0=float(np.clip(b0,-1,1)); return a,b0

def smooth_vec(v, sigma=1.0):
    out=np.zeros_like(v,'float64')
    for i in range(len(v)):
        num=0.0; den=0.0
        for j in range(len(v)):
            g=exp(-0.5*((i-j)/sigma)**2); num+=g*v[j]; den+=g
        out[i]=num/den
    return out

def perseg_pricebin_cal(oo, te, idx_tr, idx_te, nbins=20, sigma=1.0):
    eps=1e-4
    p_tr=np.log(np.clip(oo[idx_tr],eps,None)); t_tr=np.log(np.clip(y[idx_tr],eps,None))
    p_te=np.log(np.clip(te[idx_te],eps,None))
    cuts=np.quantile(oo[idx_tr], np.linspace(0,1,nbins+1)); cuts[0]-=1e-6; cuts[-1]+=1e-6
    btr=np.digitize(oo[idx_tr], cuts[1:-1], right=True); bte=np.digitize(te[idx_te], cuts[1:-1], right=True)
    a_raw=np.zeros(nbins); b_raw=np.zeros(nbins)
    for b in range(nbins):
        j=np.where(btr==b)[0]
        if j.size>=150: lam=40.0*(250.0/j.size); a_raw[b],b_raw[b]=ridge_ab(p_tr[j], t_tr[j], lam=lam)
        else: a_raw[b],b_raw[b]=1.0,0.0
    a_sm=smooth_vec(a_raw,sigma=sigma); b_sm=smooth_vec(b_raw,sigma=sigma)
    def apply_binwise(p_log, bins, a, b):
        out=np.empty_like(p_log,'float64')
        for bb in range(nbins):
            idc=(bins==bb); 
            if np.any(idc): out[idc]=np.exp(a[bb]*p_log[idc]+b[bb])
        return out
    oo_c=oo.copy(); te_c=te.copy()
    if idx_tr.size:
        res=apply_binwise(np.log(np.clip(oo[idx_tr],eps,None)), btr, a_sm, b_sm)
        oo_c[idx_tr]=res.clip(eps)
    if idx_te.size:
        resT=apply_binwise(np.log(np.clip(te[idx_te],eps,None)), bte, a_sm, b_sm)
        te_c[idx_te]=resT.clip(eps)
    return oo_c.astype('float32'), te_c.astype('float32')

ooA, teA = perseg_pricebin_cal(oo, te, A_tr, A_te_idx, nbins=20, sigma=1.0)
ooB, teB = perseg_pricebin_cal(ooA, teA, B_tr, B_te_idx, nbins=20, sigma=1.0)  # cascade B after A
cv_before=smape(y,oo); cv_after=smape(y,ooB)
print(f"[per-seg pricebin cal] CV before={cv_before:.3f}%, after={cv_after:.3f}%")

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_price': ooB}).to_csv("out/oof_meta_gated_seg_pricebin_cal.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],  'price': teB.astype(float)}).to_csv("out/test_predictions_meta_gated_seg_pricebin_cal.csv", index=False)
print("Saved -> out/test_predictions_meta_gated_seg_pricebin_cal.csv")


[per-seg pricebin cal] CV before=47.024%, after=47.051%
Saved -> out/test_predictions_meta_gated_seg_pricebin_cal.csv
